# Coincidencia emocional entre obra y visitante: proyecto completo en un notebook

**Procesamiento Digital de Imágenes · UAO 2026-2** · Jose Gabriel Castañeda Uribe · Juan José Osorio Reyes · Profesor: Nicolás Llanos Neuta

Este notebook reúne **todo el proyecto** y se ejecuta de arriba hacia abajo con **Ejecutar todo**:

1. **Importaciones y configuración**: instala lo que falte y define todas las rutas y parámetros en una sola celda.
2. **Carga y descripción del dataset**: recorre **todas** las imágenes de `positivo/` y `negativo/`, las valida, detecta duplicados, **describe y visualiza el dataset** y arma una validación cruzada estratificada de 5 folds.
3. **Entrenamiento y módulos**: CNN de obras con *transfer learning* (ResNet18 preentrenada en ImageNet) y los módulos del diagrama de arquitectura (ImageLoader → … → Integración).
4. **Resultados de la CNN**: curvas, métricas de validación cruzada, matriz de confusión, ROC, tabla imagen por imagen, galerías, **línea base de solo color**, **ejemplo con una imagen no vista** y **prueba de rendimiento (latencia)**.
5. **Aplicativo web**: levanta el servidor y abre la interfaz, que incluye el **formulario de evaluación con visitantes**.
6. **Evaluación con visitantes reales**: protocolo, procesamiento de capturas registradas por el equipo y métricas del sistema completo (rostro, CNN frente a la percepción humana y coincidencia).
7. **Análisis y conclusiones**: logros, limitaciones y oportunidades de mejora. Se genera `results/RESULTADOS.md`.
8. **Exportar al repositorio**: escribe `frontend/`, `results/` y `requirements.txt` listos para subir a GitHub.

> **Etiquetas:** las imágenes de `negativo/` se aprenden como obras que generan sentimientos **negativos** (clase 0) y las de `positivo/` como obras que generan sentimientos **positivos** (clase 1).
>
> **Importante:** en Colab activa la GPU (*Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*). En CPU el entrenamiento tarda más de una hora. Si ya tienes `cnn_obras.pt` en `Drive/Proyecto-final-de-PDI/resultados_cnn/`, pon `REENTRENAR = False` para no volver a entrenar.

## Descripción del proyecto

**Problemática.** En museos y exposiciones casi nunca se sabe si una obra produce en el visitante la emoción que su contenido sugiere. Las encuestas de salida son tardías, subjetivas y no dicen nada del momento de la observación. Tampoco existe una forma accesible de que el visitante compare su reacción con el carácter emocional de la obra.

**Justificación.** El análisis de la expresión humana con visión computacional permite medir, sin contacto y en segundos, dos cosas: el carácter emocional de una obra (a partir de su contenido visual y su paleta de color) y la expresión facial de quien la observa. Unir ambas lecturas en una misma escala de **valencia** (−1 desagradable … +1 agradable) da a curadores y educadores un indicador de cuánto *conecta* una obra con su público. Al visitante le ofrece una experiencia interactiva de reflexión sobre su propia reacción.

**Objetivo general.** Diseñar, implementar y evaluar un sistema interactivo que estime la coincidencia emocional entre una obra de arte y la expresión facial del visitante que la observa.

**Objetivos específicos.**
1. Clasificar obras en *genera sentimientos positivos / negativos* con una CNN (ResNet18 con *transfer learning*) y explicar la decisión con Grad-CAM y la paleta cromática.
2. Detectar el rostro del visitante (YuNet) y reconocer su expresión (FER+) a lo largo de una secuencia de fotogramas.
3. Integrar ambas lecturas en una medida de coincidencia emocional y presentarla en una interfaz web con manejo de excepciones.
4. Evaluar el sistema con métricas cuantitativas: validación cruzada de la CNN, rendimiento, y una prueba con visitantes reales capturados por el equipo.

**Solución propuesta.** Una aplicación web (FastAPI + HTML/JS) donde el visitante sube o elige una obra, la observa unos segundos frente a la cámara y recibe la clasificación de la obra, su expresión dominante y la coincidencia entre ambas. El equipo usa la misma interfaz para registrar la evaluación: el visitante dice qué sintió y si está de acuerdo con el resultado. Esas respuestas se comparan con la salida del sistema.

## 1. Importaciones y configuración
Instala solo las librerías que falten, importa todo lo necesario y define **en una sola celda** las rutas y parámetros editables.

In [ ]:
# Instala únicamente las dependencias que falten (la primera vez puede tardar: PyTorch pesa ~200 MB).
import importlib.util
import subprocess
import sys

assert sys.version_info >= (3, 10), "Se requiere Python 3.10 o superior"

REQUERIDAS = {  # módulo importable -> paquete de pip
    "fastapi": "fastapi", "uvicorn": "uvicorn", "multipart": "python-multipart",
    "numpy": "numpy", "PIL": "pillow", "cv2": "opencv-python", "onnxruntime": "onnxruntime",
    "torch": "torch", "torchvision": "torchvision", "sklearn": "scikit-learn",
    "matplotlib": "matplotlib", "pandas": "pandas", "scipy": "scipy",
}
faltan = [pip for modulo, pip in REQUERIDAS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
    importlib.invalidate_caches()
print("Python", sys.version.split()[0], "· dependencias listas")

In [ ]:
import base64
import colorsys
import copy
import hashlib
import io
import json
import os
import random
import re
import shutil
import socket
import tempfile
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
import webbrowser
import unicodedata
import uuid
import zipfile
from datetime import datetime
from pathlib import Path
from typing import List, Optional

os.environ.setdefault("OPENCV_LOG_LEVEL", "ERROR")  # silencia avisos internos de OpenCV

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import uvicorn
from fastapi import Body, FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse, Response
from PIL import Image, ImageFile, ImageOps, UnidentifiedImageError
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score, classification_report,
                             cohen_kappa_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

def _en_jupyter() -> bool:
    """True dentro de Jupyter/Colab; False si el notebook se exportó a .py y se ejecuta con python."""
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False


EN_JUPYTER = _en_jupyter()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(max(1, os.cpu_count() or 1))
print("Usando:", device, f"({torch.cuda.get_device_name(0)})" if device.type == "cuda" else "")
if device.type == "cpu":
    print("⚠ No hay GPU: el entrenamiento con validación cruzada puede tardar más de una hora.\n"
          "  En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU.")

In [ ]:
class config:
    """Única celda de configuración: todas las rutas y parámetros editables del proyecto."""

    # ======================= RUTAS (edita aquí) =======================
    RAIZ = Path.cwd()                                              # carpeta donde está este notebook
    DATASET_DIR = RAIZ / "Parcial2" / "Parcial2" / "dataset usado"  # carpeta con subcarpetas positivo/ y negativo/
    DATASET_ZIP = RAIZ / "dataset_usado.zip"                       # alternativa: zip del dataset (se extrae si falta la carpeta)
    DIR_TRABAJO = RAIZ / "app_pdi"                                 # salidas: dataset extraído, modelos, métricas y caché

    # Google Drive (Colab): Mi unidad/Proyecto-final-de-PDI/dataset usado/{positivo, negativo, precarga-sin-analizar}
    DRIVE_PROYECTO = Path("/content/drive/MyDrive/Proyecto-final-de-PDI")
    DATASET_DRIVE = DRIVE_PROYECTO / "dataset usado"               # dataset de entrenamiento (positivo/ y negativo/)
    PRECARGA_DIR = DATASET_DRIVE / "precarga-sin-analizar"         # imágenes para quien no quiera subir una propia
    RUTAS_PRECARGA = [PRECARGA_DIR,                                # se usa la primera que exista
                      DATASET_DIR / "precarga-sin-analizar",
                      RAIZ / "dataset usado" / "precarga-sin-analizar",
                      RAIZ / "precarga-sin-analizar"]
    DRIVE_RESULTADOS = DRIVE_PROYECTO / "resultados_cnn"           # copia persistente del modelo y los resultados

    DIR_MODELOS = DIR_TRABAJO / "modelos"
    RUTA_CNN = DIR_MODELOS / "cnn_obras.pt"
    RUTA_METRICAS = DIR_MODELOS / "metricas_cnn.json"
    RUTA_PARTICION = DIR_MODELOS / "particion_cnn.json"
    RUTA_PREDICCIONES = DIR_MODELOS / "predicciones_por_imagen.csv"   # resultado de cada imagen en la validación cruzada
    RUTA_CV = DIR_MODELOS / "validacion_cruzada.npz"                   # curvas y probabilidades por fold y época
    RUTA_YUNET = DIR_MODELOS / "face_detection_yunet_2023mar.onnx"
    RUTA_FERPLUS = DIR_MODELOS / "emotion-ferplus-8.onnx"
    DIR_DATOS = DIR_TRABAJO / "datos"

    URL_YUNET = ("https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/"
                 "face_detection_yunet/face_detection_yunet_2023mar.onnx")
    URL_FERPLUS = ("https://media.githubusercontent.com/media/onnx/models/main/validated/vision/"
                   "body_analysis/emotion_ferplus/model/emotion-ferplus-8.onnx")

    # ======================= EJECUCIÓN =======================
    REENTRENAR = True              # True: entrena en cada ejecución · False: reutiliza cnn_obras.pt (local o en Drive)
    ENTRENAR_COMPARACION = True    # evalúa también la CNN del Parcial 2 con la misma validación cruzada
    PUERTO = 8000                  # puerto del aplicativo web (si está ocupado se prueba el siguiente)
    ABRIR_NAVEGADOR = True

    # ======================= ENTRENAMIENTO =======================
    SEED = 42
    IMG_SIZE = 224
    BATCH_SIZE = 32
    K_FOLDS = 5                    # validación cruzada: cada imagen se evalúa una vez con un modelo que no la vio
    ARQUITECTURA = "resnet18"      # modelo del aplicativo: "resnet18" (transfer learning) o "cnn_parcial2"
    PREENTRENADO = True            # pesos de ImageNet para la ResNet18 (se descargan la primera vez, ~45 MB)
    CAPAS_ENTRENABLES = ("layer4",)  # bloques de la ResNet que se ajustan; el resto queda congelado
    EPOCAS = 20                    # épocas por fold (ResNet18); la mejor se elige con la pérdida media de validación
    LR = 1e-4                      # tasa de aprendizaje de las capas preentrenadas que se ajustan
    LR_CABEZA = 1e-3               # tasa de aprendizaje de la capa final nueva
    EPOCAS_CNN = 30                # épocas por fold de la CNN del Parcial 2 (comparación)
    LR_CNN = 1e-3
    DROPOUT = 0.5
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 2 if os.name != "nt" else 0
    MEDIA_IMAGENET = [0.485, 0.456, 0.406]
    STD_IMAGENET = [0.229, 0.224, 0.225]
    UMBRAL_DUPLICADO = 4           # bits de diferencia (de 64) del dHash para considerar dos imágenes casi idénticas
    CLASES = ["negativo", "positivo"]           # 0 = genera sentimientos negativos · 1 = genera sentimientos positivos
    EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    # ======================= ImageLoader =======================
    FORMATOS_PERMITIDOS = {"JPEG", "PNG"}
    MAX_MB = 10
    MIN_LADO = 32

    # ======================= Preprocesamiento =======================
    MEDIA_DEFECTO = [0.5, 0.5, 0.5]   # solo si no hubiera modelo entrenado (valor del diagrama)
    STD_DEFECTO = [0.5, 0.5, 0.5]

    # ======================= ResultadoProceso =======================
    UMBRAL_CONFIANZA_OBRA = 0.60
    UMBRAL_CONFIANZA_ROSTRO = 0.80
    UMBRAL_EMOCION = 0.40
    TAM_MIN_ROSTRO = 0.06
    BRILLO_MIN = 45
    BRILLO_MAX = 225
    NITIDEZ_MIN = 25.0

    # ======================= Paleta y captura =======================
    N_COLORES = 6
    LADO_MUESTREO_PALETA = 160
    MAX_FRAMES = 60

    # ======================= Rostros dentro de la imagen =======================
    UMBRAL_ROSTRO_OBRA = 0.60
    TAM_MIN_ROSTRO_OBRA = 0.02
    LADO_MAX_DETECCION = 1280

    # ======================= Imágenes precargadas =======================
    DATASET_MAX_ALEATORIAS = 24
    EXTS_IMAGEN = {".jpg", ".jpeg", ".png"}

    # ======================= Coincidencia emocional =======================
    FORMULA_COINCIDENCIA = "v3"    # "v3": valencia expresiva (descuenta la cara neutral) · "v2": fórmula original
    UMBRAL_EXPRESIVIDAD = 0.30     # 1 − p(neutral) mínimo para considerar concluyente la coincidencia

    # ======================= Resultados, rendimiento y evaluación =======================
    DIR_RESULTADOS = DIR_TRABAJO / "results"          # métricas, tablas y figuras para el repositorio
    DIR_FIGURAS = DIR_RESULTADOS / "figuras"          # cada plt.show() guarda aquí la figura (además de mostrarla)
    BENCH_N = 20                   # imágenes para medir la latencia del flujo completo
    BENCH_FRAMES = 30              # fotogramas para medir los fps del análisis facial
    EVAL_INTERVALO_MS = 400        # muestreo de fotogramas al procesar videos de la evaluación (igual que la GUI)
    MOSTRAR_IP_TUNEL = False       # True solo si usas Localtunnel; no subas esa salida al repositorio

config.DIR_MODELOS.mkdir(parents=True, exist_ok=True)
config.DIR_FIGURAS.mkdir(parents=True, exist_ok=True)
SEED, IMG_SIZE, BATCH_SIZE, CLASES, EXTS = config.SEED, config.IMG_SIZE, config.BATCH_SIZE, config.CLASES, config.EXTS


def dir_evaluacion() -> Path:
    """Carpeta de la evaluación con visitantes: en Drive si está montado (persiste entre sesiones)."""
    d = config.DRIVE_PROYECTO / "evaluacion" if config.DRIVE_PROYECTO.is_dir() else config.DIR_TRABAJO / "evaluacion"
    d.mkdir(parents=True, exist_ok=True)
    return d


# Cada plt.show() guarda la figura en results/figuras/<título>.png (evidencia visual para el README y el póster).
# En Jupyter además la muestra; como script .py solo la guarda, para no bloquear la ejecución.
if not EN_JUPYTER:
    plt.switch_backend("Agg")
_mostrar_original = plt.show


def _nombre_figura(fig) -> str:
    titulo = fig._suptitle.get_text() if getattr(fig, "_suptitle", None) else ""
    if not titulo and fig.axes:
        titulo = " · ".join(t for t in (a.get_title() for a in fig.axes[:2]) if t)
    ascii_ = unicodedata.normalize("NFKD", titulo or "figura").encode("ascii", "ignore").decode()   # nombres sin tildes
    return re.sub(r"[^\w-]+", "_", ascii_).strip("_")[:70] or "figura"     # mismo título → se sobrescribe


def _guardar_y_mostrar(*args, **kwargs):
    for num in plt.get_fignums():
        fig = plt.figure(num)
        try:
            fig.savefig(config.DIR_FIGURAS / f"{_nombre_figura(fig)}.png", dpi=120, bbox_inches="tight")
        except Exception as e:                       # una figura que no se pueda guardar no detiene el notebook
            print("No se pudo guardar la figura:", e)
    if EN_JUPYTER:
        _mostrar_original(*args, **kwargs)
    else:
        plt.close("all")


plt.show = _guardar_y_mostrar
print("Carpeta del notebook:", config.RAIZ)
print("Salidas en:", config.DIR_TRABAJO, "· figuras en:", config.DIR_FIGURAS)

## 2. Carga y descripción del dataset
Se localiza `dataset usado` en Google Drive y se recorren **todas** las imágenes de `positivo/` y `negativo/`, una por una (también las de subcarpetas). Cada imagen se valida, se etiqueta según su carpeta y se le calcula una huella (MD5 y dHash) para encontrar duplicados. Después se arma la validación cruzada de 5 folds. También se descargan los modelos preentrenados de rostro (YuNet) y de emociones (FER+), y se prepara la carpeta `precarga-sin-analizar`.

### ImageLoader
Valida y decodifica imágenes JPG/PNG (≤ 10 MB, > 32 px); los errores se devuelven con un código controlado.

In [ ]:
"""
ImageLoader
-----------
Función: cargar y validar el archivo de imagen seleccionado por el usuario.
Entrada: bytes del archivo (selector de la GUI o fotograma de la cámara). Se admiten JPEG y PNG.
Salida : imagen decodificada en memoria, RGB de 3 canales, con su ancho y alto originales.
Error  : archivo ausente, corrupto o no compatible  ->  ErrorProceso con código controlado.
"""


Image.MAX_IMAGE_PIXELS = 120_000_000  # protege contra imágenes descomunales


class ErrorProceso(Exception):
    """Error controlado: lleva un código (tipo) y un mensaje en español para la GUI."""

    def __init__(self, tipo: str, mensaje: str, gravedad: str = "error", modulo: str = "ImageLoader"):
        super().__init__(mensaje)
        self.tipo = tipo
        self.mensaje = mensaje
        self.gravedad = gravedad
        self.modulo = modulo

    def a_dict(self) -> dict:
        return {"tipo": self.tipo, "gravedad": self.gravedad, "mensaje": self.mensaje, "modulo": self.modulo}


def cargar_imagen(datos: bytes | None, nombre: str = "imagen", permitir_formatos=None) -> np.ndarray:
    permitidos = permitir_formatos or config.FORMATOS_PERMITIDOS
    if not datos:
        raise ErrorProceso("archivo_ausente", f"No se recibió ningún archivo para «{nombre}».")
    if len(datos) > config.MAX_MB * 1024 * 1024:
        raise ErrorProceso("formato", f"«{nombre}» pesa más de {config.MAX_MB} MB. Elige una imagen más ligera.")
    try:
        with Image.open(io.BytesIO(datos)) as im:
            formato = (im.format or "").upper()
            if formato not in permitidos:
                raise ErrorProceso("formato", f"Formato no soportado ({formato or 'desconocido'}). Ingresa una imagen JPG o PNG.")
            im.load()
            im = ImageOps.exif_transpose(im)  # respeta la orientación de fotos de celular
            rgb = im.convert("RGB")
    except ErrorProceso:
        raise
    except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombError):
        raise ErrorProceso("archivo_corrupto", f"No se pudo leer «{nombre}». El archivo está dañado o no es una imagen válida.")

    w, h = rgb.size
    if w < config.MIN_LADO or h < config.MIN_LADO:
        raise ErrorProceso("fuera_de_rango", f"«{nombre}» es demasiado pequeña ({w}×{h} px). Usa una de al menos {config.MIN_LADO} px por lado.")
    return np.asarray(rgb, dtype=np.uint8)

### Localizar el dataset

In [ ]:
import os
from pathlib import Path

# Importar librería de Colab para montar Google Drive
try:
    from google.colab import drive
    print("Entorno de Google Colab detectado. Solicitando acceso a Google Drive...")
    drive.mount('/content/drive')
except ImportError:
    print("No se detectó Google Colab.")

def localizar_dataset() -> Path:
    """Devuelve la carpeta que contiene positivo/ y negativo/ buscando en Google Drive."""
    def valida(c: Path) -> bool:
        return all((c / k).is_dir() for k in CLASES)

    # Rutas a revisar. La primera es la del Drive del proyecto; si la guardaste en otro lugar, ajusta config.DATASET_DRIVE.
    rutas_por_defecto = [
        config.DATASET_DRIVE,                       # Mi unidad/Proyecto-final-de-PDI/dataset usado
        config.DATASET_DIR,
        config.RAIZ / "dataset usado",
        config.DIR_TRABAJO / "dataset usado"
    ]

    for c in rutas_por_defecto:
        if c.is_dir() and valida(c):
            print(f"Dataset cargado exitosamente desde: {c}")
            return c

    raise FileNotFoundError(
        "No se encontró el dataset. Verifica en drive.google.com que exista "
        "'Mi unidad/Proyecto-final-de-PDI/dataset usado' con las carpetas 'positivo' y 'negativo'."
    )

# Ejecución
DATASET = localizar_dataset()

for clase in ["positivo", "negativo"]:
    n = sum(1 for f in os.listdir(DATASET / clase) if Path(f).suffix.lower() in EXTS)
    print(f"{clase:9s} {n} imágenes")

print("Dataset:", DATASET)

### Inventario imagen por imagen, duplicados y validación cruzada de 5 folds
No se separa un conjunto fijo de prueba. Las 500 imágenes se reparten en 5 grupos (*folds*) con la misma proporción de clases. Cada modelo se entrena con 4 grupos y predice el quinto, así que **cada imagen recibe una predicción de un modelo que nunca la vio**. Las imágenes casi idénticas quedan siempre en el mismo fold para que una copia no "filtre" la respuesta de la otra.

In [ ]:
def fijar_semillas(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def dhash(im: Image.Image, lado: int = 8) -> np.ndarray:
    """Huella perceptual de 64 bits: dos imágenes casi idénticas difieren en muy pocos bits."""
    g = np.asarray(im.convert("L").resize((lado + 1, lado), Image.BILINEAR), dtype=np.int16)
    return (g[:, 1:] > g[:, :-1]).flatten()


def inventario_dataset(dataset_dir: Path):
    """Recorre positivo/ y negativo/ (incluidas subcarpetas). Cada imagen se carga individualmente,
    se etiqueta por su carpeta y se guarda redimensionada en memoria para entrenar."""
    muestras, descartadas = [], []
    ImageFile.LOAD_TRUNCATED_IMAGES = True           # como en el entrenamiento original
    try:
        for etiqueta, clase in enumerate(CLASES):
            carpeta = dataset_dir / clase
            rutas = sorted(p for p in carpeta.rglob("*") if p.is_file() and p.suffix.lower() in EXTS)
            for ruta in rutas:
                rel = str(ruta.relative_to(dataset_dir))
                try:
                    datos = ruta.read_bytes()
                    with Image.open(io.BytesIO(datos)) as im:
                        im = ImageOps.exif_transpose(im).convert("RGB")   # igual que el ImageLoader de la app
                        w, h = im.size
                        if min(w, h) < config.MIN_LADO:
                            raise ValueError(f"demasiado pequeña ({w}×{h} px)")
                        pixeles = np.asarray(im.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)
                        huella = dhash(im)
                    muestras.append({"archivo": ruta.name, "rel": rel, "ruta": ruta, "clase": clase,
                                     "etiqueta": etiqueta, "ancho": w, "alto": h,
                                     "md5": hashlib.md5(datos).hexdigest(), "dhash": huella, "pixeles": pixeles})
                except Exception as e:
                    descartadas.append({"archivo": rel, "motivo": str(e) or type(e).__name__})
    finally:
        ImageFile.LOAD_TRUNCATED_IMAGES = False      # la app rechaza imágenes truncadas
    return muestras, descartadas


def agrupar_duplicados(muestras, umbral: int = config.UMBRAL_DUPLICADO) -> np.ndarray:
    """Une en un mismo grupo las imágenes idénticas (MD5) o casi idénticas (dHash a ≤ umbral bits)."""
    n = len(muestras)
    padre = list(range(n))

    def raiz(i):
        while padre[i] != i:
            padre[i] = padre[padre[i]]
            i = padre[i]
        return i

    def unir(i, j):
        ri, rj = raiz(i), raiz(j)
        if ri != rj:
            padre[max(ri, rj)] = min(ri, rj)

    huellas = np.stack([m["dhash"] for m in muestras])
    for ini in range(0, n, 256):                     # por bloques para no agotar la memoria con datasets grandes
        dist = (huellas[ini:ini + 256, None, :] != huellas[None, :, :]).sum(-1)
        for a, b in zip(*np.where(dist <= umbral)):
            if ini + a < b:
                unir(ini + a, b)
    por_md5 = {}
    for i, m in enumerate(muestras):
        if m["md5"] in por_md5:
            unir(i, por_md5[m["md5"]])
        por_md5.setdefault(m["md5"], i)
    return np.array([raiz(i) for i in range(n)])


# --------------------------------------------------------------------- inventario
t_carga = time.time()
print("Leyendo cada imagen de positivo/ y negativo/ …")
muestras, descartadas = inventario_dataset(DATASET)
N = len(muestras)
etiquetas = np.array([m["etiqueta"] for m in muestras])
for i, clase in enumerate(CLASES):
    print(f"  {clase:9s}: {(etiquetas == i).sum():4d} imágenes válidas")
print(f"  Total    : {N:4d} · leídas en {time.time() - t_carga:.0f} s")
if descartadas:
    print(f"⚠ {len(descartadas)} archivo(s) descartado(s):")
    for d in descartadas:
        print(f"   - {d['archivo']}: {d['motivo']}")
assert N > 0 and all((etiquetas == i).any() for i in range(len(CLASES))), "Faltan imágenes en alguna clase."

# --------------------------------------------------------------------- duplicados
grupos = agrupar_duplicados(muestras)
duplicados = {}
for g in np.unique(grupos):
    idx = np.where(grupos == g)[0]
    if len(idx) > 1:
        duplicados[int(g)] = idx.tolist()
contradictorios = {g: idx for g, idx in duplicados.items() if len({muestras[i]["clase"] for i in idx}) > 1}
print(f"\nGrupos de imágenes repetidas o casi idénticas: {len(duplicados)} "
      f"({sum(len(v) for v in duplicados.values())} imágenes) · con etiquetas contradictorias: {len(contradictorios)}")
for g, idx in list(contradictorios.items())[:10]:
    print("   ⚠ misma imagen en ambas clases:", ", ".join(f"{muestras[i]['rel']}" for i in idx))


def mostrar_grupos(grupos_idx, titulo, maximo=6):
    grupos_idx = list(grupos_idx)[:maximo]
    if not grupos_idx:
        return
    ancho = max(len(g) for g in grupos_idx)
    fig, ax = plt.subplots(len(grupos_idx), ancho, figsize=(2.6 * ancho, 2.8 * len(grupos_idx)), squeeze=False)
    for f, g in enumerate(grupos_idx):
        for c in range(ancho):
            ax[f][c].axis("off")
            if c < len(g):
                m = muestras[g[c]]
                ax[f][c].imshow(m["pixeles"])
                ax[f][c].set_title(f"{m['clase']}\n{m['archivo']}", fontsize=8)
    fig.suptitle(titulo)
    plt.tight_layout(); plt.show()


mostrar_grupos(contradictorios.values(), "Duplicados con etiqueta contradictoria (revisar)")

# --------------------------------------------------------------------- normalización y transformaciones
suma, suma_cuad, n_px = np.zeros(3), np.zeros(3), 0          # acumulado imagen por imagen (poca memoria)
for m in muestras:
    x = m["pixeles"].reshape(-1, 3).astype(np.float64) / 255.0
    suma += x.sum(0); suma_cuad += (x ** 2).sum(0); n_px += len(x)
MEDIA_DATASET = (suma / n_px).round(4).tolist()
STD_DATASET = np.sqrt(suma_cuad / n_px - (suma / n_px) ** 2).round(4).tolist()
print("\nMedia del dataset (R,G,B):", MEDIA_DATASET, "· Std:", STD_DATASET)


def normalizacion(arquitectura: str):
    """La ResNet preentrenada espera la normalización de ImageNet; la CNN del Parcial 2, la del dataset."""
    if arquitectura == "resnet18":
        return config.MEDIA_IMAGENET, config.STD_IMAGENET
    return MEDIA_DATASET, STD_DATASET


def tf_entrenamiento(media_n, std_n):
    return transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0), ratio=(0.9, 1.1)),  # encuadres distintos
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(8),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),   # suave: el color también comunica emoción
        transforms.ToTensor(),
        transforms.Normalize(media_n, std_n),
    ])


def tf_evaluacion(media_n, std_n):
    return transforms.Compose([transforms.ToTensor(), transforms.Normalize(media_n, std_n)])


class ObrasDataset(Dataset):
    """Devuelve (tensor, etiqueta, índice de la imagen) para poder rastrear el resultado de cada imagen."""

    def __init__(self, indices, transform):
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, k):
        i = self.indices[k]
        m = muestras[i]
        return self.transform(Image.fromarray(m["pixeles"])), m["etiqueta"], i


def crear_loader(indices, transform, barajar: bool, seed: int = SEED):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ObrasDataset(indices, transform), batch_size=BATCH_SIZE, shuffle=barajar, generator=g,
                      num_workers=config.NUM_WORKERS, pin_memory=device.type == "cuda",
                      persistent_workers=config.NUM_WORKERS > 0)


# --------------------------------------------------------------------- validación cruzada
skf = StratifiedGroupKFold(n_splits=config.K_FOLDS, shuffle=True, random_state=SEED)
folds = [(tr, va) for tr, va in skf.split(np.zeros(N), etiquetas, groups=grupos)]
FOLD_DE = np.empty(N, dtype=int)
print(f"\nValidación cruzada de {config.K_FOLDS} folds (cada imagen queda fuera del entrenamiento exactamente una vez):")
for f, (tr, va) in enumerate(folds):
    FOLD_DE[va] = f
    print(f"  Fold {f + 1}: entrena con {len(tr)} ({(etiquetas[tr] == 1).sum()} pos / {(etiquetas[tr] == 0).sum()} neg)"
          f" · evalúa {len(va)} ({(etiquetas[va] == 1).sum()} pos / {(etiquetas[va] == 0).sum()} neg)")


def mostrar_batch(n=8):
    media_n, std_n = normalizacion(config.ARQUITECTURA)
    imgs, ys, _ = next(iter(DataLoader(ObrasDataset(folds[0][0], tf_entrenamiento(media_n, std_n)),
                                       batch_size=n, shuffle=True)))
    imgs = (imgs * torch.tensor(std_n).view(1, 3, 1, 1) + torch.tensor(media_n).view(1, 3, 1, 1)).clamp(0, 1)
    plt.figure(figsize=(16, 4))
    for i in range(len(imgs)):
        plt.subplot(1, n, i + 1)
        plt.imshow(imgs[i].permute(1, 2, 0))
        plt.title(CLASES[ys[i]])
        plt.axis("off")
    plt.suptitle("Lote de entrenamiento con aumento de datos")
    plt.show()


fijar_semillas()
mostrar_batch()

### Descripción y visualización del dataset
Resumen de lo que entra al modelo: cantidad por clase, resolución original, formato y dos rasgos de color (brillo y saturación) que también usa la paleta. Las figuras y `results/dataset_resumen.csv` quedan listos para la sección **Datos** del README.

In [ ]:
# Descripción del dataset: tabla por imagen, estadísticas por clase y muestras visuales
def rasgos_basicos(px: np.ndarray) -> dict:
    hsv = cv2.cvtColor(px, cv2.COLOR_RGB2HSV)
    return {"brillo": float(hsv[..., 2].mean()), "saturacion": float(hsv[..., 1].mean())}


tabla_ds = pd.DataFrame([{
    "archivo": m["archivo"], "clase": m["clase"], "formato": m["ruta"].suffix.lower().lstrip("."),
    "ancho": m["ancho"], "alto": m["alto"], **rasgos_basicos(m["pixeles"]),
} for m in muestras])
tabla_ds["lado_mayor"] = tabla_ds[["ancho", "alto"]].max(axis=1)
tabla_ds["aspecto"] = (tabla_ds["ancho"] / tabla_ds["alto"]).round(3)
config.DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)
tabla_ds.to_csv(config.DIR_RESULTADOS / "dataset_resumen.csv", index=False, encoding="utf-8")

resumen_ds = tabla_ds.groupby("clase").agg(
    imagenes=("archivo", "count"), ancho_mediana=("ancho", "median"), alto_mediana=("alto", "median"),
    aspecto_medio=("aspecto", "mean"), brillo_medio=("brillo", "mean"), saturacion_media=("saturacion", "mean")).round(2)
print("Resumen por clase (resolución original; el modelo recibe 224×224):")
print(resumen_ds.to_string())
print("\nFormatos:", tabla_ds["formato"].value_counts().to_dict(),
      f"· descartadas al leer: {len(descartadas)} · grupos duplicados: {len(duplicados)}")

# Muestras aleatorias por clase
rng_ds = np.random.default_rng(SEED)
n_cols = 8
fig, ax = plt.subplots(2, n_cols, figsize=(18, 5))
for f, clase in enumerate(CLASES):
    idx = [i for i, m in enumerate(muestras) if m["clase"] == clase]
    for c, i in enumerate(rng_ds.choice(idx, size=min(n_cols, len(idx)), replace=False)):
        ax[f, c].imshow(muestras[i]["pixeles"]); ax[f, c].axis("off")
        ax[f, c].set_title(f"{clase}\n{muestras[i]['archivo']}", fontsize=8)
fig.suptitle("Dataset — muestras aleatorias por clase"); plt.tight_layout(); plt.show()

# Distribuciones por clase
colores_clase = {"negativo": "#eb6834", "positivo": "#2a78d6"}
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
for clase in CLASES:
    sub = tabla_ds[tabla_ds["clase"] == clase]
    ax[0].hist(sub["lado_mayor"], bins=25, alpha=0.6, color=colores_clase[clase], label=clase)
ax[0].set_xlabel("Lado mayor original (px)"); ax[0].set_ylabel("Imágenes"); ax[0].legend()
ax[0].set_title("Resolución original")
for k, (col, nombre) in enumerate([("brillo", "Brillo medio (V de HSV)"), ("saturacion", "Saturación media (S de HSV)")], 1):
    ax[k].boxplot([tabla_ds.loc[tabla_ds["clase"] == c, col] for c in CLASES], widths=0.5)
    ax[k].set_xticks(range(1, len(CLASES) + 1), CLASES); ax[k].set_title(nombre); ax[k].set_ylabel("0-255")
fig.suptitle("Dataset — distribuciones por clase"); plt.tight_layout(); plt.show()

### Modelos preentrenados de rostro y emociones

In [ ]:
# Modelos preentrenados de rostro (YuNet, OpenCV Zoo) y emociones (FER+, ONNX Model Zoo).
MODELOS_EXTERNOS = [(config.RUTA_YUNET, config.URL_YUNET, 200_000),
                    (config.RUTA_FERPLUS, config.URL_FERPLUS, 30_000_000)]


def descargar_si_falta(verbose: bool = True) -> list:
    errores = []
    config.DIR_MODELOS.mkdir(parents=True, exist_ok=True)
    for ruta, url, tam_min in MODELOS_EXTERNOS:
        if ruta.exists() and ruta.stat().st_size >= tam_min:
            continue
        local = config.RAIZ / "ProyectoEmociones" / "modelos" / ruta.name   # copia del proyecto anterior, si existe
        if local.is_file() and local.stat().st_size >= tam_min:
            shutil.copy2(local, ruta)
            continue
        try:
            if verbose:
                print(f"Descargando {ruta.name} …", flush=True)
            tmp = ruta.with_suffix(".part")
            urllib.request.urlretrieve(url, tmp)
            if tmp.stat().st_size < tam_min:
                raise RuntimeError("archivo incompleto")
            tmp.replace(ruta)
        except Exception as e:  # sin internet, proxy, etc.
            errores.append(f"No se pudo descargar {ruta.name}: {e}")
    return errores


errores_descarga = descargar_si_falta()
print("\n".join(errores_descarga) if errores_descarga else "Modelos de rostro y emociones listos en " + str(config.DIR_MODELOS))

### Imágenes precargadas (`precarga-sin-analizar`)
Lee directamente la carpeta `Mi unidad/Proyecto-final-de-PDI/dataset usado/precarga-sin-analizar` de Google Drive y entrega selecciones aleatorias. Es la fuente de imágenes cuando el visitante no quiere subir una propia.

In [ ]:
"""
DatasetPrecarga
---------------
Función: entregar imágenes elegidas al azar de la carpeta de Google Drive
         Mi unidad/Proyecto-final-de-PDI/dataset usado/precarga-sin-analizar
         para quien no quiera subir una imagen propia.
Entrada: la carpeta config.PRECARGA_DIR (se lee directamente de Drive: sin descargas ni credenciales).
Salida : nombres, rutas y bytes de las imágenes JPG/PNG de la carpeta.
Error  : carpeta ausente o vacía  ->  ErrorProceso con código controlado.
"""

MODULO = "DatasetPrecarga"
_azar = random.SystemRandom()  # selección completamente aleatoria


def _error(tipo: str, mensaje: str, gravedad: str = "error") -> ErrorProceso:
    return ErrorProceso(tipo, mensaje, gravedad, MODULO)


def localizar_precarga() -> Path:
    """Primera carpeta de precarga que exista; si ninguna existe, la ruta principal de Drive."""
    for c in config.RUTAS_PRECARGA:
        if c.is_dir():
            return c
    return config.PRECARGA_DIR


class DatasetPrecarga:
    def __init__(self):
        self.dir_origen = localizar_precarga()
        self._lock = threading.Lock()
        self._indice: dict | None = None
        self.particion = self._cargar_particion()

    @staticmethod
    def _cargar_particion() -> dict:
        """Partición de la CNN por nombre de archivo: avisa si una imagen ya la vio al entrenar."""
        try:
            return json.loads(config.RUTA_PARTICION.read_text(encoding="utf-8"))["archivos"]
        except (OSError, ValueError, KeyError):
            return {}

    def _leer_indice(self) -> dict | None:
        if self._indice is not None:
            return self._indice
        if not self.dir_origen.is_dir():
            self.dir_origen = localizar_precarga()   # por si Drive se montó después de crear el objeto
            if not self.dir_origen.is_dir():
                return None
        archivos = sorted(p for p in self.dir_origen.rglob("*")
                          if p.is_file() and p.suffix.lower() in config.EXTS_IMAGEN)
        if not archivos:
            return None
        self._indice = {
            "fecha": time.time(),
            "base": str(self.dir_origen),
            "archivos": {p.name: {"bytes": p.stat().st_size, "rel": str(p.relative_to(self.dir_origen))}
                         for p in archivos},
        }
        return self._indice

    def asegurar(self, forzar: bool = False) -> dict:
        """Garantiza el índice de imágenes; forzar=True vuelve a leer la carpeta (imágenes nuevas)."""
        with self._lock:
            if forzar:
                self._indice = None
            indice = self._leer_indice()
            if not self.dir_origen.is_dir():
                raise _error("dataset_no_encontrado",
                             f"No se encontró la carpeta «{self.dir_origen}». Monta Google Drive y verifica que exista "
                             "Mi unidad/Proyecto-final-de-PDI/dataset usado/precarga-sin-analizar.")
            if not indice:
                raise _error("dataset_no_disponible",
                             f"La carpeta «{self.dir_origen}» está vacía o no tiene imágenes JPG/PNG.")
            return indice

    def estado(self) -> dict:
        indice = self._leer_indice()
        return {
            "carpeta": str(self.dir_origen),
            "listo": bool(indice),
            "n_imagenes": len(indice["archivos"]) if indice else 0,
            "fecha": indice["fecha"] if indice else None,
            "n_particion_cnn": len(self.particion),
        }

    def info_imagen(self, nombre: str, indice: dict | None = None) -> dict:
        indice = indice or self._leer_indice() or {"archivos": {}}
        meta = indice["archivos"].get(nombre, {})
        return {
            "nombre": nombre,
            "bytes": meta.get("bytes"),
            "url": "/api/dataset/imagen?nombre=" + urllib.parse.quote(nombre, safe=""),
            "cnn": self.particion.get(nombre),
        }

    def aleatorias(self, n: int, excluir: list[str] | None = None) -> dict:
        n = max(1, min(int(n), config.DATASET_MAX_ALEATORIAS))
        indice = self.asegurar()
        excluir = set(excluir or [])
        pool = [k for k in indice["archivos"] if k not in excluir] or list(indice["archivos"])
        elegidas = _azar.sample(pool, min(n, len(pool)))
        return {"imagenes": [self.info_imagen(x, indice) for x in elegidas],
                "total_dataset": len(indice["archivos"]), "avisos": []}

    def ruta(self, nombre: str) -> Path:
        """Ruta segura a la imagen: solo nombres que estén en el índice de la carpeta."""
        indice = self.asegurar()
        if nombre not in indice["archivos"]:
            raise _error("imagen_no_encontrada", f"«{nombre}» no está en la carpeta precarga-sin-analizar.")
        ruta = Path(indice["base"]) / indice["archivos"][nombre]["rel"]
        if not ruta.is_file():
            raise _error("imagen_no_encontrada", f"«{nombre}» ya no está en la carpeta. Pulsa «Volver a leer la carpeta».")
        return ruta

    def leer(self, nombre: str) -> bytes:
        return self.ruta(nombre).read_bytes()


precarga = DatasetPrecarga()
print("Carpeta de imágenes precargadas:", precarga.dir_origen)
print(json.dumps(precarga.estado(), ensure_ascii=False, indent=1))

## 3. Entrenamiento y análisis de rostros
Se entrena la **CNN de obras con transfer learning**: una ResNet18 preentrenada en ImageNet a la que solo se le ajustan el último bloque (`layer4`) y una capa final nueva con Dropout 0.5 y Weight Decay 1e-4.

1. **Validación cruzada (5 folds):** se entrenan 5 modelos; cada uno predice las 100 imágenes que no vio. La mejor época es la de menor pérdida media de validación.
2. **Modelo final:** se entrena con **todas** las imágenes durante esa cantidad de épocas y es el que usa el aplicativo.

Si `ENTRENAR_COMPARACION` está activo, la CNN del Parcial 2 se evalúa con los mismos folds para comparar. Después se definen los módulos del diagrama: preprocesamiento, clasificador con Grad-CAM, paleta, gestos faciales, reglas de resultado e integración.

### Arquitecturas: ResNet18 con transfer learning (aplicativo) y CNN del Parcial 2 (comparación)

In [ ]:
"""
ClasificadorDeObras
-------------------
Función: clasificar la obra en una de las dos categorías aprendidas (negativo / positivo).
Entrada: tensor float32 [1, 3, 224, 224] (de PreprocesamientoImagen).
Salida : tensor de logits [1, 2]  ->  puntuaciones de "negativo" y "positivo".

Ambas arquitecturas exponen features(x) -> mapa de activaciones y head(a) -> logits,
que es lo que usa Grad-CAM para mostrar qué zonas de la obra influyeron en la decisión.
"""


class CNN(nn.Module):
    """Arquitectura idéntica a la del notebook del Parcial 2 (se conserva para la comparación)."""

    def __init__(self, dropout: float = 0.0, img_size: int = 224):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        lado = img_size // 8
        self.clasificador = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * lado * lado, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 2),
        )

    def features(self, x):
        return self.convs[:8](x)                     # salida de la última ReLU (1, 64, 56, 56)

    def head(self, a):
        return self.clasificador(self.convs[8](a))

    def forward(self, x):
        return self.head(self.features(x))


class ResNetObras(nn.Module):
    """ResNet18 preentrenada en ImageNet: se congelan los primeros bloques y se ajustan los indicados."""

    BLOQUES = {"layer1": 4, "layer2": 5, "layer3": 6, "layer4": 7}

    def __init__(self, dropout: float = 0.5, preentrenado: bool = True, capas_entrenables=("layer4",)):
        super().__init__()
        pesos = models.ResNet18_Weights.IMAGENET1K_V1 if preentrenado else None
        base = models.resnet18(weights=pesos)
        self.base = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool,
                                  base.layer1, base.layer2, base.layer3, base.layer4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(512, 2))
        entrenables = {self.BLOQUES[c] for c in capas_entrenables}
        self._congelados = [i for i in range(len(self.base)) if i not in entrenables]
        for i in self._congelados:
            for p in self.base[i].parameters():
                p.requires_grad = False

    def train(self, mode: bool = True):
        super().train(mode)
        # Las BatchNorm congeladas conservan las estadísticas de ImageNet (más fiables que las de lotes de 32)
        for i in self._congelados:
            self.base[i].eval()
        return self

    def features(self, x):
        return self.base(x)                          # (1, 512, 7, 7)

    def head(self, a):
        return self.fc(self.pool(a))

    def forward(self, x):
        return self.head(self.features(x))


def crear_modelo(arquitectura: str, preentrenado: Optional[bool] = None, dropout: Optional[float] = None,
                 img_size: int = IMG_SIZE) -> nn.Module:
    preentrenado = config.PREENTRENADO if preentrenado is None else preentrenado
    dropout = config.DROPOUT if dropout is None else dropout
    if arquitectura == "resnet18":
        return ResNetObras(dropout, preentrenado, config.CAPAS_ENTRENABLES)
    if arquitectura == "cnn_parcial2":
        return CNN(dropout=dropout, img_size=img_size)
    raise ValueError(f"Arquitectura desconocida: {arquitectura}")


def contar_parametros(modelo):
    total = sum(p.numel() for p in modelo.parameters())
    entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    return total, entrenables


for arq in ["resnet18", "cnn_parcial2"]:
    t, e = contar_parametros(crear_modelo(arq, preentrenado=False))
    print(f"{arq:13s}: {t:>10,d} parámetros · {e:>10,d} entrenables")

### Funciones de entrenamiento, validación cruzada y métricas

In [ ]:
USAR_AMP = device.type == "cuda"   # precisión mixta en GPU: entrena ~2× más rápido


def crear_optimizador(modelo, arquitectura: str):
    if arquitectura == "resnet18":
        cabeza = list(modelo.fc.parameters())
        ids = {id(p) for p in cabeza}
        base = [p for p in modelo.parameters() if p.requires_grad and id(p) not in ids]
        grupos = [{"params": cabeza, "lr": config.LR_CABEZA}, {"params": base, "lr": config.LR}]
        return optim.AdamW(grupos, weight_decay=config.WEIGHT_DECAY)
    return optim.Adam(modelo.parameters(), lr=config.LR_CNN, weight_decay=config.WEIGHT_DECAY)


def una_epoca(modelo, loader, criterio, opt=None, scaler=None) -> dict:
    """Recorre el loader una vez. Con opt entrena; sin opt evalúa y devuelve la probabilidad de cada imagen."""
    entrenando = opt is not None
    modelo.train(entrenando)
    perdida, aciertos, n = 0.0, 0, 0
    probs, indices = [], []
    with torch.set_grad_enabled(entrenando):
        for x, y, idx in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USAR_AMP):
                out = modelo(x)
                loss = criterio(out, y)
            if entrenando:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                probs.append(torch.softmax(out.float(), dim=1)[:, 1].cpu())
                indices.append(idx)
            perdida += loss.item() * x.size(0)
            aciertos += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    res = {"loss": perdida / n, "acc": aciertos / n}
    if not entrenando:
        res["p_pos"], res["idx"] = torch.cat(probs).numpy(), torch.cat(indices).numpy()
    return res


def validacion_cruzada(arquitectura: str, epocas: int) -> dict:
    """Entrena un modelo por fold. Guarda, para cada época, la probabilidad que el modelo le dio a cada
    imagen que NO vio. Al final se elige la época con menor pérdida media de validación."""
    media_n, std_n = normalizacion(arquitectura)
    criterio = nn.CrossEntropyLoss()
    K = len(folds)
    hist = {k: np.zeros((K, epocas)) for k in ("train_loss", "val_loss", "train_acc", "val_acc")}
    p_oof = np.full((epocas, N), np.nan, dtype=np.float32)
    t0 = time.time()
    for f, (tr, va) in enumerate(folds):
        fijar_semillas(SEED + f)
        modelo = crear_modelo(arquitectura).to(device)
        opt = crear_optimizador(modelo, arquitectura)
        scaler = torch.amp.GradScaler(device.type, enabled=USAR_AMP)
        l_tr = crear_loader(tr, tf_entrenamiento(media_n, std_n), barajar=True, seed=SEED + f)
        l_va = crear_loader(va, tf_evaluacion(media_n, std_n), barajar=False)
        for ep in range(epocas):
            rt = una_epoca(modelo, l_tr, criterio, opt, scaler)
            rv = una_epoca(modelo, l_va, criterio)
            hist["train_loss"][f, ep], hist["train_acc"][f, ep] = rt["loss"], rt["acc"]
            hist["val_loss"][f, ep], hist["val_acc"][f, ep] = rv["loss"], rv["acc"]
            p_oof[ep, rv["idx"]] = rv["p_pos"]
            print(f"[{arquitectura}] Fold {f + 1}/{K} · Época {ep + 1:02d} | train_loss {rt['loss']:.4f} acc {rt['acc']:.3f}"
                  f" | val_loss {rv['loss']:.4f} acc {rv['acc']:.3f}", flush=True)
        print(f"   fold {f + 1} terminado · {time.time() - t0:.0f} s acumulados")
        del modelo, opt
        if device.type == "cuda":
            torch.cuda.empty_cache()
    mejor = int(np.argmin(hist["val_loss"].mean(axis=0))) + 1
    print(f"[{arquitectura}] Mejor época (pérdida media de validación mínima): {mejor}")
    return {"arquitectura": arquitectura, "hist": hist, "p_oof": p_oof, "mejor_epoca": mejor,
            "segundos": round(time.time() - t0, 1)}


def entrenar_final(arquitectura: str, epocas: int):
    """Modelo del aplicativo: se entrena con TODAS las imágenes durante las épocas elegidas en la validación cruzada."""
    media_n, std_n = normalizacion(arquitectura)
    fijar_semillas()
    modelo = crear_modelo(arquitectura).to(device)
    opt = crear_optimizador(modelo, arquitectura)
    scaler = torch.amp.GradScaler(device.type, enabled=USAR_AMP)
    loader = crear_loader(range(N), tf_entrenamiento(media_n, std_n), barajar=True)
    criterio = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "train_acc": []}
    for ep in range(epocas):
        r = una_epoca(modelo, loader, criterio, opt, scaler)
        hist["train_loss"].append(round(r["loss"], 4)); hist["train_acc"].append(round(r["acc"], 4))
        print(f"[final] Época {ep + 1:02d}/{epocas} | train_loss {r['loss']:.4f} acc {r['acc']:.3f}", flush=True)
    return modelo.eval(), hist


def calcular_metricas(y, p_pos, umbral: float = 0.5) -> dict:
    pred = (p_pos >= umbral).astype(int)
    return {
        "accuracy": round(accuracy_score(y, pred), 3),
        "precision": round(precision_score(y, pred, pos_label=1, zero_division=0), 3),
        "recall": round(recall_score(y, pred, pos_label=1, zero_division=0), 3),
        "f1": round(f1_score(y, pred, pos_label=1, zero_division=0), 3),
        "auc": round(roc_auc_score(y, p_pos), 3) if len(np.unique(y)) > 1 else None,
        "matriz_confusion": confusion_matrix(y, pred, labels=[0, 1]).tolist(),
        "n": int(len(y)),
    }


def resumen_cv(cv: dict) -> dict:
    """Métricas de las predicciones fuera de fold en la mejor época (cada imagen evaluada sin haberla visto)."""
    p = cv["p_oof"][cv["mejor_epoca"] - 1]
    global_ = calcular_metricas(etiquetas, p)
    por_fold = []
    for f, (_, va) in enumerate(folds):
        m = calcular_metricas(etiquetas[va], p[va])
        m["fold"] = f + 1
        por_fold.append(m)
    accs = [m["accuracy"] for m in por_fold]
    return {"global": global_, "por_fold": por_fold,
            "accuracy_media": round(float(np.mean(accs)), 3), "accuracy_std": round(float(np.std(accs)), 3)}


def tabla_por_imagen(cv: dict) -> pd.DataFrame:
    """Una fila por imagen: su clase real, el fold en que se evaluó y lo que predijo el modelo que no la vio."""
    p = cv["p_oof"][cv["mejor_epoca"] - 1]
    pred = (p >= 0.5).astype(int)
    filas = []
    for i, m in enumerate(muestras):
        conf = float(max(p[i], 1 - p[i]))
        filas.append({
            "archivo": m["archivo"], "ruta_relativa": m["rel"], "clase_real": m["clase"],
            "fold": int(FOLD_DE[i]) + 1, "p_positivo": round(float(p[i]), 4), "p_negativo": round(float(1 - p[i]), 4),
            "prediccion": CLASES[pred[i]], "acierto": bool(pred[i] == m["etiqueta"]), "confianza": round(conf, 4),
            "confianza_suficiente": conf >= config.UMBRAL_CONFIANZA_OBRA,
            "grupo_duplicado": int(grupos[i]) if int(grupos[i]) in duplicados else None,
            "ancho": m["ancho"], "alto": m["alto"],
        })
    tabla_df = pd.DataFrame(filas)
    tabla_df["grupo_duplicado"] = tabla_df["grupo_duplicado"].astype("Int64")   # entero o vacío
    return tabla_df


def graficar_cv(cv: dict, titulo: str):
    hist, ep = cv["hist"], np.arange(1, cv["hist"]["train_loss"].shape[1] + 1)
    plt.figure(figsize=(13, 4))
    for k, (clave, nombre) in enumerate([("loss", "Pérdida"), ("acc", "Accuracy")]):
        plt.subplot(1, 2, k + 1)
        for serie, etiqueta, color in [("train", "Entrenamiento", "#2a78d6"), ("val", "Validación (imágenes no vistas)", "#eb6834")]:
            datos = hist[f"{serie}_{clave}"]
            plt.plot(ep, datos.mean(0), color=color, label=etiqueta)
            plt.fill_between(ep, datos.mean(0) - datos.std(0), datos.mean(0) + datos.std(0), color=color, alpha=0.15)
        plt.axvline(cv["mejor_epoca"], color="#9aa0b4", ls="--", label=f"mejor época ({cv['mejor_epoca']})")
        if clave == "acc":
            plt.axhline(0.5, color="#c33", ls=":", lw=1, label="azar (50 %)")
        plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        plt.xlabel("Época"); plt.ylabel(nombre); plt.title(f"{titulo} — {nombre} (media ± desv. de {len(folds)} folds)")
        plt.legend(fontsize=8)
    plt.tight_layout(); plt.show()

### Entrenamiento (validación cruzada + modelo final con todas las imágenes)

In [ ]:
t0 = time.time()
NOMBRE_PRINCIPAL = {"resnet18": "ResNet18 (transfer learning)", "cnn_parcial2": "CNN Parcial 2"}[config.ARQUITECTURA]
resultados_cv = {}


def copiar_a_drive():
    """Colab borra /content al cerrar la sesión: se guarda una copia del modelo y los resultados en Drive."""
    if not config.DRIVE_PROYECTO.is_dir():
        return
    config.DRIVE_RESULTADOS.mkdir(parents=True, exist_ok=True)
    for ruta in [config.RUTA_CNN, config.RUTA_METRICAS, config.RUTA_PARTICION, config.RUTA_PREDICCIONES, config.RUTA_CV]:
        if ruta.exists():
            shutil.copy2(ruta, config.DRIVE_RESULTADOS / ruta.name)
    print("Copia guardada en Drive:", config.DRIVE_RESULTADOS)


# Si no se reentrena y el modelo solo está en Drive (sesión nueva de Colab), se recupera de allí
if not config.REENTRENAR and not config.RUTA_CNN.exists() and (config.DRIVE_RESULTADOS / config.RUTA_CNN.name).exists():
    for ruta in [config.RUTA_CNN, config.RUTA_METRICAS, config.RUTA_PARTICION, config.RUTA_PREDICCIONES, config.RUTA_CV]:
        if (config.DRIVE_RESULTADOS / ruta.name).exists():
            shutil.copy2(config.DRIVE_RESULTADOS / ruta.name, ruta)
    print("Modelo y resultados recuperados desde Drive.")

if config.REENTRENAR or not config.RUTA_CNN.exists():
    # 1) Validación cruzada: cada imagen recibe una predicción de un modelo que no la vio
    cv = validacion_cruzada(config.ARQUITECTURA, config.EPOCAS)
    resultados_cv[NOMBRE_PRINCIPAL] = cv
    resumen = resumen_cv(cv)
    g = resumen["global"]
    print(f"\nValidación cruzada ({N} imágenes) · accuracy {g['accuracy']} · F1 {g['f1']} · AUC {g['auc']}"
          f" · accuracy por fold {resumen['accuracy_media']} ± {resumen['accuracy_std']}")

    # 2) Modelo final con TODAS las imágenes, tantas épocas como la mejor de la validación cruzada
    print(f"\nEntrenando el modelo final con las {N} imágenes durante {cv['mejor_epoca']} épocas…")
    modelo_final, hist_final = entrenar_final(config.ARQUITECTURA, cv["mejor_epoca"])
    media_n, std_n = normalizacion(config.ARQUITECTURA)
    torch.save({"state_dict": {k: v.cpu() for k, v in modelo_final.state_dict().items()},
                "arquitectura": config.ARQUITECTURA, "media": list(media_n), "std": list(std_n), "clases": CLASES,
                "img_size": IMG_SIZE, "dropout": config.DROPOUT}, config.RUTA_CNN)

    # 3) Resultados imagen por imagen
    tabla = tabla_por_imagen(cv)
    tabla.to_csv(config.RUTA_PREDICCIONES, index=False, encoding="utf-8")
    np.savez_compressed(config.RUTA_CV, p_oof=cv["p_oof"], fold=FOLD_DE, etiquetas=etiquetas,
                        mejor_epoca=cv["mejor_epoca"], **{k: v for k, v in cv["hist"].items()})

    # 4) Métricas para el aplicativo («Acerca del modelo») y partición por nombre de archivo
    descripcion = ("ResNet18 preentrenada en ImageNet (se ajustan " + ", ".join(config.CAPAS_ENTRENABLES) +
                   " y la capa final) · Dropout 0.5 · Weight Decay 1e-4" if config.ARQUITECTURA == "resnet18"
                   else "CNN 3 bloques conv + Dropout 0.5 + Weight Decay 1e-4")
    with open(config.RUTA_METRICAS, "w", encoding="utf-8") as f:
        json.dump({"modelo": descripcion, "arquitectura": config.ARQUITECTURA,
                   "evaluacion": f"Validación cruzada estratificada de {len(folds)} folds: cada imagen se predijo con un modelo que no la vio",
                   "k_folds": len(folds), "mejor_epoca": cv["mejor_epoca"], "prueba": g,
                   "por_fold": resumen["por_fold"], "accuracy_media": resumen["accuracy_media"],
                   "accuracy_std": resumen["accuracy_std"],
                   "historial": {k: v.mean(axis=0).round(4).tolist() for k, v in cv["hist"].items()},
                   "historial_final": hist_final,
                   "particion": {"entrenamiento": N, "validacion": 0, "prueba": 0},
                   "imagenes": {c: int((etiquetas == i).sum()) for i, c in enumerate(CLASES)},
                   "descartadas": descartadas, "grupos_duplicados": len(duplicados),
                   "duplicados_contradictorios": len(contradictorios),
                   "media": list(media_n), "std": list(std_n), "segundos": round(time.time() - t0, 1)},
                  f, ensure_ascii=False, indent=2)
    with open(config.RUTA_PARTICION, "w", encoding="utf-8") as f:
        json.dump({"descripcion": "Imágenes usadas para entrenar la CNN final (todas) y su predicción en la validación cruzada",
                   "archivos": {r.archivo: {"particion": "entrenamiento", "clase": r.clase_real, "fold": int(r.fold),
                                            "prediccion_cv": r.prediccion, "p_positivo_cv": float(r.p_positivo)}
                                for r in tabla.itertuples()}},
                  f, ensure_ascii=False, indent=1)
    print(f"Modelo guardado en {config.RUTA_CNN} · resultados por imagen en {config.RUTA_PREDICCIONES}")
    copiar_a_drive()
else:
    print("Se reutiliza el modelo entrenado:", config.RUTA_CNN)
    datos_cv = np.load(config.RUTA_CV)
    cv = {"arquitectura": config.ARQUITECTURA, "mejor_epoca": int(datos_cv["mejor_epoca"]), "p_oof": datos_cv["p_oof"],
          "hist": {k: datos_cv[k] for k in ("train_loss", "val_loss", "train_acc", "val_acc")}}
    assert len(datos_cv["etiquetas"]) == N, "El dataset cambió desde el último entrenamiento: usa REENTRENAR = True."
    resultados_cv[NOMBRE_PRINCIPAL] = cv
    tabla = pd.read_csv(config.RUTA_PREDICCIONES)

if config.ENTRENAR_COMPARACION:
    print("\n--- Comparación: CNN del Parcial 2 con los mismos folds ---")
    epocas_cmp = config.EPOCAS_CNN
    resultados_cv["CNN Parcial 2"] = validacion_cruzada("cnn_parcial2", epocas_cmp) if config.ARQUITECTURA != "cnn_parcial2" else cv
print(f"\nEntrenamiento terminado en {time.time() - t0:.0f} s")

### PreprocesamientoImagen

In [ ]:
"""
PreprocesamientoImagen
----------------------
Función: adaptar la imagen al formato requerido por la CNN.
Entrada: imagen RGB (ImageLoader) de dimensiones variables.
Salida : tensor PyTorch float32 [1, 3, 224, 224]: redimensionado a 224x224, convertido a tensor
         y normalizado por canal. Se usan la media y desviación calculadas con el conjunto de
         entrenamiento (guardadas junto al modelo); si no existen, 0.5 como indica el diagrama.

También mide la calidad de la imagen (brillo y nitidez), retomando el análisis de
iluminación del Parcial 2.
"""


class PreprocesamientoImagen:
    def __init__(self, media=None, std=None, img_size: int = config.IMG_SIZE):
        self.media = np.array(media or config.MEDIA_DEFECTO, dtype=np.float32)
        self.std = np.array(std or config.STD_DEFECTO, dtype=np.float32)
        self.img_size = img_size

    def __call__(self, img_rgb: np.ndarray) -> torch.Tensor:
        im = Image.fromarray(img_rgb).resize((self.img_size, self.img_size), Image.BILINEAR)
        x = np.asarray(im, dtype=np.float32) / 255.0             # ToTensor: 0-255 -> 0-1
        x = (x - self.media) / self.std                          # Normalize por canal
        return torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0).contiguous()  # (1, 3, H, W)


def medir_calidad(img_rgb: np.ndarray) -> dict:
    """Brillo medio (0-255) y nitidez (varianza del Laplaciano) sobre una versión reducida."""
    h, w = img_rgb.shape[:2]
    escala = min(1.0, 512 / max(h, w))
    peq = cv2.resize(img_rgb, (max(1, int(w * escala)), max(1, int(h * escala))), interpolation=cv2.INTER_AREA)
    gris = cv2.cvtColor(peq, cv2.COLOR_RGB2GRAY)
    return {
        "brillo": round(float(gris.mean()), 1),
        "nitidez": round(float(cv2.Laplacian(gris, cv2.CV_64F).var()), 1),
        "ancho": int(w),
        "alto": int(h),
    }

### ClasificadorDeObras + Grad-CAM

In [ ]:
class ClasificadorDeObras:
    def __init__(self, ruta_pesos: Path):
        self.disponible = False
        self.error = None
        self.clases = ["negativo", "positivo"]
        self.media = None
        self.std = None
        self.modelo = None
        self.arquitectura = None
        # El modelo se comparte entre peticiones concurrentes (vista en vivo, galería del dataset):
        # todo uso de la red pasa por este candado para evitar carreras en PyTorch.
        self._lock = threading.Lock()
        try:
            ckpt = torch.load(ruta_pesos, map_location="cpu", weights_only=False)
            self.arquitectura = ckpt.get("arquitectura", "cnn_parcial2")   # los modelos antiguos eran la CNN
            self.modelo = crear_modelo(self.arquitectura, preentrenado=False,   # los pesos vienen del checkpoint
                                       dropout=ckpt.get("dropout", 0.5), img_size=ckpt.get("img_size", 224))
            self.modelo.load_state_dict(ckpt["state_dict"])
            self.modelo.eval()
            self.clases = ckpt.get("clases", self.clases)
            self.media = ckpt.get("media")
            self.std = ckpt.get("std")
            self.disponible = True
        except FileNotFoundError:
            self.error = f"No existe {ruta_pesos.name}. Ejecuta la celda de entrenamiento del notebook."
        except Exception as e:  # pesos corruptos o incompatibles
            self.error = f"No se pudo cargar la CNN: {e}"

    # ---------------------------------------------------------------
    def predecir(self, tensor: torch.Tensor) -> torch.Tensor:
        """Devuelve los logits [1, 2]."""
        with self._lock, torch.no_grad():
            return self.modelo(tensor)

    # ---------------------------------------------------------------
    def gradcam(self, tensor: torch.Tensor, clase: int) -> np.ndarray:
        """
        Mapa Grad-CAM normalizado 0-1 (tamaño 224x224) para la clase indicada.
        No usa hooks sobre el modelo compartido: se divide el paso hacia adelante en
        features(x) -> activaciones A del último bloque conv, y head(A) -> logits.
        ResNet18: A = (1, 512, 7, 7) · CNN del Parcial 2: A = (1, 64, 56, 56).
        """
        with self._lock, torch.enable_grad():
            a = self.modelo.features(tensor).detach().requires_grad_(True)
            out = self.modelo.head(a)
            grad = torch.autograd.grad(out[0, clase], a)[0][0]   # dY_c / dA
            pesos = grad.mean(dim=(1, 2))                         # importancia por canal
            cam = torch.relu((pesos[:, None, None] * a[0]).sum(0)).detach().numpy()
        cam = cv2.resize(cam, (tensor.shape[3], tensor.shape[2]))
        cam = cv2.GaussianBlur(cam, (0, 0), 6)
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam.astype(np.float32)


def cam_a_overlay(img_rgb: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> str:
    """Superpone el mapa sobre la imagen original y lo devuelve como data URL PNG."""
    h, w = img_rgb.shape[:2]
    escala = min(1.0, 640 / max(h, w))
    base = cv2.resize(img_rgb, (int(w * escala), int(h * escala)), interpolation=cv2.INTER_AREA)
    mapa = cv2.resize(cam, (base.shape[1], base.shape[0]))
    color = cv2.applyColorMap((mapa * 255).astype(np.uint8), cv2.COLORMAP_JET)
    color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)
    mezcla = (base * (1 - alpha) + color * alpha).astype(np.uint8)
    ok, buf = cv2.imencode(".jpg", cv2.cvtColor(mezcla, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 85])
    return "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode()


def regiones_desde_cam(cam: np.ndarray, max_regiones: int = 2, umbral: float = 0.6) -> list[dict]:
    """Convierte las zonas más activas del Grad-CAM en cajas normalizadas 0-1.
    Si la zona activa cubre casi toda la obra se sube el umbral para localizar el foco."""
    h, w = cam.shape
    for u in (umbral, 0.72, 0.84):
        mascara = (cam >= u).astype(np.uint8)
        if mascara.mean() < 0.45:
            break
    n, _, stats, _ = cv2.connectedComponentsWithStats(mascara, connectivity=8)
    cajas = []
    for i in range(1, n):
        x, y, bw, bh, area = stats[i]
        if area < 0.01 * h * w:
            continue
        intensidad = float(cam[y:y + bh, x:x + bw].mean())
        cajas.append({"x": float(x / w), "y": float(y / h), "w": float(bw / w), "h": float(bh / h), "intensidad": intensidad, "area": int(area)})
    cajas.sort(key=lambda c: c["area"] * c["intensidad"], reverse=True)
    out = []
    for i, c in enumerate(cajas[:max_regiones]):
        out.append({
            "x": round(c["x"], 4), "y": round(c["y"], 4), "w": round(c["w"], 4), "h": round(c["h"], 4),
            "etiqueta": "Zona de mayor influencia" if i == 0 else "Zona secundaria",
            "confianza": round(c["intensidad"], 3),
        })
    return out

### ExtractorDePaleta

In [ ]:
"""
ExtractorDePaleta
-----------------
Función: obtener los colores predominantes de la obra y estimar la proporción de la imagen
         que representa cada uno.
Entrada: imagen RGB validada por ImageLoader (matriz de píxeles, dimensiones variables).
Salida : lista de colores, cada uno con código HEX y proporción estimada (suman 1).

Método: K-Means en el espacio CIE Lab (perceptualmente más uniforme que RGB) sobre una versión
reducida de la imagen. Se añaden métricas cromáticas globales: brillo, saturación y calidez.
"""


# Nombres aproximados por tono (grados HSV) para describir la paleta en español
_TONOS = [(15, "rojo"), (40, "naranja"), (65, "amarillo"), (150, "verde"), (195, "cian"),
          (255, "azul"), (290, "violeta"), (335, "magenta"), (360, "rojo")]


def nombre_color(rgb) -> str:
    r, g, b = [c / 255 for c in rgb]
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    if l < 0.12:
        return "negro"
    if l > 0.92:
        return "blanco"
    if s < 0.15:
        return "gris oscuro" if l < 0.4 else ("gris claro" if l > 0.7 else "gris")
    grados = h * 360
    base = next(n for lim, n in _TONOS if grados <= lim)
    if base in ("naranja", "amarillo", "rojo") and s < 0.45:   # tonos tierra poco saturados
        return "beige" if l > 0.65 else ("marrón" if l < 0.5 else "ocre")
    if base in ("naranja", "rojo") and l < 0.35 and s < 0.6:
        base = "marrón"
    if l < 0.3:
        return f"{base} oscuro"
    if l > 0.75:
        return f"{base} claro"
    return base


def _es_calido(rgb) -> bool:
    h, l, s = colorsys.rgb_to_hls(*[c / 255 for c in rgb])
    grados = h * 360
    return s >= 0.15 and (grados < 75 or grados > 320)


class ExtractorDePaleta:
    def __init__(self, n_colores: int = config.N_COLORES, lado: int = config.LADO_MUESTREO_PALETA):
        self.k = n_colores
        self.lado = lado

    def __call__(self, img_rgb: np.ndarray) -> dict:
        h, w = img_rgb.shape[:2]
        escala = min(1.0, self.lado / max(h, w))
        peq = cv2.resize(img_rgb, (max(1, int(w * escala)), max(1, int(h * escala))), interpolation=cv2.INTER_AREA)
        lab = cv2.cvtColor(peq, cv2.COLOR_RGB2LAB).reshape(-1, 3).astype(np.float32)

        k = int(min(self.k, max(1, len(np.unique(lab, axis=0)))))
        criterio = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.5)
        cv2.setRNGSeed(42)
        _, etiquetas, centros = cv2.kmeans(lab, k, None, criterio, 4, cv2.KMEANS_PP_CENTERS)
        conteos = np.bincount(etiquetas.ravel(), minlength=k).astype(np.float64)
        proporciones = conteos / conteos.sum()

        centros_rgb = cv2.cvtColor(centros.reshape(1, -1, 3).astype(np.uint8), cv2.COLOR_LAB2RGB).reshape(-1, 3)
        orden = np.argsort(-proporciones)
        colores = []
        for i in orden:
            if proporciones[i] <= 0:
                continue
            rgb = [int(v) for v in centros_rgb[i]]
            colores.append({
                "hex": "#{:02X}{:02X}{:02X}".format(*rgb),
                "rgb": rgb,
                "proporcion": round(float(proporciones[i]), 4),
                "nombre": nombre_color(rgb),
            })

        hsv = cv2.cvtColor(peq, cv2.COLOR_RGB2HSV)
        calidez = sum(c["proporcion"] for c in colores if _es_calido(c["rgb"]))
        return {
            "colores": colores,
            "brillo_medio": round(float(hsv[..., 2].mean() / 255), 3),
            "saturacion_media": round(float(hsv[..., 1].mean() / 255), 3),
            "proporcion_calida": round(float(calidez), 3),
        }

### AnalizarGestosFaciales (detección, emoción y calificación del rostro)

In [ ]:
"""
AnalizarGestosFaciales
----------------------
Función: detectar el rostro y extraer características de los gestos faciales observables.
Entrada: fotograma RGB (CapturaDeFrame), con las dimensiones del fotograma recibido.
Salida : lista de detecciones faciales. Si no hay rostro: lista vacía y estado "sin_rostro".

Componentes:
  * Detector de rostros YuNet (OpenCV Zoo, ONNX) -> caja + 5 puntos (ojos, nariz, comisuras).
    Si el modelo no está disponible se usa el clasificador Haar que trae OpenCV.
  * Reconocedor de expresiones FER+ (ONNX Model Zoo) -> 8 emociones sobre el rostro 64x64 en gris.
  * Gestos geométricos a partir de los puntos faciales: inclinación de la cabeza, giro
    aproximado (yaw) y ancho de boca relativo a la distancia entre ojos (proxy de sonrisa).
"""


EMOCIONES = ["neutral", "felicidad", "sorpresa", "tristeza", "enojo", "asco", "miedo", "desprecio"]
EMOCIONES_ES = {
    "neutral": "Neutral", "felicidad": "Felicidad", "sorpresa": "Sorpresa", "tristeza": "Tristeza",
    "enojo": "Enojo", "asco": "Asco", "miedo": "Miedo", "desprecio": "Desprecio",
}
# Valencia aproximada de cada emoción (-1 desagradable … +1 agradable) usada para comparar con la obra
VALENCIA = {
    "felicidad": 1.0, "sorpresa": 0.3, "neutral": 0.0, "desprecio": -0.5,
    "miedo": -0.7, "tristeza": -0.8, "asco": -0.8, "enojo": -0.9,
}


def _softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


class AnalizarGestosFaciales:
    def __init__(self, ruta_yunet, ruta_ferplus):
        self._lock = threading.Lock()
        self.detector = None
        self.haar = None
        self.emociones = None
        self.tipo_detector = None
        self.errores = []

        try:
            if ruta_yunet.exists() and hasattr(cv2, "FaceDetectorYN"):
                self.detector = cv2.FaceDetectorYN.create(str(ruta_yunet), "", (320, 320), 0.6, 0.3, 5000)
                self.tipo_detector = "YuNet"
        except Exception as e:
            self.errores.append(f"YuNet no disponible: {e}")
        if self.detector is None:
            ruta_haar = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
            self.haar = cv2.CascadeClassifier(ruta_haar)
            self.tipo_detector = "Haar (respaldo)"

        try:
            if ruta_ferplus.exists():
                import onnxruntime as ort
                opciones = ort.SessionOptions()
                opciones.intra_op_num_threads = 2
                self.emociones = ort.InferenceSession(str(ruta_ferplus), opciones, providers=["CPUExecutionProvider"])
                self._entrada = self.emociones.get_inputs()[0].name
            else:
                self.errores.append(f"No existe {ruta_ferplus.name}. Ejecuta la celda de modelos de rostro y emociones.")
        except Exception as e:
            self.errores.append(f"FER+ no disponible: {e}")

    @property
    def disponible(self) -> bool:
        return self.emociones is not None

    # ------------------------------------------------------------------
    def _detectar(self, bgr: np.ndarray) -> list[dict]:
        h, w = bgr.shape[:2]
        caras = []
        if self.detector is not None:
            self.detector.setInputSize((w, h))
            _, res = self.detector.detect(bgr)
            for f in (res if res is not None else []):
                x, y, bw, bh = [float(v) for v in f[:4]]
                pts = f[4:14].reshape(5, 2).astype(float)  # ojo der., ojo izq., nariz, comisura der., comisura izq.
                caras.append({"caja": [x, y, bw, bh], "score": float(f[14]), "puntos": pts})
        else:
            gris = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
            for (x, y, bw, bh) in self.haar.detectMultiScale(gris, 1.1, 5, minSize=(40, 40)):
                caras.append({"caja": [float(x), float(y), float(bw), float(bh)], "score": 0.9, "puntos": None})
        return caras

    def _emocion(self, gris: np.ndarray, caja) -> np.ndarray | None:
        if self.emociones is None:
            return None
        h, w = gris.shape[:2]
        x, y, bw, bh = caja
        lado = max(bw, bh) * 1.15                       # recorte cuadrado con un pequeño margen
        cx, cy = x + bw / 2, y + bh / 2
        x0, y0 = int(max(0, cx - lado / 2)), int(max(0, cy - lado / 2))
        x1, y1 = int(min(w, cx + lado / 2)), int(min(h, cy + lado / 2))
        if x1 - x0 < 8 or y1 - y0 < 8:
            return None
        rec = cv2.resize(gris[y0:y1, x0:x1], (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32)
        logits = self.emociones.run(None, {self._entrada: rec[None, None]})[0][0]
        return _softmax(logits)

    @staticmethod
    def _gestos(puntos, caja) -> dict:
        if puntos is None:
            return {}
        ojo_d, ojo_i, nariz, boca_d, boca_i = puntos
        dx, dy = ojo_i - ojo_d
        dist_ojos = float(np.hypot(dx, dy)) or 1.0
        inclinacion = float(np.degrees(np.arctan2(dy, dx)))
        centro_ojos = (ojo_d + ojo_i) / 2
        giro = float((nariz[0] - centro_ojos[0]) / dist_ojos)          # ~0 de frente; ±0.3 girado
        ancho_boca = float(np.hypot(*(boca_i - boca_d)) / dist_ojos)   # aumenta al sonreír
        return {
            "inclinacion_cabeza_grados": round(inclinacion, 1),
            "giro_relativo": round(giro, 3),
            "ancho_boca_relativo": round(ancho_boca, 3),
            "mirando_al_frente": bool(abs(giro) < 0.25 and abs(inclinacion) < 20),
        }

    @staticmethod
    def calificar(gris: np.ndarray, caja_px, score: float, gestos: dict) -> dict:
        """
        Calificación del rostro (0-100): qué tan apto es el rostro para leer su expresión.
          30 % confianza del detector · 20 % tamaño en el cuadro · 20 % frontalidad
          15 % nitidez del recorte (varianza del Laplaciano) · 15 % iluminación del recorte
        """
        h, w = gris.shape[:2]
        x, y, bw, bh = caja_px
        x0, y0 = int(max(0, x)), int(max(0, y))
        x1, y1 = int(min(w, x + bw)), int(min(h, y + bh))
        recorte = gris[y0:y1, x0:x1]
        if recorte.size == 0:
            recorte = gris
        lado = cv2.resize(recorte, (96, 96), interpolation=cv2.INTER_AREA)
        nitidez = float(cv2.Laplacian(lado, cv2.CV_64F).var())
        brillo = float(lado.mean())
        comp = {
            "deteccion": float(np.clip(score, 0, 1)),
            "tamano": float(np.clip((bw / w) / 0.25, 0, 1)),
            "frontalidad": (float(np.clip(1 - abs(gestos["giro_relativo"]) / 0.35, 0, 1) *
                                  np.clip(1 - abs(gestos["inclinacion_cabeza_grados"]) / 30, 0, 1))
                            if gestos else 0.7),
            "nitidez": float(np.clip(nitidez / 150, 0, 1)),
            "iluminacion": float(np.clip(1 - abs(brillo - 128) / 128, 0, 1)),
        }
        pesos = {"deteccion": .30, "tamano": .20, "frontalidad": .20, "nitidez": .15, "iluminacion": .15}
        puntaje = round(100 * sum(comp[k] * pesos[k] for k in pesos), 1)
        nivel = "Excelente" if puntaje >= 80 else "Buena" if puntaje >= 60 else "Regular" if puntaje >= 40 else "Baja"
        return {"puntaje": puntaje, "nivel": nivel, "componentes": {k: round(v, 3) for k, v in comp.items()}}

    # ------------------------------------------------------------------
    def __call__(self, img_rgb: np.ndarray) -> dict:
        """Devuelve {'estado': 'ok'|'sin_rostro', 'detecciones': [...]} con cajas normalizadas 0-1."""
        h, w = img_rgb.shape[:2]
        bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        gris = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        with self._lock:
            caras = self._detectar(bgr)
            detecciones = []
            for c in caras:
                x, y, bw, bh = c["caja"]
                probs = self._emocion(gris, c["caja"])
                gestos = self._gestos(c["puntos"], c["caja"])
                det = {
                    "caja": {"x": round(max(0, x) / w, 4), "y": round(max(0, y) / h, 4),
                             "w": round(min(bw, w - max(0, x)) / w, 4), "h": round(min(bh, h - max(0, y)) / h, 4)},
                    "score_deteccion": round(c["score"], 3),
                    "gestos": gestos,
                    "calificacion": self.calificar(gris, c["caja"], c["score"], gestos),
                }
                if c["puntos"] is not None:
                    det["puntos"] = [[round(px / w, 4), round(py / h, 4)] for px, py in c["puntos"]]
                if probs is not None:
                    idx = int(np.argmax(probs))
                    det["probabilidades"] = {EMOCIONES[i]: round(float(p), 4) for i, p in enumerate(probs)}
                    det["emocion"] = EMOCIONES[idx]
                    det["confianza"] = round(float(probs[idx]), 4)
                detecciones.append(det)
        return {"estado": "ok" if detecciones else "sin_rostro", "detecciones": detecciones}


def valencia_visitante(probs: dict) -> dict:
    """v2: Σ p(e)·valencia(e) · v3: la misma suma dividida entre 1 − p(neutral) (neutral aporta 0) · intensidad."""
    v2 = float(sum(probs[e] * VALENCIA[e] for e in EMOCIONES))
    intensidad = float(np.clip(1 - probs["neutral"], 0, 1))
    v3 = float(np.clip(v2 / intensidad, -1, 1)) if intensidad > 1e-3 else 0.0
    return {"v2": round(v2, 4), "v3": round(v3, 4), "intensidad": round(intensidad, 4)}

### ResultadoProceso

In [ ]:
"""
ResultadoProceso
----------------
Función: convertir las salidas crudas en resultados utilizables y aplicar las reglas de
aceptación definidas para cada módulo.
Entrada: logits de la CNN (2 clases), lista de colores y proporciones, detecciones faciales.
Operaciones:
  * Softmax a los logits -> puntuaciones normalizadas; se elige la clase con mayor puntuación.
  * Umbral de confianza configurable -> si no se alcanza, "revisión requerida".
  * Filtrado de detecciones faciales según score y tamaño mínimo del rostro.
  * Validación de las proporciones y la estructura de la paleta.
  * Reglas de calidad (brillo y nitidez).
Salida: resultados procesados y estados de cada módulo + excepciones controladas.
"""


ETIQUETAS_OBRA = {"positivo": "Positiva", "negativo": "Negativa"}
DESCRIPCION_OBRA = {
    "positivo": "transmite una sensación agradable o serena",
    "negativo": "transmite una sensación inquietante o tensa",
}


def exc(tipo, mensaje, gravedad="aviso", modulo="ResultadoProceso"):
    """gravedad: 'error' (impide parte del resultado) · 'aviso' (conviene revisar) · 'info' (solo informa)."""
    return {"tipo": tipo, "gravedad": gravedad, "mensaje": mensaje, "modulo": modulo}


def nivel_calificacion(p: float) -> str:
    return "Excelente" if p >= 80 else "Buena" if p >= 60 else "Regular" if p >= 40 else "Baja"


class ResultadoProceso:
    def __init__(self, clases):
        self.clases = list(clases)

    # ------------------------------------------------------------- CNN
    def procesar_obra(self, logits: torch.Tensor) -> tuple[dict, list]:
        probs = torch.softmax(logits, dim=1)[0].tolist()
        idx = int(np.argmax(probs))
        categoria = self.clases[idx]
        conf = float(probs[idx])
        revision = conf < config.UMBRAL_CONFIANZA_OBRA
        out = {
            "categoria": categoria,
            "etiqueta": ETIQUETAS_OBRA.get(categoria, categoria),
            "descripcion": DESCRIPCION_OBRA.get(categoria, ""),
            "confianza": round(conf, 4),
            "puntuaciones": {c: round(float(p), 4) for c, p in zip(self.clases, probs)},
            "logits": [round(float(v), 4) for v in logits[0].tolist()],
            "valencia": round(float(probs[self.clases.index("positivo")] - probs[self.clases.index("negativo")]), 4),
            "revision_requerida": revision,
            "umbral": config.UMBRAL_CONFIANZA_OBRA,
        }
        excepciones = []
        if revision:
            excepciones.append(exc(
                "fuera_de_rango",
                f"La CNN no está segura de la categoría de la obra ({conf:.0%} < umbral {config.UMBRAL_CONFIANZA_OBRA:.0%}). "
                "El resultado se muestra, pero requiere revisión.", "aviso", "ClasificadorDeObras"))
        return out, excepciones

    # ------------------------------------------------------------- Paleta
    @staticmethod
    def validar_paleta(paleta: dict) -> list:
        errores = []
        colores = paleta.get("colores", [])
        suma = sum(c["proporcion"] for c in colores)
        if not colores:
            errores.append(exc("no_reconocido", "No se pudieron extraer colores de la obra.", "aviso", "ExtractorDePaleta"))
        elif abs(suma - 1) > 0.02:
            errores.append(exc("fuera_de_rango", f"Las proporciones de la paleta suman {suma:.2f} (se esperaba 1).", "aviso", "ExtractorDePaleta"))
        if any(not re.fullmatch(r"#[0-9A-F]{6}", c["hex"]) for c in colores):
            errores.append(exc("formato", "La paleta contiene códigos HEX inválidos.", "aviso", "ExtractorDePaleta"))
        return errores

    # ------------------------------------------------------------- Calidad
    @staticmethod
    def validar_calidad(calidad: dict, que: str, modulo: str) -> list:
        out = []
        if calidad["brillo"] < config.BRILLO_MIN:
            out.append(exc("calidad_insuficiente",
                           f"La imagen de {que} está muy oscura (brillo {calidad['brillo']:.0f}/255). Mejora la iluminación.",
                           "aviso", modulo))
        elif calidad["brillo"] > config.BRILLO_MAX:
            out.append(exc("calidad_insuficiente",
                           f"La imagen de {que} está sobreexpuesta (brillo {calidad['brillo']:.0f}/255).", "aviso", modulo))
        if calidad["nitidez"] < config.NITIDEZ_MIN:
            out.append(exc("calidad_insuficiente",
                           f"La imagen de {que} está borrosa (nitidez {calidad['nitidez']:.0f}). Mantén la cámara estable.",
                           "aviso", modulo))
        return out

    # ------------------------------------------------------------- Rostro
    @staticmethod
    def filtrar_detecciones(analisis: dict) -> tuple[dict | None, int]:
        validas = [d for d in analisis.get("detecciones", [])
                   if d["score_deteccion"] >= config.UMBRAL_CONFIANZA_ROSTRO and d["caja"]["w"] >= config.TAM_MIN_ROSTRO]
        if not validas:
            return None, 0
        principal = max(validas, key=lambda d: d["caja"]["w"] * d["caja"]["h"])
        return principal, len(validas)

    def procesar_secuencia(self, frames: list[dict]) -> tuple[dict | None, list]:
        """
        frames: [{'t': ms, 'analisis': {...}, 'calidad': {...}, 'indice': i}]
        Agrega la secuencia temporal: probabilidad media de cada emoción en los frames con rostro,
        línea de tiempo y frame representativo.
        """
        excepciones = []
        linea, probs_validas, principales, multiples = [], [], [], 0
        for f in frames:
            principal, n = self.filtrar_detecciones(f["analisis"])
            if n > 1:
                multiples += 1
            punto = {"t": f["t"], "rostro": principal is not None}
            if principal and "probabilidades" in principal:
                p = np.array([principal["probabilidades"][e] for e in EMOCIONES])
                probs_validas.append(p)
                principales.append((f, principal))
                punto.update({
                    "emocion": principal["emocion"],
                    "confianza": principal["confianza"],
                    "valencia": round(float(sum(p[i] * VALENCIA[e] for i, e in enumerate(EMOCIONES))), 4),
                })
            linea.append(punto)

        total = len(frames)
        con_rostro = len(probs_validas)
        if total and con_rostro == 0:
            excepciones.append(exc("rostro_no_detectado",
                                   "No se detectó ningún rostro. Ajusta la iluminación o la posición: la persona debe estar de frente y dentro del cuadro.",
                                   "error", "AnalizarGestosFaciales"))
            return None, excepciones
        if total and con_rostro / total < 0.5:
            excepciones.append(exc("rostro_no_detectado",
                                   f"El rostro solo se detectó en {con_rostro} de {total} fotogramas. Mantente frente a la cámara.",
                                   "aviso", "AnalizarGestosFaciales"))
        if multiples:
            excepciones.append(exc("multiples_rostros",
                                   f"Se detectó más de una persona en {multiples} fotograma(s); se analizó el rostro más cercano.",
                                   "aviso", "AnalizarGestosFaciales"))

        media = np.mean(probs_validas, axis=0)
        idx = int(np.argmax(media))
        emocion = EMOCIONES[idx]
        confianza = float(media[idx])
        if confianza < config.UMBRAL_EMOCION:
            excepciones.append(exc("fuera_de_rango",
                                   f"La expresión es ambigua: la emoción dominante ({EMOCIONES_ES[emocion]}) solo alcanza {confianza:.0%}.",
                                   "aviso", "AnalizarGestosFaciales"))

        # Frame representativo: el que más se parece a la emoción dominante
        f_rep, det_rep = max(principales, key=lambda fp: fp[1]["probabilidades"][emocion])
        gestos = [d["gestos"] for _, d in principales if d.get("gestos")]
        resumen_gestos = {}
        if gestos:
            for k in ("inclinacion_cabeza_grados", "giro_relativo", "ancho_boca_relativo"):
                resumen_gestos[k] = round(float(np.mean([g[k] for g in gestos])), 3)
            resumen_gestos["porcentaje_mirando_al_frente"] = round(float(np.mean([g["mirando_al_frente"] for g in gestos])), 3)

        valencia = float(sum(media[i] * VALENCIA[e] for i, e in enumerate(EMOCIONES)))
        vv = valencia_visitante({e: float(v) for e, v in zip(EMOCIONES, media)})
        puntajes = [d["calificacion"]["puntaje"] for _, d in principales if d.get("calificacion")]
        calificacion = None
        if puntajes:
            p_media = round(float(np.mean(puntajes)), 1)
            calificacion = {"puntaje": p_media, "minimo": round(float(np.min(puntajes)), 1),
                            "nivel": nivel_calificacion(p_media), "componentes": det_rep.get("calificacion", {}).get("componentes")}
        cambios = sum(1 for a, b in zip(principales, principales[1:]) if a[1]["emocion"] != b[1]["emocion"])
        rostro = {
            "emocion": emocion,
            "etiqueta": EMOCIONES_ES[emocion],
            "confianza": round(confianza, 4),
            "probabilidades": {e: round(float(v), 4) for e, v in zip(EMOCIONES, media)},
            "valencia": round(valencia, 4),
            "valencia_expresiva": vv["v3"],
            "intensidad": vv["intensidad"],
            "caja": det_rep["caja"],
            "puntos": det_rep.get("puntos"),
            "indice_frame": f_rep["indice"],
            "gestos": resumen_gestos,
            "calificacion": calificacion,
            "frames_analizados": total,
            "frames_con_rostro": con_rostro,
            "cambios_de_expresion": cambios,
            "linea_tiempo": linea,
        }
        return rostro, excepciones

    # ------------------------------------------------------------- Rostros dentro de una imagen
    @staticmethod
    def procesar_rostros_imagen(analisis: dict, que: str) -> tuple[dict, list]:
        """Filtra y ordena los rostros detectados dentro de la obra (subida o del dataset)."""
        validas = [d for d in analisis.get("detecciones", [])
                   if d["score_deteccion"] >= config.UMBRAL_ROSTRO_OBRA and d["caja"]["w"] >= config.TAM_MIN_ROSTRO_OBRA]
        validas.sort(key=lambda d: d["caja"]["w"] * d["caja"]["h"], reverse=True)
        for i, d in enumerate(validas, 1):
            d["id"] = i
            if "emocion" in d:
                d["etiqueta"] = EMOCIONES_ES[d["emocion"]]
                vv = valencia_visitante(d["probabilidades"])
                d["valencia"], d["valencia_expresiva"], d["intensidad"] = vv["v2"], vv["v3"], vv["intensidad"]
        excepciones = []
        if not validas:
            excepciones.append(exc("rostro_no_detectado",
                                   f"No se detectó ningún rostro en {que}. Se muestran la clasificación de la CNN y la paleta, "
                                   "pero no hay emoción ni calificación facial que reportar.", "info", "AnalizarGestosFaciales"))
        out = {"estado": "ok" if validas else "sin_rostro", "n": len(validas), "detecciones": validas,
               "principal": validas[0] if validas else None}
        if validas and all("emocion" in d for d in validas):
            conteo = {}
            for d in validas:
                conteo[d["emocion"]] = conteo.get(d["emocion"], 0) + 1
            out["emocion_predominante"] = max(conteo, key=conteo.get)
            out["calificacion_media"] = round(float(np.mean([d["calificacion"]["puntaje"] for d in validas])), 1)
        return out, excepciones

    # ------------------------------------------------------------- Comparación
    @staticmethod
    def coincidencia_detalle(obra: dict | None, rostro: dict | None) -> dict | None:
        """
        Coincidencia = 1 − |v_obra − v_visitante| / 2   (0 = opuestas · 1 = idénticas)

        v_obra = p(positivo) − p(negativo) de la CNN.
        v2 (original): v_visitante = Σ p(e)·valencia(e) con las 8 emociones. Una cara neutral vale 0, así que
            frente a una obra clasificada con seguridad la coincidencia queda fija en ~50 % aunque no haya evidencia.
        v3 (por defecto): v_visitante = v2 / (1 − p(neutral)), la valencia de la parte *expresiva* del rostro.
            Además se reporta la intensidad (1 − p(neutral)); si es baja, el resultado es poco concluyente.
        La evaluación con visitantes (sección 6) compara ambas fórmulas con lo que reportan las personas.
        """
        if not obra or not rostro or "probabilidades" not in rostro:
            return None
        vv = valencia_visitante(rostro["probabilidades"])
        c2 = round(1 - abs(obra["valencia"] - vv["v2"]) / 2, 4)
        c3 = round(1 - abs(obra["valencia"] - vv["v3"]) / 2, 4)
        formula = config.FORMULA_COINCIDENCIA if config.FORMULA_COINCIDENCIA in ("v2", "v3") else "v3"
        return {
            "formula": formula,
            "coincidencia": c3 if formula == "v3" else c2,
            "coincidencia_v2": c2, "coincidencia_v3": c3,
            "valencia_obra": round(float(obra["valencia"]), 4),
            "valencia_visitante": vv[formula], "valencia_visitante_v2": vv["v2"], "valencia_visitante_v3": vv["v3"],
            "intensidad": vv["intensidad"],
            "concluyente": vv["intensidad"] >= config.UMBRAL_EXPRESIVIDAD,
            "umbral_expresividad": config.UMBRAL_EXPRESIVIDAD,
        }

    def coincidencia(self, obra: dict | None, rostro: dict | None):
        d = self.coincidencia_detalle(obra, rostro)
        return d["coincidencia"] if d else None

### Integración
Reúne todos los módulos; `sistema` es la instancia que usan los resultados y el aplicativo.

In [ ]:
"""
Integración
-----------
Función: reunir los resultados en una estructura común para la GUI.
Entrada: obra (bytes opcionales) y secuencia de fotogramas con marca temporal (CapturaDeFrame).
Salida : categoría, puntuaciones, colores, datos faciales y estados de procesamiento;
         los errores se representan mediante códigos y mensajes controlados.

Flujo (según el diagrama del Momento 2):
  ImageLoader -> PreprocesamientoImagen -> ClasificadorDeObras ─┐
  ImageLoader -> ExtractorDePaleta ─────────────────────────────┼-> ResultadoProceso -> Integración -> GUI
  CámaraAutorizada -> CapturaDeFrame -> AnalizarGestosFaciales ─┘
  Integración -> RegistroEvaluacion (autorreporte del visitante, sección 6)
"""


def _data_url(img_rgb: np.ndarray, lado_max: int = 640) -> str:
    h, w = img_rgb.shape[:2]
    e = min(1.0, lado_max / max(h, w))
    peq = cv2.resize(img_rgb, (int(w * e), int(h * e)), interpolation=cv2.INTER_AREA) if e < 1 else img_rgb
    _, buf = cv2.imencode(".jpg", cv2.cvtColor(peq, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 85])
    return "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode()


def nivel(c):
    return "alta" if c >= 0.7 else "media" if c >= 0.4 else "baja"


class Sistema:
    """Instancia única que carga los modelos una vez y atiende todas las peticiones."""

    def __init__(self):
        self.clasificador = ClasificadorDeObras(config.RUTA_CNN)
        self.pre = PreprocesamientoImagen(self.clasificador.media, self.clasificador.std)
        self.paleta = ExtractorDePaleta()
        self.gestos = AnalizarGestosFaciales(config.RUTA_YUNET, config.RUTA_FERPLUS)
        self.resultado = ResultadoProceso(self.clasificador.clases)
        self.dataset = DatasetPrecarga()   # se crea tras entrenar: lee la partición nueva

    # ----------------------------------------------------------------- estado
    def estado(self) -> dict:
        return {
            "cnn": {"disponible": self.clasificador.disponible, "error": self.clasificador.error,
                    "media": self.pre.media.round(4).tolist(), "std": self.pre.std.round(4).tolist()},
            "rostro": {"detector": self.gestos.tipo_detector, "emociones": self.gestos.disponible,
                       "errores": self.gestos.errores},
            "umbrales": {
                "obra": config.UMBRAL_CONFIANZA_OBRA, "rostro": config.UMBRAL_CONFIANZA_ROSTRO,
                "emocion": config.UMBRAL_EMOCION, "tam_min_rostro": config.TAM_MIN_ROSTRO,
            },
        }

    # ----------------------------------------------------------------- en vivo
    def analizar_frame(self, datos: bytes) -> dict:
        """Análisis rápido de un fotograma para la vista en vivo (sin agregar)."""
        img = cargar_imagen(datos, "fotograma")
        analisis = self.gestos(img)
        principal, n = self.resultado.filtrar_detecciones(analisis)
        cal = medir_calidad(img)
        return {
            "estado": "ok" if principal else "sin_rostro",
            "rostros": n,
            "principal": principal,
            "detecciones": analisis["detecciones"],
            "calidad": cal,
            "excepciones": self.resultado.validar_calidad(cal, "la cámara", "CapturaDeFrame") if principal is None else [],
        }

    # ----------------------------------------------------------------- obra
    def _rostros_en_imagen(self, img: np.ndarray, que: str, excepciones: list, tiempos: dict) -> dict:
        """AnalizarGestosFaciales sobre la propia imagen (rostros pintados o fotografiados)."""
        t0 = time.perf_counter()
        h, w = img.shape[:2]
        e = min(1.0, config.LADO_MAX_DETECCION / max(h, w))
        peq = cv2.resize(img, (int(w * e), int(h * e)), interpolation=cv2.INTER_AREA) if e < 1 else img
        rostros, exc_r = self.resultado.procesar_rostros_imagen(self.gestos(peq), que)
        excepciones += exc_r
        tiempos["rostros_obra"] = round((time.perf_counter() - t0) * 1000, 1)
        return rostros

    def _analizar_obra(self, datos: bytes, excepciones: list, tiempos: dict, que: str = "la obra"):
        t0 = time.perf_counter()
        img = cargar_imagen(datos, que)
        tiempos["image_loader"] = round((time.perf_counter() - t0) * 1000, 1)
        calidad = medir_calidad(img)
        excepciones += self.resultado.validar_calidad(calidad, que, "ImageLoader")

        obra = {"dimensiones": {"ancho": calidad["ancho"], "alto": calidad["alto"]}, "calidad": calidad}
        if self.gestos.disponible:
            obra["rostros"] = self._rostros_en_imagen(img, que, excepciones, tiempos)

        t0 = time.perf_counter()
        paleta = self.paleta(img)
        tiempos["paleta"] = round((time.perf_counter() - t0) * 1000, 1)
        excepciones += self.resultado.validar_paleta(paleta)
        obra["paleta"] = paleta

        if not self.clasificador.disponible:
            excepciones.append(exc("servidor", self.clasificador.error or "La CNN no está disponible.", "error", "ClasificadorDeObras"))
            return obra
        t0 = time.perf_counter()
        tensor = self.pre(img)
        tiempos["preprocesamiento"] = round((time.perf_counter() - t0) * 1000, 1)
        t0 = time.perf_counter()
        logits = self.clasificador.predecir(tensor)
        tiempos["cnn"] = round((time.perf_counter() - t0) * 1000, 1)
        info, exc_obra = self.resultado.procesar_obra(logits)
        excepciones += exc_obra
        obra.update(info)
        obra["tensor"] = {"forma": list(tensor.shape), "tipo": str(tensor.dtype).replace("torch.", "")}

        t0 = time.perf_counter()
        cam = self.clasificador.gradcam(tensor, self.clasificador.clases.index(info["categoria"]))
        obra["gradcam"] = cam_a_overlay(img, cam)
        obra["regiones"] = regiones_desde_cam(cam)
        tiempos["gradcam"] = round((time.perf_counter() - t0) * 1000, 1)
        return obra

    # ----------------------------------------------------------------- rostro
    def _analizar_frames(self, frames: list[tuple[bytes, float]], excepciones: list, tiempos: dict):
        t0 = time.perf_counter()
        procesados, imagenes, ilegibles = [], [], 0
        for i, (datos, t) in enumerate(frames[: config.MAX_FRAMES]):
            try:
                img = cargar_imagen(datos, f"fotograma {i + 1}")
            except ErrorProceso:
                ilegibles += 1
                continue
            imagenes.append(img)
            procesados.append({"t": t, "indice": len(imagenes) - 1, "analisis": self.gestos(img), "calidad": medir_calidad(img)})
        tiempos["gestos"] = round((time.perf_counter() - t0) * 1000, 1)
        if ilegibles:
            excepciones.append(exc("formato", f"{ilegibles} fotograma(s) no se pudieron leer y se descartaron.", "aviso", "CapturaDeFrame"))
        if not procesados:
            excepciones.append(exc("servidor", "No llegó ningún fotograma válido de la cámara.", "error", "CapturaDeFrame"))
            return None
        if not self.gestos.disponible:
            excepciones.append(exc("servidor", "El reconocedor de emociones no está disponible. " + " ".join(self.gestos.errores),
                                   "error", "AnalizarGestosFaciales"))
            return None

        rostro, exc_r = self.resultado.procesar_secuencia(procesados)
        excepciones += exc_r
        brillo = float(np.mean([p["calidad"]["brillo"] for p in procesados]))
        nitidez = float(np.median([p["calidad"]["nitidez"] for p in procesados]))
        excepciones += self.resultado.validar_calidad({"brillo": brillo, "nitidez": nitidez}, "la cámara", "CapturaDeFrame")
        if rostro is None:
            return {"frame": _data_url(imagenes[len(imagenes) // 2]), "sin_rostro": True,
                    "frames_analizados": len(procesados), "frames_con_rostro": 0}
        rostro["frame"] = _data_url(imagenes[rostro.pop("indice_frame")])
        rostro["calidad"] = {"brillo": round(brillo, 1), "nitidez": round(nitidez, 1)}
        return rostro

    # ----------------------------------------------------------------- todo
    def analizar(self, obra_bytes: bytes | None, frames: list[tuple[bytes, float]], modo: str = "visitante",
                 que: str = "la obra") -> dict:
        """
        modo "visitante": obra + cámara (flujo original).
        modo "dataset"  : solo la imagen (subida o precargada); no exige la cámara.
        """
        t_total = time.perf_counter()
        excepciones, tiempos = [], {}
        obra = rostro = None

        if obra_bytes is not None:
            try:
                obra = self._analizar_obra(obra_bytes, excepciones, tiempos, que)
            except ErrorProceso as e:
                excepciones.append(e.a_dict())
        if frames:
            rostro = self._analizar_frames(frames, excepciones, tiempos)

        if modo == "dataset":
            return self._cerrar_dataset(obra, excepciones, tiempos, t_total, que)

        if obra_bytes is None or not frames:
            falta = "el rostro (cámara)" if obra_bytes is not None else "la obra"
            excepciones.append(exc("entrada_incompleta",
                                   f"Falta {falta}. Para calcular la coincidencia emocional se necesitan ambos.", "aviso", "Integración"))

        rostro_valido = rostro if rostro and not rostro.get("sin_rostro") else None
        obra_valida = obra if obra and "categoria" in obra else None
        detalle = self.resultado.coincidencia_detalle(obra_valida, rostro_valido)
        coincidencia = detalle["coincidencia"] if detalle else None
        if detalle and not detalle["concluyente"]:
            excepciones.append(exc(
                "expresion_neutral",
                f"La expresión fue mayormente neutral (intensidad {detalle['intensidad']:.0%} < {config.UMBRAL_EXPRESIVIDAD:.0%}). "
                "La coincidencia se calcula con la parte expresiva del rostro, pero es poco concluyente.",
                "info", "ResultadoProceso"))

        if obra_valida and rostro_valido:
            texto = (f"La obra se clasificó como {obra_valida['etiqueta'].lower()} ({obra_valida['confianza']:.0%}): "
                     f"{obra_valida['descripcion']}. El visitante expresó principalmente {rostro_valido['etiqueta'].lower()} "
                     f"({rostro_valido['confianza']:.0%}). La coincidencia emocional es {nivel(coincidencia)} ({coincidencia:.0%})"
                     + ("." if detalle["concluyente"] else ", aunque es poco concluyente porque la expresión fue mayormente neutral."))
        elif obra_valida:
            texto = f"La obra se clasificó como {obra_valida['etiqueta'].lower()} ({obra_valida['confianza']:.0%}). Falta el rostro para calcular la coincidencia."
        elif rostro_valido:
            texto = f"El visitante expresó principalmente {rostro_valido['etiqueta'].lower()} ({rostro_valido['confianza']:.0%}). Falta la obra para calcular la coincidencia."
        else:
            texto = "No se obtuvieron resultados válidos con las imágenes enviadas."

        hay_error = any(e["gravedad"] == "error" for e in excepciones)
        estado = "ok" if (obra_valida or rostro_valido) and not hay_error else ("parcial" if (obra_valida or rostro_valido) else "error")
        tiempos["total"] = round((time.perf_counter() - t_total) * 1000, 1)
        return {
            "modo": "visitante",
            "estado": estado,
            "obra": obra,
            "rostro": rostro,
            "coincidencia": coincidencia,
            "nivel_coincidencia": nivel(coincidencia) if coincidencia is not None else None,
            "coincidencia_detalle": detalle,
            "interpretacion": texto,
            "excepciones": excepciones,
            "tiempos_ms": tiempos,
        }

    # ----------------------------------------------------------------- imagen sola / dataset
    def _cerrar_dataset(self, obra, excepciones, tiempos, t_total, que):
        obra_valida = obra if obra and "categoria" in obra else None
        rostros = (obra or {}).get("rostros") or {}
        principal = rostros.get("principal")
        coherencia = None
        partes = []
        if obra_valida:
            partes.append(f"La CNN clasificó {que} como {obra_valida['etiqueta'].lower()} ({obra_valida['confianza']:.0%}).")
        if principal and "emocion" in principal:
            n = rostros["n"]
            partes.append(f"Se detect{'ó 1 rostro' if n == 1 else f'aron {n} rostros'}; el principal expresa "
                          f"{principal['etiqueta'].lower()} ({principal['confianza']:.0%}) con una calificación de "
                          f"{principal['calificacion']['puntaje']:.0f}/100 ({principal['calificacion']['nivel'].lower()}).")
            if obra_valida:
                coherencia = self.resultado.coincidencia(obra_valida, principal)   # misma fórmula que con el visitante
            if coherencia is not None:
                partes.append(f"Coherencia entre la categoría de la obra y la expresión del rostro: {nivel(coherencia)} ({coherencia:.0%}).")
        elif obra is not None and rostros:
            partes.append("No se detectaron rostros en la imagen.")
        texto = " ".join(partes) or "No se obtuvieron resultados válidos con la imagen enviada."
        hay_error = any(e["gravedad"] == "error" for e in excepciones)
        estado = "error" if not obra else ("parcial" if hay_error else "ok")
        tiempos["total"] = round((time.perf_counter() - t_total) * 1000, 1)
        return {
            "modo": "dataset",
            "estado": estado,
            "obra": obra,
            "rostro": None,
            "coincidencia": None,
            "nivel_coincidencia": None,
            "coherencia": coherencia,
            "nivel_coherencia": nivel(coherencia) if coherencia is not None else None,
            "interpretacion": texto,
            "excepciones": excepciones,
            "tiempos_ms": tiempos,
        }

    def analizar_dataset(self, nombre: str) -> dict:
        """Analiza una imagen precargada con el mismo flujo que una imagen subida."""
        datos = self.dataset.leer(nombre)          # ErrorProceso si no existe / no está en caché
        res = self.analizar(datos, [], modo="dataset", que="la imagen precargada")
        info = self.dataset.info_imagen(nombre)
        res["dataset"] = info
        if info["cnn"]:
            parte = {"entrenamiento": "entrenamiento", "validacion": "validación", "prueba": "prueba"}[info["cnn"]["particion"]]
            gravedad = "info" if info["cnn"]["particion"] == "prueba" else "aviso"
            res["excepciones"].append(exc(
                "imagen_conocida",
                f"Esta imagen formó parte del conjunto de {parte} de la CNN (etiqueta real: {info['cnn']['clase']}). "
                + ("Su predicción no es una evaluación independiente." if gravedad == "aviso" else "Sirve como evaluación independiente.")
                + (f" En la validación cruzada, sin haberla visto, se predijo «{info['cnn']['prediccion_cv']}» "
                   f"(probabilidad de positivo {info['cnn']['p_positivo_cv']:.0%})."
                   if info["cnn"].get("prediccion_cv") is not None else ""),
                gravedad, "ClasificadorDeObras"))
        return res


sistema = Sistema()
print(json.dumps(sistema.estado(), ensure_ascii=False, indent=1))

### RegistroEvaluacion
Guarda cada prueba con un visitante como una línea JSON (`registros_visitantes.jsonl`) en `Drive/Proyecto-final-de-PDI/evaluacion/`. Se guardan **solo números y respuestas**, nunca imágenes del rostro, como promete el consentimiento de la GUI. La GUI (formulario al final del resultado) y el procesamiento de videos (sección 6) usan el mismo formato.

In [ ]:
"""
RegistroEvaluacion
------------------
Función: almacenar el autorreporte del visitante junto a la salida del sistema para medir el desempeño real.
Entrada: resultado de Sistema.analizar (modo visitante) + respuestas del visitante.
Salida : una línea JSON por prueba en dir_evaluacion()/registros_visitantes.jsonl (sin imágenes).
"""
ESCALA_VALENCIA_OBRA = {-2: "Muy negativa", -1: "Negativa", 0: "Neutra", 1: "Positiva", 2: "Muy positiva"}
_lock_registro = threading.Lock()


def ruta_registros() -> Path:
    return dir_evaluacion() / "registros_visitantes.jsonl"


def _num(x, nd=4):
    return None if x is None else round(float(x), nd)


def resumen_resultado(r: dict) -> dict:
    """Extrae del resultado solo los valores numéricos necesarios para evaluar (descarta imágenes y Grad-CAM)."""
    r = r or {}
    obra, ro, det = r.get("obra") or {}, r.get("rostro") or {}, r.get("coincidencia_detalle") or {}
    obra_ok = "categoria" in obra
    ro_ok = bool(ro) and not ro.get("sin_rostro") and "probabilidades" in ro
    return {
        "estado": r.get("estado"),
        "obra": {"categoria": obra.get("categoria"), "confianza": _num(obra.get("confianza")),
                 "valencia": _num(obra.get("valencia"))} if obra_ok else None,
        "rostro": {"emocion": ro.get("emocion"), "confianza": _num(ro.get("confianza")),
                   "probabilidades": {e: _num(v) for e, v in (ro.get("probabilidades") or {}).items()},
                   "valencia_v2": _num(ro.get("valencia")), "valencia_v3": _num(ro.get("valencia_expresiva")),
                   "intensidad": _num(ro.get("intensidad")),
                   "frames_analizados": ro.get("frames_analizados"), "frames_con_rostro": ro.get("frames_con_rostro"),
                   "aptitud": _num((ro.get("calificacion") or {}).get("puntaje"), 1)} if ro_ok else None,
        "frames_analizados": ro.get("frames_analizados"),
        "frames_con_rostro": ro.get("frames_con_rostro", 0) if ro else 0,
        "coincidencia": _num(r.get("coincidencia")),
        "coincidencia_v2": _num(det.get("coincidencia_v2")), "coincidencia_v3": _num(det.get("coincidencia_v3")),
        "formula": det.get("formula"), "concluyente": det.get("concluyente"),
        "tiempos_ms": {k: _num(v, 1) for k, v in (r.get("tiempos_ms") or {}).items()},
        "excepciones": sorted({e.get("tipo") for e in r.get("excepciones", []) if isinstance(e, dict)}),
    }


def validar_autorreporte(a: dict) -> dict:
    """Valida y normaliza las respuestas; lanza ErrorProceso con un mensaje claro si algo falta."""
    def error(msg):
        raise ErrorProceso("formato", msg, "error", "RegistroEvaluacion")
    participante = str(a.get("participante", "")).strip()[:20]
    if not re.fullmatch(r"[\w\-]{1,20}", participante):
        error("El código del participante debe tener 1-20 letras, números, guion o guion bajo (p. ej. P01).")
    emocion = a.get("emocion")
    if emocion not in EMOCIONES:
        error("Selecciona la emoción que sentiste.")
    try:
        valencia_obra = int(a.get("valencia_obra"))
        assert valencia_obra in ESCALA_VALENCIA_OBRA
    except (TypeError, ValueError, AssertionError):
        error("Indica qué te transmitió la obra (de −2 a +2).")
    acuerdo = a.get("acuerdo_coincidencia")
    if acuerdo in ("", None):
        acuerdo = None
    else:
        try:
            acuerdo = int(acuerdo)
            assert 1 <= acuerdo <= 5
        except (TypeError, ValueError, AssertionError):
            error("El acuerdo con la coincidencia va de 1 a 5.")
    return {"participante": participante.upper(), "emocion": emocion, "valencia_obra": valencia_obra,
            "acuerdo_coincidencia": acuerdo, "comentario": str(a.get("comentario", "")).strip()[:300]}


def guardar_registro(autorreporte: dict, resultado: dict, obra_id: str, fuente: str = "gui", extra: dict | None = None) -> dict:
    reg = {"id": uuid.uuid4().hex[:12], "fecha": datetime.now().isoformat(timespec="seconds"), "fuente": fuente,
           "obra_id": str(obra_id or "")[:120], "autorreporte": validar_autorreporte(autorreporte),
           "sistema": resumen_resultado(resultado), **(extra or {})}
    with _lock_registro:
        with open(ruta_registros(), "a", encoding="utf-8") as f:
            f.write(json.dumps(reg, ensure_ascii=False) + "\n")
    return reg


def leer_registros() -> list[dict]:
    ruta = ruta_registros()
    if not ruta.exists():
        return []
    registros = []
    for k, linea in enumerate(ruta.read_text(encoding="utf-8").splitlines(), 1):
        if linea.strip():
            try:
                registros.append(json.loads(linea))
            except json.JSONDecodeError:
                print(f"⚠ Línea {k} de {ruta.name} dañada: se omite.")
    return registros


print("Registros de evaluación en:", ruta_registros(), f"({len(leer_registros())} guardados)")

## 4. Resultados
Todas las métricas salen de la **validación cruzada**: cada una de las imágenes se evaluó con un modelo que no la vio durante el entrenamiento. Se muestran curvas, métricas globales y por fold, matriz de confusión, curva ROC, la **tabla imagen por imagen** y un análisis completo de ejemplo con el mismo flujo del aplicativo. También se compara con una **línea base de solo color**, se analiza una imagen **no vista** y se mide la **latencia**.

### Curvas, métricas, matriz de confusión y ROC

In [ ]:
for nombre, cv_i in resultados_cv.items():
    graficar_cv(cv_i, nombre)

resumenes = {nombre: resumen_cv(cv_i) for nombre, cv_i in resultados_cv.items()}
for nombre, cv_i in resultados_cv.items():
    p = cv_i["p_oof"][cv_i["mejor_epoca"] - 1]
    pred = (p >= 0.5).astype(int)
    print(f"=== {nombre} · validación cruzada, {N} imágenes, época {cv_i['mejor_epoca']} ===")
    print(classification_report(etiquetas, pred, target_names=CLASES, zero_division=0, digits=3))
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    ConfusionMatrixDisplay(confusion_matrix(etiquetas, pred, labels=[0, 1]), display_labels=CLASES).plot(cmap="Blues", ax=ax[0], colorbar=False)
    ax[0].set_title(f"Matriz de confusión — {nombre}")
    RocCurveDisplay.from_predictions(etiquetas, p, ax=ax[1], name=nombre)
    ax[1].plot([0, 1], [0, 1], ls=":", color="#c33", label="azar"); ax[1].legend(fontsize=8)
    ax[1].set_title("Curva ROC")
    plt.tight_layout(); plt.show()

print("\nResultados por fold (modelo principal):")
print(pd.DataFrame(resumenes[NOMBRE_PRINCIPAL]["por_fold"])[["fold", "n", "accuracy", "precision", "recall", "f1", "auc"]].to_string(index=False))

print(f"\n{'Modelo':30s} {'Accuracy':>9s} {'± fold':>7s} {'Precision':>10s} {'Recall':>7s} {'F1':>6s} {'AUC':>6s}")
for nombre, r in resumenes.items():
    g = r["global"]
    print(f"{nombre:30s} {g['accuracy']:9.3f} {r['accuracy_std']:7.3f} {g['precision']:10.3f} {g['recall']:7.3f} {g['f1']:6.3f} {g['auc']:6.3f}")

### Línea base: clasificador de solo color
Para saber cuánto aporta la CNN frente a una regla simple, se entrena una regresión logística **solo con rasgos de color**: histograma HSV de 8×4×4 + brillo, saturación y *colorfulness*. Usa **los mismos 5 folds**. Si la CNN no la superara, el contenido de la imagen no estaría aportando nada más allá de la paleta.

In [ ]:
def rasgos_color(px: np.ndarray) -> np.ndarray:
    """Histograma HSV normalizado (128 valores) + 4 estadísticos globales de color."""
    hsv = cv2.cvtColor(px, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 4, 4], [0, 180, 0, 256, 0, 256]).flatten()
    hist /= max(hist.sum(), 1)
    r, g, b = [px[..., k].astype(float) for k in range(3)]
    rg, yb = r - g, 0.5 * (r + g) - b
    colorfulness = np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean())   # Hasler & Süsstrunk (2003)
    extra = [hsv[..., 2].mean() / 255, hsv[..., 1].mean() / 255, hsv[..., 2].std() / 255, colorfulness / 100]
    return np.concatenate([hist, extra])


X_color = np.stack([rasgos_color(m["pixeles"]) for m in muestras])
p_color = np.zeros(N)
for tr, va in folds:
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000))
    clf.fit(X_color[tr], etiquetas[tr])
    p_color[va] = clf.predict_proba(X_color[va])[:, 1]

NOMBRE_BASELINE = "Línea base de color (reg. logística)"
m_color = calcular_metricas(etiquetas, p_color)
accs_color = [calcular_metricas(etiquetas[va], p_color[va])["accuracy"] for _, va in folds]
resumenes[NOMBRE_BASELINE] = {"global": m_color, "accuracy_media": round(float(np.mean(accs_color)), 3),
                               "accuracy_std": round(float(np.std(accs_color)), 3), "por_fold": []}

print(f"{'Modelo':38s} {'Accuracy':>9s} {'± fold':>7s} {'Precision':>10s} {'Recall':>7s} {'F1':>6s} {'AUC':>6s}")
for nombre, r in resumenes.items():
    g = r["global"]
    print(f"{nombre:38s} {g['accuracy']:9.3f} {r['accuracy_std']:7.3f} {g['precision']:10.3f} {g['recall']:7.3f} {g['f1']:6.3f} {g['auc']:6.3f}")

pd.DataFrame([{"modelo": n, **{k: r["global"][k] for k in ("accuracy", "precision", "recall", "f1", "auc")},
               "accuracy_std_folds": r["accuracy_std"]} for n, r in resumenes.items()]
             ).to_csv(config.DIR_RESULTADOS / "comparacion_modelos.csv", index=False, encoding="utf-8")

plt.figure(figsize=(8, 3.6))
nombres_m = list(resumenes)
plt.barh(nombres_m, [resumenes[n]["global"]["accuracy"] for n in nombres_m],
         xerr=[resumenes[n]["accuracy_std"] for n in nombres_m], color=["#232B5D", "#7D839F", "#C9CCD8"][:len(nombres_m)])
plt.axvline(0.5, ls=":", color="#c33", label="azar")
plt.xlim(0.4, 1); plt.xlabel("Accuracy (validación cruzada, ± desviación entre folds)"); plt.legend()
plt.title("Comparación de modelos con los mismos 5 folds"); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

### Resultados imagen por imagen
Cada fila es una imagen de `positivo/` o `negativo/` con la predicción del modelo que **no la vio**. La tabla completa queda en `app_pdi/modelos/predicciones_por_imagen.csv` (y en Drive, en `resultados_cnn/`). Con `detalle_imagen("nombre.jpg")` puedes revisar cualquier imagen individualmente.

In [ ]:
tabla_cv = tabla.copy()
print("Aciertos por clase real:")
print(tabla_cv.groupby("clase_real")["acierto"].agg(imagenes="count", aciertos="sum", accuracy="mean").round(3).to_string())
print(f"\nPredicciones con confianza ≥ {config.UMBRAL_CONFIANZA_OBRA:.0%}: {tabla_cv['confianza_suficiente'].mean():.1%} de las imágenes"
      f" · accuracy en ellas: {tabla_cv.loc[tabla_cv['confianza_suficiente'], 'acierto'].mean():.3f}")

# Distribución de la probabilidad de "positivo" según la carpeta real
plt.figure(figsize=(9, 3.6))
for clase, color in [("negativo", "#eb6834"), ("positivo", "#2a78d6")]:
    plt.hist(tabla_cv.loc[tabla_cv["clase_real"] == clase, "p_positivo"], bins=20, range=(0, 1), alpha=0.6,
             color=color, label=f"carpeta {clase}")
plt.axvline(0.5, color="#232B5D", ls="--", lw=1)
plt.xlabel("Probabilidad de 'genera sentimientos positivos'"); plt.ylabel("Imágenes"); plt.legend()
plt.title("Separación entre clases (cuanto menos se solapen, mejor)"); plt.show()

columnas = ["archivo", "clase_real", "fold", "prediccion", "p_positivo", "confianza", "acierto"]
print("\nPrimeras 20 filas de la tabla (ordenadas por clase y archivo):")
print(tabla_cv.sort_values(["clase_real", "archivo"])[columnas].head(20).to_string(index=False))


def galeria(filas: pd.DataFrame, titulo: str, n: int = 12):
    filas = filas.head(n)
    if filas.empty:
        print(titulo, "→ ninguna"); return
    cols = 6
    nf = (len(filas) + cols - 1) // cols
    plt.figure(figsize=(18, 3.5 * nf))
    por_nombre = {m["rel"]: m for m in muestras}
    for k, r in enumerate(filas.itertuples()):
        plt.subplot(nf, cols, k + 1)
        plt.imshow(por_nombre[r.ruta_relativa]["pixeles"])
        plt.title(f"{r.archivo}\nreal: {r.clase_real} · pred: {r.prediccion}\np(pos) = {r.p_positivo:.2f}", fontsize=8,
                  color="#2F9E6A" if r.acierto else "#C0392B")
        plt.axis("off")
    plt.suptitle(titulo); plt.tight_layout(); plt.show()


errores = tabla_cv[~tabla_cv["acierto"]].sort_values("confianza", ascending=False)
print(f"\n{len(errores)} errores de {N} imágenes")
galeria(errores, "Errores más seguros (revisar si la etiqueta de la carpeta es la adecuada)")
for clase in CLASES:
    ok = tabla_cv[(tabla_cv["acierto"]) & (tabla_cv["clase_real"] == clase)].sort_values("confianza", ascending=False)
    galeria(ok, f"Aciertos más seguros — carpeta {clase}", n=6)
galeria(tabla_cv.assign(d=(tabla_cv["p_positivo"] - 0.5).abs()).sort_values("d"), "Imágenes más dudosas (p ≈ 0.5)", n=6)


def detalle_imagen(nombre: str):
    """Muestra el resultado individual de una imagen: predicción en la validación cruzada y Grad-CAM del modelo final."""
    filas = tabla_cv[(tabla_cv["archivo"] == nombre) | (tabla_cv["ruta_relativa"] == nombre)]
    if filas.empty:
        print("No está en positivo/ ni negativo/:", nombre); return
    r = filas.iloc[0]
    m = next(x for x in muestras if x["rel"] == r["ruta_relativa"])
    img = cargar_imagen(m["ruta"].read_bytes(), m["archivo"], permitir_formatos={"JPEG", "PNG", "BMP", "WEBP", "MPO"})
    tensor = sistema.pre(img)
    p_final = torch.softmax(sistema.clasificador.predecir(tensor), dim=1)[0, 1].item()
    cam = sistema.clasificador.gradcam(tensor, int(p_final >= 0.5))
    gc = cargar_imagen(base64.b64decode(cam_a_overlay(img, cam).split(",", 1)[1]), permitir_formatos={"JPEG"})
    print(f"{r['ruta_relativa']} · carpeta: {r['clase_real']} · fold {r['fold']}")
    print(f"  Validación cruzada (sin verla): {r['prediccion']} · p(positivo) = {r['p_positivo']:.3f} · "
          f"{'ACIERTO' if r['acierto'] else 'ERROR'}")
    print(f"  Modelo final (la usó para entrenar): p(positivo) = {p_final:.3f}")
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
    ax[0].imshow(img); ax[0].set_title(f"Real: {r['clase_real']}")
    ax[1].imshow(gc); ax[1].set_title("Grad-CAM (modelo final)")
    for a in ax:
        a.axis("off")
    plt.tight_layout(); plt.show()

### Análisis de ejemplo con el flujo completo (imagen no vista)
El ejemplo usa una imagen de `precarga-sin-analizar`, que **no participó en el entrenamiento**, para que la predicción sea una evaluación independiente. El modelo final se entrenó con las 500 imágenes de `positivo/` y `negativo/`, así que sobre ellas casi siempre acierta (las memorizó). Por eso, en esas imágenes la cifra válida es la de la validación cruzada.

In [ ]:
# Análisis completo de una imagen NO vista en el entrenamiento, con el mismo flujo del aplicativo.
def muestra_precarga(n: int) -> list[str]:
    """Nombres de n imágenes de precarga-sin-analizar elegidas con la semilla del proyecto (reproducible)."""
    archivos = sorted(sistema.dataset.asegurar()["archivos"])     # ErrorProceso si no hay carpeta
    return random.Random(SEED).sample(archivos, min(n, len(archivos)))


def elegir_imagen_demo(n_candidatas: int = 40):
    """Prefiere una imagen de precarga-sin-analizar con rostro; si no hay carpeta, usa una del dataset y lo advierte."""
    try:
        nombres = muestra_precarga(n_candidatas)
        fuentes = [(n, sistema.dataset.leer(n), "precarga") for n in nombres]
    except ErrorProceso as e:
        print("⚠ No hay imágenes precargadas:", e.mensaje, "→ se usa una imagen del dataset de entrenamiento.")
        jpg_png = [m for m in muestras if m["ruta"].suffix.lower() in {".jpg", ".jpeg", ".png"}]
        fuentes = [(m["archivo"], m["ruta"].read_bytes(), "dataset")
                   for m in random.Random(SEED).sample(jpg_png, min(n_candidatas, len(jpg_png)))]
    for nombre, datos, fuente in fuentes:
        try:
            if sistema.gestos(cargar_imagen(datos))["estado"] == "ok":
                return nombre, datos, fuente
        except ErrorProceso:
            continue
    return fuentes[0]


nombre_demo, datos_demo, fuente_demo = elegir_imagen_demo()
res = sistema.analizar(datos_demo, [], modo="dataset", que="la imagen")
obra = res["obra"]
print("Imagen:", nombre_demo, "· origen:", "precarga-sin-analizar (no vista por la CNN)" if fuente_demo == "precarga" else "dataset de entrenamiento")
if fuente_demo == "dataset":
    fila = tabla_cv[tabla_cv["archivo"] == nombre_demo].iloc[0]
    print(f"  Etiqueta real: {fila['clase_real']} · validación cruzada (sin verla): {fila['prediccion']} "
          f"(p_positivo = {fila['p_positivo']:.2f}) · la predicción del modelo final NO es independiente.")
print(res["interpretacion"])
print("Excepciones:", [(e["tipo"], e["gravedad"]) for e in res["excepciones"]] or "ninguna")
print("Tiempos (ms):", res["tiempos_ms"])

img = cargar_imagen(datos_demo)
h, w = img.shape[:2]
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].imshow(img); ax[0].set_title("Rostros detectados en la obra")
for d in obra.get("rostros", {}).get("detecciones", []):
    c = d["caja"]
    ax[0].add_patch(plt.Rectangle((c["x"] * w, c["y"] * h), c["w"] * w, c["h"] * h, fill=False, color="#2F9E6A", lw=2))
    ax[0].text(c["x"] * w, c["y"] * h - 4, f"#{d['id']} {d.get('etiqueta', '')} · aptitud {d['calificacion']['puntaje']:.0f}/100",
               color="white", backgroundcolor="#2F9E6A", fontsize=8)
if obra.get("gradcam"):
    gc = cargar_imagen(base64.b64decode(obra["gradcam"].split(",", 1)[1]), permitir_formatos={"JPEG"})
    ax[1].imshow(gc); ax[1].set_title(f"Grad-CAM · CNN: {obra['etiqueta']} {obra['confianza']:.0%}")
cols = obra["paleta"]["colores"]
ax[2].barh(range(len(cols)), [c["proporcion"] for c in cols], color=[c["hex"] for c in cols], edgecolor="#232B5D")
ax[2].set_yticks(range(len(cols)), [f"{c['hex']} · {c['nombre']}" for c in cols]); ax[2].invert_yaxis()
ax[2].set_xlabel("Proporción"); ax[2].set_title("Paleta cromática (K-Means en CIE Lab)")
for a in ax[:2]:
    a.axis("off")
fig.suptitle("Ejemplo del flujo completo con una imagen no vista"); plt.tight_layout(); plt.show()
print("La «aptitud» (0-100) indica qué tan legible es el rostro para leer su expresión: 30 % confianza del detector, "
      "20 % tamaño, 20 % frontalidad, 15 % nitidez y 15 % iluminación.")

### Rendimiento del sistema (latencia y fps)
Mide el tiempo de cada módulo sobre `BENCH_N` imágenes no vistas (flujo de obra) y los fotogramas por segundo del análisis facial (vista en vivo). Son métricas de **funcionamiento** del sistema, complementarias a las de precisión. Ten en cuenta el hardware: en Colab con GPU T4, la CNN corre en GPU y FER+ en CPU.

In [ ]:
# Latencia por módulo (flujo de obra) y fps del análisis facial
def imagenes_para_benchmark(n: int):
    try:
        nombres = muestra_precarga(n)
        return [sistema.dataset.leer(x) for x in nombres], "precarga-sin-analizar"
    except ErrorProceso:
        jpg_png = [m for m in muestras if m["ruta"].suffix.lower() in {".jpg", ".jpeg", ".png"}]
        return [m["ruta"].read_bytes() for m in random.Random(SEED).sample(jpg_png, min(n, len(jpg_png)))], "dataset"


def a_jpeg(img_rgb: np.ndarray, ancho: int = 640) -> bytes:
    h, w = img_rgb.shape[:2]
    e = ancho / w
    peq = cv2.resize(img_rgb, (ancho, max(1, int(h * e))), interpolation=cv2.INTER_AREA)
    return cv2.imencode(".jpg", cv2.cvtColor(peq, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 85])[1].tobytes()


imgs_bench, origen_bench = imagenes_para_benchmark(config.BENCH_N)
sistema.analizar(imgs_bench[0], [], modo="dataset")            # calentamiento (carga perezosa, cachés)
tiempos_bench = [sistema.analizar(b, [], modo="dataset")["tiempos_ms"] for b in imgs_bench]
df_lat = pd.DataFrame(tiempos_bench)
lat = pd.DataFrame({"media_ms": df_lat.mean(), "std_ms": df_lat.std(), "p95_ms": df_lat.quantile(0.95)}).round(1)

# fps del análisis facial: fotogramas de 640 px de ancho, como los que envía la GUI en vivo
frames_bench = [a_jpeg(cargar_imagen(imgs_bench[i % len(imgs_bench)])) for i in range(config.BENCH_FRAMES)]
t_frames = []
for fb in frames_bench:
    t0 = time.perf_counter(); sistema.analizar_frame(fb); t_frames.append((time.perf_counter() - t0) * 1000)
t_frames = np.array(t_frames)
rendimiento = {
    "dispositivo": f"{device.type}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""),
    "origen_imagenes": origen_bench, "n_imagenes": len(imgs_bench),
    "latencia_ms": lat.to_dict(orient="index"),
    "frame_ms": {"media": round(float(t_frames.mean()), 1), "std": round(float(t_frames.std()), 1),
                 "p95": round(float(np.percentile(t_frames, 95)), 1)},
    "fps_analisis_facial": round(float(1000 / t_frames.mean()), 1),
}
print(f"Latencia por módulo · {len(imgs_bench)} imágenes de {origen_bench} · {rendimiento['dispositivo']}")
print(lat.to_string())
print(f"\nAnálisis facial por fotograma: {rendimiento['frame_ms']['media']} ± {rendimiento['frame_ms']['std']} ms "
      f"→ {rendimiento['fps_analisis_facial']} fps (sin contar la red ni el navegador)")
with open(config.DIR_RESULTADOS / "rendimiento.json", "w", encoding="utf-8") as f:
    json.dump(rendimiento, f, ensure_ascii=False, indent=2)

modulos_lat = [k for k in lat.index if k != "total"]
plt.figure(figsize=(8, 3.6))
plt.barh(modulos_lat, lat.loc[modulos_lat, "media_ms"], xerr=lat.loc[modulos_lat, "std_ms"], color="#232B5D")
plt.xlabel(f"ms (media ± desviación) · total {lat.loc['total', 'media_ms']:.0f} ms por obra"); plt.gca().invert_yaxis()
plt.title("Latencia por módulo del flujo de obra"); plt.tight_layout(); plt.show()

## 5. Aplicativo web
La interfaz (HTML, CSS y JavaScript) vive dentro del notebook y la sirve FastAPI. Las celdas siguientes levantan el servidor en segundo plano y dan el enlace para abrirlo:

- **Obra y visitante**: sube una obra, activa o detén la cámara, mira la obra unos segundos y compara la emoción de la obra con la tuya.
- **Imágenes precargadas**: imágenes aleatorias de `Mi unidad/Proyecto-final-de-PDI/dataset usado/precarga-sin-analizar` analizadas con el mismo flujo. En el paso 1 también está el botón «Usar una imagen precargada» para quien no quiera subir una imagen propia.

Para detener el servidor ejecuta `detener_servidor()` en una celda nueva. Si exportas el notebook a `.py` y lo ejecutas con `python`, el servidor queda en primer plano (se detiene con Ctrl+C).
- **Evaluación del sistema**: al final de cada resultado del modo *Obra y visitante* aparece un formulario para registrar lo que el visitante sintió (sección 6). Solo se guardan números y respuestas, nunca imágenes.

In [ ]:
# Interfaz web: index.html (se sirve desde memoria, no se escribe en disco)
INDEX_HTML = r'''<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<meta name="description" content="Coincidencia emocional entre obra y visitante: CNN para clasificar obras, YuNet + FER+ para gestos faciales, integrados en una sola lectura de valencia.">
<meta name="theme-color" content="#F3F4F8" media="(prefers-color-scheme: light)">
<meta name="theme-color" content="#0E1230" media="(prefers-color-scheme: dark)">
<meta name="color-scheme" content="light dark">
<title>Coincidencia emocional · Obra y visitante</title>
<link rel="icon" href="data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 32 32'%3E%3Crect width='32' height='32' rx='8' fill='%23232B5D'/%3E%3Ccircle cx='16' cy='16' r='7' fill='none' stroke='white' stroke-width='2.5'/%3E%3C/svg%3E">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Bricolage+Grotesque:wght@500;700&family=Instrument+Sans:wght@400;500;600&display=swap" rel="stylesheet">
<link rel="stylesheet" href="styles.css">
</head>
<body>
<a class="skip-link" href="#contenido">Saltar al contenido principal</a>
<main class="wrap" id="contenido">
  <header class="top">
    <div>
      <p class="eyebrow">Procesamiento Digital de Imágenes · UAO</p>
      <h1>Coincidencia emocional entre obra y visitante</h1>
      <p class="sub">Carga una obra, autoriza la cámara y obsérvala unos segundos. El sistema clasifica la obra con una CNN, extrae su paleta cromática, analiza tus gestos faciales y compara ambas lecturas.</p>
    </div>
    <div class="top-side">
      <ul class="status" id="status" aria-label="Estado de los modelos">
        <li data-k="cnn"><i></i>CNN de obras</li>
        <li data-k="det"><i></i>Detector de rostro</li>
        <li data-k="emo"><i></i>Emociones FER+</li>
      </ul>
      <button class="ghost small" id="btnAcerca">Acerca del modelo</button>
    </div>
  </header>

  <div class="msg err banner" id="bannerServidor" role="alert" hidden></div>

  <nav class="modos" role="tablist" aria-label="Modo de trabajo">
    <button role="tab" id="tabVisitante" aria-selected="true" aria-controls="vistaVisitante">Obra y visitante <small>Sube una obra y usa la cámara</small></button>
    <button role="tab" id="tabDataset" aria-selected="false" aria-controls="vistaDataset" tabindex="-1">Imágenes precargadas <small>Imágenes aleatorias de la carpeta del proyecto</small></button>
  </nav>

  <div id="vistaVisitante" role="tabpanel" aria-labelledby="tabVisitante" tabindex="0">
  <ol class="stepper" id="stepper" aria-label="Progreso">
    <li><strong>1 · Entrada</strong><span>Obra y cámara</span></li>
    <li><strong>2 · Análisis</strong><span>Procesando</span></li>
    <li><strong>3 · Excepciones</strong><span>Revisión</span></li>
    <li><strong>4 · Resultado</strong><span>Comparación</span></li>
  </ol>

  <!-- ===================== PASO 1 ===================== -->
  <section class="panel" id="paso1" aria-labelledby="t1">
    <div class="panel-head">
      <h2 id="t1">Entrada</h2>
      <p class="lead">Necesitas al menos una de las dos entradas; con ambas se calcula la coincidencia emocional.</p>
    </div>
    <div class="grid2">
      <!-- Obra -->
      <div class="slot">
        <div class="slot-head"><h3>Imagen de la obra</h3><span class="pill" id="pillObra">Pendiente</span></div>
        <div class="drop" id="drop" tabindex="0" role="button" aria-label="Seleccionar imagen de la obra">
          <div>
            <svg width="40" height="40" viewBox="0 0 24 24" aria-hidden="true"><path fill="none" stroke="currentColor" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round" d="M4 16l4-4 4 4 3-3 5 5M4 4h16v16H4zM15 8.5a1.5 1.5 0 1 0 0 .01"/></svg>
            <p><b>Arrastra tu imagen aquí</b> o haz clic para elegirla</p>
            <small>JPG o PNG, hasta 10 MB</small>
          </div>
        </div>
        <input type="file" id="fileObra" accept="image/jpeg,image/png" hidden>
        <div class="preview" id="prevObra" hidden><img id="imgObra" alt="Vista previa de la obra"></div>
        <p class="meta" id="metaObra" hidden></p>
        <div class="msg err" id="errObra" role="alert" hidden></div>
        <div class="row"><button class="ghost" id="btnObraPrecargada" title="Elige al azar una imagen de la carpeta precarga-sin-analizar">Usar una imagen precargada</button><button id="btnQuitarObra" hidden>Quitar imagen</button></div>
      </div>

      <!-- Cámara -->
      <div class="slot">
        <div class="slot-head"><h3>Cámara · gestos faciales</h3><span class="pill" id="pillCam">Apagada</span></div>

        <div class="consent" id="consent">
          <label><input type="checkbox" id="chkConsent"> Autorizo el uso de la cámara para detectar mi rostro y analizar mis gestos durante la observación. Las imágenes se procesan en este equipo y no se guardan.</label>
        </div>

        <div class="cam" id="cam">
          <div class="cam-off" id="camOff">
            <svg width="40" height="40" viewBox="0 0 24 24" aria-hidden="true"><path fill="none" stroke="currentColor" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round" d="M3 8h3l2-3h8l2 3h3v11H3zM12 17a4 4 0 1 0 0-8 4 4 0 0 0 0 8z"/></svg>
            <span>Cámara apagada</span>
          </div>
          <div class="mirror"><video id="video" autoplay playsinline muted hidden></video></div>
          <canvas id="overlay" aria-hidden="true"></canvas>
          <div class="live-tag" id="liveTag" hidden><span class="dot"></span><span id="liveTxt">En vivo</span></div>
        </div>

        <div class="live" id="livePanel" hidden>
          <div class="live-top">
            <div><span class="k">Emoción actual</span><b id="liveEmo">—</b></div>
            <div><span class="k">Rostro</span><b id="liveFace">—</b></div>
            <div><span class="k">Calificación</span><b id="liveCalif">—</b></div>
            <div><span class="k">Iluminación</span><b id="liveLuz">—</b></div>
          </div>
          <ul class="bars mini" id="liveBars"></ul>
        </div>

        <div class="opts" id="opts" hidden>
          <label>Duración
            <select id="selDur"><option value="3">3 s</option><option value="6" selected>6 s</option><option value="10">10 s</option></select>
          </label>
          <label class="chk"><input type="checkbox" id="chkMostrarObra" checked> Mostrar la obra en grande durante la observación</label>
        </div>

        <div class="frames" id="framesBox" hidden>
          <div class="frames-head"><b id="framesTxt"></b><button class="link" id="btnDescartar">Descartar</button></div>
          <div class="strip" id="strip"></div>
        </div>

        <div class="msg err" id="errCam" role="alert" hidden></div>
        <div class="row">
          <button id="btnCam" disabled>Activar cámara</button>
          <button id="btnObservar" class="primary" hidden>Iniciar observación</button>
          <button id="btnFoto" hidden>Foto única</button>
          <button id="btnApagar" class="ghost" hidden>Detener cámara</button>
        </div>
      </div>
    </div>
    <div class="actions">
      <span class="hint" id="hintAnalizar">Carga una obra o captura tu rostro para continuar.</span>
      <button class="primary big" id="btnAnalizar" disabled>Confirmar y analizar</button>
    </div>
  </section>

  <!-- ===================== PASO 2 ===================== -->
  <section class="panel" id="paso2" aria-labelledby="t2" hidden>
    <div class="panel-head">
      <h2 id="t2">Análisis en curso</h2>
      <p class="lead">Las imágenes recorren los módulos del sistema. Esto toma unos segundos.</p>
    </div>
    <div class="loading" role="status" aria-live="polite">
      <div class="load-card"><div class="spinner"></div><b>Procesando obra…</b><span>CNN · paleta · Grad-CAM</span></div>
      <div class="load-card"><div class="spinner"></div><b>Detectando rostro…</b><span>YuNet · FER+ · gestos</span></div>
    </div>
    <ol class="pipeline" id="pipeline"></ol>
  </section>

  <!-- ===================== PASO 3 ===================== -->
  <section class="panel" id="paso3" aria-labelledby="t3" hidden>
    <div class="panel-head">
      <h2 id="t3">Excepciones</h2>
      <p class="lead" id="leadExc"></p>
    </div>
    <ul class="exc-list" id="listaExc"></ul>
    <div class="actions">
      <button id="btnReintentar">Ajustar y reintentar</button>
      <button class="primary" id="btnVerResultado">Ver resultado</button>
    </div>
  </section>

  <!-- ===================== PASO 4 ===================== -->
  <section class="panel" id="paso4" aria-labelledby="t4" hidden>
    <div class="panel-head">
      <h2 id="t4">Resultado</h2>
      <p class="lead">Comparación entre la emoción que transmite la obra y la que expresa el visitante.</p>
    </div>
    <div class="result-grid">
      <div class="rcard" id="rObra"></div>
      <div class="gauge" id="gauge"></div>
      <div class="rcard" id="rRostro"></div>
    </div>
    <div class="summary" id="resumen"></div>
    <div class="exc-chips" id="excChips"></div>
    <div class="details">
      <article class="card" id="cardClasif"></article>
      <article class="card" id="cardPaleta"></article>
      <article class="card" id="cardEmociones"></article>
      <article class="card" id="cardGestos"></article>
      <article class="card wide" id="cardRostrosObra"></article>
      <article class="card wide" id="cardTiempo"></article>
      <article class="card wide" id="cardTiempos"></article>
      <article class="card wide eval" id="cardEval" hidden>
        <h3>Evaluación del sistema</h3>
        <p class="meta">Para la prueba con visitantes. Responde después de ver el resultado. Se guardan tus respuestas y los valores numéricos del análisis; <b>no se guarda ninguna imagen</b>.</p>
        <div class="eval-grid">
          <label>Código del participante<input id="evPart" maxlength="20" placeholder="P01" autocomplete="off"></label>
          <label>Obra<input id="evObra" maxlength="120" placeholder="nombre o código de la obra"></label>
          <label>¿Qué emoción sentiste al mirar la obra?
            <select id="evEmo">
              <option value="">Selecciona…</option><option value="felicidad">Felicidad</option><option value="sorpresa">Sorpresa</option>
              <option value="neutral">Ninguna en particular (neutral)</option><option value="tristeza">Tristeza</option>
              <option value="miedo">Miedo</option><option value="enojo">Enojo</option><option value="asco">Asco</option><option value="desprecio">Desprecio</option>
            </select></label>
          <fieldset><legend>¿Qué te transmitió la obra?</legend>
            <div class="likert" id="evVal">
              <label><input type="radio" name="evVal" value="-2">Muy negativa</label><label><input type="radio" name="evVal" value="-1">Negativa</label>
              <label><input type="radio" name="evVal" value="0">Neutra</label><label><input type="radio" name="evVal" value="1">Positiva</label>
              <label><input type="radio" name="evVal" value="2">Muy positiva</label>
            </div></fieldset>
          <fieldset><legend>¿Qué tan de acuerdo estás con la coincidencia calculada?</legend>
            <div class="likert" id="evAcu">
              <label><input type="radio" name="evAcu" value="1">1 · Nada</label><label><input type="radio" name="evAcu" value="2">2</label>
              <label><input type="radio" name="evAcu" value="3">3</label><label><input type="radio" name="evAcu" value="4">4</label>
              <label><input type="radio" name="evAcu" value="5">5 · Totalmente</label>
            </div></fieldset>
          <label class="full">Comentario (opcional)<textarea id="evCom" maxlength="300" rows="2"></textarea></label>
        </div>
        <div class="row"><button class="primary" id="btnEval">Guardar respuesta</button><span class="hint" id="evMsg" aria-live="polite"></span></div>
      </article>
    </div>
    <div class="actions noprint">
      <button id="btnJSON">Descargar reporte (JSON)</button>
      <button id="btnImprimir">Imprimir / guardar PDF</button>
      <button id="btnVolverExc" hidden>Ver excepciones</button>
      <button class="primary" id="btnNueva">Analizar otra obra</button>
    </div>
  </section>
  </div><!-- /vistaVisitante -->

  <!-- ===================== IMÁGENES PRECARGADAS ===================== -->
  <section class="panel" id="vistaDataset" role="tabpanel" aria-labelledby="tabDataset" tabindex="0" hidden>
    <div class="panel-head ds-head">
      <div>
        <h2>Explorar las imágenes precargadas</h2>
        <p class="lead">Imágenes elegidas al azar de la carpeta <code>Proyecto-final-de-PDI / dataset usado / precarga-sin-analizar</code> del Google Drive del proyecto, para quien no quiera subir una imagen propia. Cada una pasa por el mismo análisis que una imagen subida: CNN, Grad-CAM, paleta, rostros, emociones y calificación del rostro.</p>
      </div>
      <p class="ds-estado" id="dsEstado" aria-live="polite">Consultando la carpeta de imágenes precargadas…</p>
    </div>
    <div class="ds-bar">
      <label>Cantidad
        <select id="dsN"><option>4</option><option selected>6</option><option>8</option><option>12</option></select>
      </label>
      <button class="primary" id="btnDsAleatorias">Obtener imágenes aleatorias</button>
      <button id="btnDsAnalizarTodas" disabled>Analizar todas</button>
      <label class="chk"><input type="checkbox" id="chkDsAuto" checked> Analizar al cargar</label>
      <span class="sep"></span>
      <button class="ghost" id="btnDsPropia">Analizar una imagen propia</button>
      <input type="file" id="fileDsPropia" accept="image/jpeg,image/png" hidden>
      <button class="ghost small" id="btnDsRecargar" title="Vuelve a leer la carpeta precarga-sin-analizar para incluir imágenes nuevas">Volver a leer la carpeta</button>
    </div>
    <div class="ds-error" id="dsError" role="alert" hidden></div>
    <p class="ds-vacio" id="dsVacio">Pulsa «Obtener imágenes aleatorias» para elegir imágenes al azar de la carpeta <code>precarga-sin-analizar</code>. Si prefieres, usa «Analizar una imagen propia».</p>
    <div class="ds-grid" id="dsGrid" aria-live="polite"></div>
  </section>

  <footer class="foot">Prototipo académico · Jose Gabriel Castañeda Uribe · Juan José Osorio Reyes · Procesamiento Digital de Imágenes, UAO 2026. Los resultados son exploratorios y no constituyen una evaluación psicológica.</footer>
</main>

<!-- Observación a pantalla completa -->
<div class="observe" id="observe" hidden role="dialog" aria-modal="true" aria-labelledby="obsTitulo">
  <div class="observe-inner">
    <img id="obsImg" alt="Obra en observación">
    <div class="obs-fallback" id="obsFallback" hidden>Mira hacia la cámara con naturalidad</div>
    <div class="obs-count" id="obsCount" hidden></div>
  </div>
  <div class="obs-bar">
    <div><b id="obsTitulo">Observa la obra</b><span id="obsSub">La cámara registra tu expresión.</span></div>
    <div class="progress"><i id="obsProg"></i></div>
    <button class="ghost light" id="btnCancelarObs">Cancelar (Esc)</button>
  </div>
</div>

<!-- Acerca del modelo -->
<dialog id="dlgAcerca" aria-labelledby="dlgT">
  <div class="dlg-head"><h2 id="dlgT">Acerca del modelo</h2><button class="ghost small" id="btnCerrarDlg" aria-label="Cerrar">Cerrar</button></div>
  <div id="dlgBody"><p>Cargando…</p></div>
</dialog>

<!-- Detalle de una imagen del dataset -->
<dialog id="dlgDetalle" class="ancho" aria-labelledby="detT">
  <div class="dlg-head"><h2 id="detT">Detalle del análisis</h2>
    <div class="row"><button class="small" id="btnDetUsar">Usar con mi cámara</button><button class="small" id="btnDetJSON">JSON</button><button class="ghost small" id="btnCerrarDet">Cerrar</button></div>
  </div>
  <div id="detBody"></div>
</dialog>

<div class="tooltip" id="tooltip" role="tooltip" hidden></div>
<script src="app.js"></script>
</body>
</html>'''
print('index.html:', len(INDEX_HTML), 'caracteres')

In [ ]:
# Interfaz web: styles.css (se sirve desde memoria, no se escribe en disco)
STYLES_CSS = r''':root{
  --ink:#232B5D; --ink-2:#3A4378; --ink-soft:#5A6185; --muted:#7D839F;
  --paper:#F3F4F8; --card:#FFFFFF; --line:#D9DCE8; --line-2:#E8EAF2; --tint:#ECEEFA;
  --ok:#2F9E6A; --ok-bg:#EAF7F0; --warn:#B7791F; --warn-bg:#FFF6E0; --err:#C2503A; --err-bg:#FDECE8;
  --pos:#2a78d6; --neg:#eb6834; --neutral:#9aa0b4;
  --img-bg:#E9EBF3; --chip-bg:rgba(255,255,255,.7);
  --r-lg:18px; --r-md:10px; --r-sm:6px;
  --f-head:"Bricolage Grotesque","Segoe UI",system-ui,sans-serif;
  --f-body:"Instrument Sans","Segoe UI",system-ui,sans-serif;
  --shadow:0 1px 2px rgba(35,43,93,.06),0 8px 24px -12px rgba(35,43,93,.18);
  --ring:#7C86E8;
}
*{box-sizing:border-box}
html{scroll-behavior:smooth}
body{margin:0;background:var(--paper);color:var(--ink);font:16px/1.55 var(--f-body);-webkit-font-smoothing:antialiased}
h1,h2,h3{font-family:var(--f-head);line-height:1.15;margin:0;letter-spacing:-.01em}
.wrap{max-width:1120px;margin:0 auto;padding:28px 18px 48px}
[hidden]{display:none!important}

/* ---------- Skip link ---------- */
.skip-link{
  position:fixed;top:0;left:-9999px;z-index:999;
  background:var(--ink);color:var(--card);
  padding:10px 16px;border-radius:0 0 var(--r-md) 0;
  text-decoration:none;font:600 .9rem var(--f-body);
}
.skip-link:focus{left:0;outline:3px solid var(--ring);outline-offset:2px}

/* ---------- Header ---------- */
.top{display:flex;gap:24px;justify-content:space-between;align-items:flex-start;flex-wrap:wrap}
.top h1{font-size:clamp(1.7rem,3.6vw,2.45rem);max-width:20ch}
.eyebrow{margin:0 0 6px;font-size:.78rem;font-weight:600;letter-spacing:.08em;text-transform:uppercase;color:var(--ink-soft)}
.sub{color:var(--ink-soft);margin:10px 0 0;max-width:64ch}
.top-side{display:flex;flex-direction:column;align-items:flex-end;gap:10px}
.status{list-style:none;margin:0;padding:0;display:flex;flex-wrap:wrap;gap:6px;justify-content:flex-end}
.status li{display:flex;align-items:center;gap:6px;font-size:.8rem;background:var(--card);border:1px solid var(--line);border-radius:99px;padding:4px 10px;color:var(--ink-soft)}
.status i{width:8px;height:8px;border-radius:50%;background:var(--line)}
.status li.ok i{background:var(--ok)} .status li.bad i{background:var(--err)}
.status li.bad{color:var(--err)}

/* ---------- Stepper ---------- */
.stepper{display:flex;gap:8px;list-style:none;padding:0;margin:26px 0 18px}
.stepper li{flex:1;padding:10px 14px;border-radius:var(--r-md);border:1.5px solid var(--line);background:var(--card);font-size:.86rem;color:var(--ink-soft);transition:background .2s,border-color .2s,color .2s}
.stepper li strong{display:block;font-family:var(--f-head);font-size:1rem;color:inherit}
.stepper li.done{border-color:var(--ok);color:var(--ok)}
.stepper li.active{border-color:var(--ink);background:var(--ink);color:var(--card)}
.stepper li.skip{opacity:.55}
@media(max-width:640px){.stepper li span{display:none}.stepper li{padding:8px;text-align:center}}

/* ---------- Panels ---------- */
.panel{background:var(--card);border:1.5px solid var(--line);border-radius:var(--r-lg);padding:clamp(16px,3vw,28px);box-shadow:var(--shadow);animation:aparece .25s ease}
@keyframes aparece{from{opacity:0;transform:translateY(6px)}to{opacity:1;transform:none}}
.panel-head{margin-bottom:18px}
.panel-head h2{font-size:1.55rem;outline:none}
.lead{color:var(--ink-soft);margin:4px 0 0}
.grid2{display:grid;grid-template-columns:1fr 1fr;gap:18px}
@media(max-width:820px){.grid2{grid-template-columns:1fr}}
.slot{border:1.5px solid var(--line);border-radius:var(--r-md);padding:16px;display:flex;flex-direction:column;gap:12px;min-height:360px}
.slot-head{display:flex;justify-content:space-between;align-items:center;gap:8px}
.slot h3{font-size:1.1rem}
.pill{font-size:.75rem;font-weight:600;border-radius:99px;padding:3px 10px;background:var(--line-2);color:var(--ink-soft)}
.pill.ok{background:var(--ok-bg);color:#1D6B47} .pill.live{background:#FDE8E8;color:#A23434} .pill.info{background:var(--tint);color:var(--ink)}

.drop{flex:1;border:2px dashed #9AA0C4;border-radius:var(--r-md);display:grid;place-items:center;text-align:center;padding:20px;color:var(--ink-soft);cursor:pointer;min-height:220px;transition:background .15s,border-color .15s}
.drop p{margin:8px 0 2px}
.drop.over,.drop:hover{background:var(--tint);border-color:var(--ink)}
.preview{flex:1;display:grid;place-items:center;background:var(--img-bg);border-radius:var(--r-md);overflow:hidden;min-height:220px}
.preview img{max-width:100%;max-height:300px;display:block}
.meta{margin:0;font-size:.85rem;color:var(--ink-soft)}

.consent{background:var(--tint);border-radius:var(--r-md);padding:10px 12px;font-size:.86rem}
.consent label{display:flex;gap:10px;align-items:flex-start;cursor:pointer}
.consent input{margin-top:4px;accent-color:var(--ok);width:16px;height:16px;flex:none}

.cam{position:relative;background:#1B2044;border-radius:var(--r-md);overflow:hidden;aspect-ratio:4/3;display:grid;place-items:center}
.cam-off{display:grid;place-items:center;gap:6px;color:#AEB3D6;text-align:center}
.mirror{position:absolute;inset:0;transform:scaleX(-1)}
.mirror video{width:100%;height:100%;object-fit:cover;display:block}
#overlay{position:absolute;inset:0;width:100%;height:100%;pointer-events:none}
.live-tag{position:absolute;top:10px;left:10px;display:flex;align-items:center;gap:6px;background:rgba(15,18,40,.7);color:#fff;font-size:.75rem;font-weight:600;padding:4px 10px;border-radius:99px}
.live-tag .dot{width:8px;height:8px;border-radius:50%;background:#FF5A5A;animation:latido 1.2s infinite}
@keyframes latido{50%{opacity:.3}}
@media(prefers-reduced-motion:reduce){.live-tag .dot{animation:none}}

.live{border:1.5px solid var(--line);border-radius:var(--r-md);padding:10px 12px}
.live-top{display:grid;grid-template-columns:repeat(4,1fr);gap:8px;margin-bottom:8px}
.live-top .k{display:block;font-size:.72rem;color:var(--muted);text-transform:uppercase;letter-spacing:.05em}
.live-top b{font-family:var(--f-head);font-size:1rem}
.opts{display:flex;flex-wrap:wrap;gap:10px 16px;align-items:center;font-size:.88rem;color:var(--ink-soft)}
.opts select{font:inherit;padding:4px 8px;border-radius:var(--r-sm);border:1.5px solid var(--line);margin-left:6px;color:var(--ink);background:var(--card)}
.opts .chk{display:flex;gap:6px;align-items:center}
.opts input{accent-color:var(--ok)}
.frames{border:1.5px solid var(--ok);background:var(--ok-bg);border-radius:var(--r-md);padding:10px 12px}
.frames-head{display:flex;justify-content:space-between;align-items:center;font-size:.9rem;color:#1D6B47}
.strip{display:flex;gap:4px;overflow-x:auto;margin-top:8px;padding-bottom:2px}
.strip img{height:46px;border-radius:4px;transform:scaleX(-1);flex:none}

/* ---------- Buttons ---------- */
.row{display:flex;flex-wrap:wrap;gap:8px}
button{font:600 .93rem var(--f-body);padding:10px 16px;border-radius:var(--r-md);border:1.5px solid var(--ink);background:var(--card);color:var(--ink);cursor:pointer;transition:background .15s,transform .05s}
button:hover:not(:disabled){background:var(--tint)}
button:active:not(:disabled){transform:translateY(1px)}
button.primary{background:var(--ink);color:var(--card)}
button.primary:hover:not(:disabled){background:var(--ink-2);color:var(--card)}
button.big{padding:12px 22px;font-size:1rem}
button.ghost{border-color:var(--line)}
button.ghost.light{background:transparent;color:#fff;border-color:rgba(255,255,255,.4)}
button.ghost.light:hover{background:rgba(255,255,255,.1)}
button.small{padding:6px 12px;font-size:.82rem}
button.link{border:0;background:none;padding:0;text-decoration:underline;color:inherit;font-weight:500}
button:disabled{opacity:.45;cursor:not-allowed}
:focus-visible{outline:3px solid var(--ring);outline-offset:2px}

.msg{border-radius:var(--r-md);padding:10px 12px;font-size:.92rem;border:1.5px solid}
.msg.err{background:var(--err-bg);border-color:var(--err);color:#8A2F1E}
.msg.warn{background:var(--warn-bg);border-color:var(--warn);color:#7A4E0E}
.banner{margin-top:18px}
.actions{display:flex;flex-wrap:wrap;gap:10px;justify-content:flex-end;align-items:center;margin-top:22px}
.hint{color:var(--muted);font-size:.88rem;margin-right:auto}

/* ---------- Loading ---------- */
.loading{display:grid;grid-template-columns:1fr 1fr;gap:16px}
@media(max-width:760px){.loading{grid-template-columns:1fr}}
.load-card{border:1.5px solid var(--line);border-radius:var(--r-md);padding:30px 16px;text-align:center;color:var(--ink-soft);display:grid;gap:4px}
.load-card b{color:var(--ink)} .load-card span{font-size:.85rem}
.spinner{width:44px;height:44px;margin:0 auto 10px;border-radius:50%;border:4px solid var(--line-2);border-top-color:var(--ink);animation:spin .9s linear infinite}
@keyframes spin{to{transform:rotate(360deg)}}
@media(prefers-reduced-motion:reduce){.spinner{animation-duration:3s}}
.pipeline{list-style:none;padding:0;margin:18px 0 0;display:flex;flex-wrap:wrap;gap:8px}
.pipeline li{font-size:.82rem;padding:5px 12px;border-radius:99px;border:1.5px solid var(--line);color:var(--muted);transition:all .2s}
.pipeline li.run{border-color:var(--ink);color:var(--ink)}
.pipeline li.done{border-color:var(--ok);color:#1D6B47;background:var(--ok-bg)}

/* ---------- Exceptions ---------- */
.exc-list{display:grid;gap:12px;margin:0;padding:0;list-style:none}
.exc{border:1.5px solid;border-radius:var(--r-md);padding:12px 14px;display:grid;grid-template-columns:auto 1fr auto;gap:2px 12px;align-items:start}
.exc.error{background:var(--err-bg);border-color:var(--err)}
.exc.aviso{background:var(--warn-bg);border-color:var(--warn)}
.exc.info{background:var(--tint);border-color:#9AA0C4}
.exc .ic{font-size:1.3rem;grid-row:span 2}
.exc b{font-family:var(--f-head)}
.exc .mod{grid-row:span 2;font-size:.72rem;font-weight:600;padding:2px 8px;border-radius:99px;background:var(--chip-bg);color:var(--ink-soft);white-space:nowrap}
.ok-box{border:1.5px solid var(--ok);background:var(--ok-bg);color:#1D6B47;border-radius:var(--r-md);padding:14px}

/* ---------- Result ---------- */
.result-grid{display:grid;grid-template-columns:1fr 240px 1fr;gap:22px;align-items:center}
@media(max-width:900px){.result-grid{grid-template-columns:1fr}}
.rcard{text-align:center;display:flex;flex-direction:column;align-items:center;gap:10px}
.rcard .tag{font-family:var(--f-head);font-size:1.25rem}
.rcard .tag small{display:block;font:500 .85rem var(--f-body);color:var(--ink-soft)}
.rcard .tag.face{color:var(--ok)}
.rcard .vacio{color:var(--muted);border:1.5px dashed var(--line);border-radius:var(--r-md);padding:40px 16px;width:100%}
.toggle{display:inline-flex;border:1.5px solid var(--line);border-radius:99px;padding:2px}
.toggle button{border:0;border-radius:99px;padding:4px 12px;font-size:.8rem;background:transparent}
.toggle button[aria-pressed="true"]{background:var(--ink);color:var(--card)}
.stage{position:relative;display:inline-block;max-width:100%;line-height:0;border-radius:var(--r-md);overflow:hidden;background:var(--img-bg)}
.stage img{max-width:100%;max-height:330px;display:block}
.stage.mirror-img img{transform:scaleX(-1)}
.box{position:absolute;border:3px solid var(--ink);border-radius:4px}
.box.face{border-color:var(--ok)}
.box span{position:absolute;left:-3px;top:-3px;transform:translateY(-100%);background:var(--ink);color:var(--card);font:500 .72rem/1.3 var(--f-body);padding:2px 6px;border-radius:4px 4px 4px 0;white-space:nowrap}
.box.face span{background:var(--ok);color:#fff}
.box.top span{transform:none;top:0;left:0;border-radius:0 0 4px 0}
.box.bottom span{top:auto;bottom:0;left:0;transform:none;border-radius:0 4px 0 0}
.lm{position:absolute;width:6px;height:6px;margin:-3px 0 0 -3px;border-radius:50%;background:#fff;box-shadow:0 0 0 2px var(--ok)}
.gauge{text-align:center}
.gauge svg{width:100%;max-width:240px}
.gauge .val{font:700 2.4rem var(--f-head);margin-top:-46px}
.gauge .cap{color:var(--ink-soft);font-size:.9rem}
.gauge .lvl{display:inline-block;margin-top:6px;font-size:.8rem;font-weight:600;padding:3px 10px;border-radius:99px;background:var(--tint)}
.summary{margin-top:22px;padding:14px 16px;border-left:5px solid var(--ink);background:var(--tint);border-radius:0 var(--r-md) var(--r-md) 0}
.exc-chips{display:flex;flex-wrap:wrap;gap:6px;margin-top:10px}
.exc-chips span{font-size:.8rem;padding:3px 10px;border-radius:99px;border:1.5px solid var(--warn);background:var(--warn-bg);color:#7A4E0E}
.exc-chips span.info{border-color:#9AA0C4;background:var(--tint);color:var(--ink)}
.exc-chips span.error{border-color:var(--err);background:var(--err-bg);color:#8A2F1E}

.details{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-top:24px}
@media(max-width:820px){.details{grid-template-columns:1fr}}
.card{border:1.5px solid var(--line);border-radius:var(--r-md);padding:16px 18px}
.card.wide{grid-column:1/-1}
.card h3{font-size:1.08rem;margin-bottom:2px}
.card .note{color:var(--muted);font-size:.84rem;margin:2px 0 12px}
.card:empty{display:none}

.bars{list-style:none;margin:0;padding:0;display:grid;gap:7px}
.bars li{display:grid;grid-template-columns:92px 1fr 46px;align-items:center;gap:8px;font-size:.86rem;border-radius:4px}
.bars li:hover{background:var(--paper)}
.bars .track{height:10px;background:var(--line-2);border-radius:4px;position:relative}
.bars .fill{position:absolute;left:0;top:0;bottom:0;border-radius:0 4px 4px 0;min-width:2px;transition:width .25s ease}
.bars .num{text-align:right;font-variant-numeric:tabular-nums;color:var(--ink-soft)}
.bars li.top .lab{font-weight:600}
.bars.ancho li{grid-template-columns:140px 1fr 64px}
.bars.mini li{grid-template-columns:78px 1fr 38px;font-size:.78rem;gap:6px}
.bars.mini .track{height:7px}
.bars .thr{position:absolute;top:-3px;bottom:-3px;width:2px;background:var(--ink)}

.palette-bar{display:flex;gap:2px;height:34px;border-radius:6px;overflow:hidden;margin:4px 0 12px}
.palette-bar i{display:block;height:100%}
.swatches{display:grid;grid-template-columns:repeat(auto-fill,minmax(130px,1fr));gap:8px}
.sw{display:flex;gap:8px;align-items:center;border:1.5px solid var(--line-2);border-radius:var(--r-sm);padding:6px;cursor:pointer;background:var(--card);text-align:left;font-weight:400;color:var(--ink)}
.sw i{width:30px;height:30px;border-radius:4px;flex:none;box-shadow:inset 0 0 0 1px rgba(0,0,0,.08)}
.sw b{display:block;font:600 .82rem ui-monospace,Consolas,monospace}
.sw span{font-size:.75rem;color:var(--ink-soft)}
.kv{display:grid;grid-template-columns:repeat(auto-fill,minmax(140px,1fr));gap:10px;margin:12px 0 0}
.kv div{background:var(--paper);border-radius:var(--r-sm);padding:8px 10px}
.kv span{display:block;font-size:.72rem;color:var(--muted);text-transform:uppercase;letter-spacing:.04em}
.kv b{font-family:var(--f-head);font-size:1.05rem;font-variant-numeric:tabular-nums}
.badge{display:inline-block;font-size:.75rem;font-weight:600;padding:2px 8px;border-radius:99px;vertical-align:middle;margin-left:6px}
.badge.warn{background:var(--warn-bg);color:#7A4E0E;border:1px solid var(--warn)}
.badge.ok{background:var(--ok-bg);color:#1D6B47;border:1px solid var(--ok)}
.legend{display:flex;flex-wrap:wrap;gap:6px 14px;font-size:.8rem;color:var(--ink-soft);margin-top:8px}
.legend i{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:5px;vertical-align:-1px}
.chart svg{width:100%;height:auto;display:block;overflow:visible}
.chart .grid line{stroke:var(--line-2)}
.chart text{font:12px var(--f-body);fill:var(--muted)}
table.mini{border-collapse:collapse;font-size:.86rem;margin-top:8px;width:100%}
table.mini th,table.mini td{border:1px solid var(--line);padding:6px 8px;text-align:center}
table.mini th{background:var(--paper);font-weight:600}
details.tabla{margin-top:10px;font-size:.86rem}
details.tabla summary{cursor:pointer;color:var(--ink-soft)}

/* ---------- Observación ---------- */
.observe{position:fixed;inset:0;z-index:50;background:#0E1230;display:flex;flex-direction:column}
.observe-inner{flex:1;display:grid;place-items:center;padding:24px;position:relative;min-height:0}
.observe-inner img{position:absolute;inset:24px;width:calc(100% - 48px);height:calc(100% - 48px);object-fit:contain;border-radius:8px;filter:drop-shadow(0 20px 40px rgba(0,0,0,.45))}
.obs-fallback{color:#C9CDEB;font:500 1.4rem var(--f-head)}
.obs-count{position:absolute;inset:0;display:grid;place-content:center;text-align:center;background:rgba(14,18,48,.85);color:#fff;font:700 6rem/1 var(--f-head)}
.obs-count small{display:block;font:500 1.1rem var(--f-body);margin-top:10px;color:#C9CDEB}
.obs-bar{display:flex;gap:18px;align-items:center;padding:14px 22px;background:#161B42;color:#fff}
.obs-bar b{display:block;font-family:var(--f-head)} .obs-bar span{font-size:.85rem;color:#C9CDEB}
.progress{flex:1;height:8px;background:rgba(255,255,255,.15);border-radius:99px;overflow:hidden}
.progress i{display:block;height:100%;width:0;background:#7C86E8;border-radius:99px;transition:width .1s linear}

/* ---------- Dialog ---------- */
dialog{border:0;border-radius:var(--r-lg);padding:22px 24px;max-width:760px;width:calc(100% - 32px);color:var(--ink);background:var(--card);box-shadow:0 30px 80px rgba(14,18,48,.35)}
dialog::backdrop{background:rgba(14,18,48,.5)}
.dlg-head{display:flex;justify-content:space-between;align-items:center;margin-bottom:10px}
dialog p,dialog li{color:var(--ink-soft);font-size:.92rem}
dialog h3{margin:16px 0 6px;font-size:1.05rem}
.flow{display:flex;flex-wrap:wrap;gap:6px;align-items:center;font-size:.8rem;margin:6px 0}
.flow span{padding:3px 9px;border-radius:6px;background:var(--tint)}
.flow em{color:var(--muted);font-style:normal}

.tooltip{position:fixed;z-index:60;pointer-events:none;background:var(--ink);color:var(--card);font-size:.8rem;padding:6px 10px;border-radius:6px;box-shadow:var(--shadow);max-width:240px}
.foot{margin-top:28px;font-size:.8rem;color:var(--muted);text-align:center}

/* ---------- Tabs de modo ---------- */
.modos{display:flex;gap:8px;margin:24px 0 0;border-bottom:1.5px solid var(--line)}
.modos button{border:0;border-bottom:3px solid transparent;border-radius:var(--r-md) var(--r-md) 0 0;background:transparent;padding:10px 16px 9px;text-align:left;color:var(--ink-soft);margin-bottom:-1.5px}
.modos button small{display:block;font-weight:400;font-size:.78rem;color:var(--muted)}
.modos button[aria-selected="true"]{border-bottom-color:var(--ink);color:var(--ink);background:var(--card);font-weight:600}
.modos button:hover{background:var(--tint)}
#vistaDataset{margin-top:18px}

/* ---------- Dataset ---------- */
.ds-head{display:flex;justify-content:space-between;gap:18px;flex-wrap:wrap;align-items:flex-start}
.ds-head .lead{max-width:70ch} .ds-head a{color:var(--ink)}
.ds-estado{margin:0;font-size:.82rem;color:var(--ink-soft);background:var(--paper);border-radius:var(--r-md);padding:8px 12px;max-width:360px}
.ds-estado b{color:var(--ink)}
.ds-bar{display:flex;flex-wrap:wrap;gap:10px;align-items:center;padding:12px;border:1.5px solid var(--line);border-radius:var(--r-md);background:var(--paper)}
.ds-bar label{font-size:.88rem;color:var(--ink-soft);display:flex;align-items:center;gap:6px}
.ds-bar select{font:inherit;padding:6px 8px;border-radius:var(--r-sm);border:1.5px solid var(--line);color:var(--ink);background:var(--card)}
.ds-bar input{accent-color:var(--ok)}
.ds-bar .sep{flex:1}
.ds-error{margin-top:14px;border:1.5px solid var(--err);background:var(--err-bg);color:#8A2F1E;border-radius:var(--r-md);padding:12px 14px;font-size:.92rem}
.ds-error b{display:block;font-family:var(--f-head);font-size:1rem;margin-bottom:2px}
.ds-error ol{margin:6px 0 0;padding-left:20px} .ds-error code{background:var(--chip-bg);padding:1px 5px;border-radius:4px}
.ds-vacio{color:var(--muted);text-align:center;padding:36px 12px;border:1.5px dashed var(--line);border-radius:var(--r-md);margin-top:14px}
.ds-grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(250px,1fr));gap:14px;margin-top:14px}
.ds-card{border:1.5px solid var(--line);border-radius:var(--r-md);overflow:hidden;display:flex;flex-direction:column;background:var(--card);animation:aparece .25s ease}
.ds-card.analizando{border-color:var(--ink)}
.ds-img{position:relative;background:var(--img-bg);display:grid;place-items:center;height:200px;overflow:hidden}
.ds-img .stage{max-height:200px;border-radius:0}
.ds-img .stage img{max-height:200px}
.ds-img .box span{font-size:.65rem;padding:1px 4px}
.ds-tag{position:absolute;left:8px;top:8px;font-size:.7rem;font-weight:600;padding:2px 8px;border-radius:99px;background:var(--card);color:var(--ink);border:1px solid var(--line)}
.ds-tag.vista{color:#7A4E0E;border-color:var(--warn)} .ds-tag.prueba{color:#1D6B47;border-color:var(--ok)}
.ds-body{padding:10px 12px;display:flex;flex-direction:column;gap:6px;flex:1;font-size:.86rem}
.ds-name{font:600 .8rem ui-monospace,Consolas,monospace;color:var(--ink-soft);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.ds-res{display:grid;grid-template-columns:1fr 1fr;gap:6px}
.ds-res div{background:var(--paper);border-radius:var(--r-sm);padding:6px 8px}
.ds-res span{display:block;font-size:.68rem;color:var(--muted);text-transform:uppercase;letter-spacing:.04em}
.ds-res b{font-family:var(--f-head);font-size:.95rem}
.ds-res .full{grid-column:1/-1}
.meter{height:6px;background:var(--line-2);border-radius:4px;margin-top:4px;overflow:hidden}
.meter i{display:block;height:100%;background:var(--ok);border-radius:0 4px 4px 0}
.ds-msg{color:var(--muted);font-size:.84rem}
.ds-msg.err{color:var(--err)}
.ds-actions{display:flex;gap:6px;flex-wrap:wrap;padding:0 12px 12px}
.ds-actions button{padding:6px 10px;font-size:.8rem}
.spin-sm{display:inline-block;width:14px;height:14px;border-radius:50%;border:2px solid var(--line-2);border-top-color:var(--ink);animation:spin .9s linear infinite;vertical-align:-2px;margin-right:6px}
dialog.ancho{max-width:1040px}
.det-grid{display:grid;grid-template-columns:minmax(0,1.1fr) minmax(0,1fr);gap:16px;align-items:start}
@media(max-width:820px){.det-grid{grid-template-columns:1fr}}
.det-img{display:flex;flex-direction:column;gap:8px;align-items:center}
.det-cards{display:grid;gap:12px}
.det-cards .card{padding:12px 14px}
table.mini td.l{text-align:left}
.calif{display:inline-flex;align-items:center;gap:6px}
.calif i{display:inline-block;width:54px;height:6px;background:var(--line-2);border-radius:4px;overflow:hidden;position:relative}
.calif i::after{content:"";position:absolute;inset:0;width:var(--v);background:var(--ok);border-radius:0 4px 4px 0}

/* ---------- Toasts ---------- */
.toast-container{position:fixed;bottom:20px;right:20px;z-index:200;display:flex;flex-direction:column;gap:8px;pointer-events:none;max-width:min(380px,calc(100vw - 40px))}
.toast{background:var(--ink);color:var(--card);padding:10px 16px;border-radius:var(--r-md);font-size:.9rem;line-height:1.4;box-shadow:0 8px 24px rgba(0,0,0,.2);animation:toast-in .25s ease;pointer-events:auto}
.toast.ok{background:var(--ok);color:#fff}
.toast.warn{background:var(--warn);color:#fff}
.toast.err{background:var(--err);color:#fff}
.toast.out{animation:toast-out .25s ease forwards}
@keyframes toast-in{from{opacity:0;transform:translateY(10px)}to{opacity:1;transform:none}}
@keyframes toast-out{to{opacity:0;transform:translateX(20px)}}

/* ---------- Skeleton ---------- */
.skel{background:linear-gradient(90deg,var(--line-2) 25%,var(--tint) 50%,var(--line-2) 75%);background-size:200% 100%;animation:skel 1.4s infinite;border-radius:var(--r-sm)}
@keyframes skel{0%{background-position:200% 0}100%{background-position:-200% 0}}

/* ---------- Scrollbar ---------- */
*{scrollbar-width:thin;scrollbar-color:var(--line) transparent}
*::-webkit-scrollbar{width:10px;height:10px}
*::-webkit-scrollbar-track{background:transparent}
*::-webkit-scrollbar-thumb{background:var(--line);border-radius:99px;border:2px solid var(--paper)}
*::-webkit-scrollbar-thumb:hover{background:var(--muted)}

/* ---------- Modo oscuro ---------- */
@media (prefers-color-scheme: dark){
  :root{
    --ink:#E8EAF3;--ink-2:#C8CCE0;--ink-soft:#A0A6C2;--muted:#7A80A0;
    --paper:#0E1230;--card:#161B42;--line:#2A3160;--line-2:#232A52;--tint:#1E2550;
    --img-bg:#0A0E20;--chip-bg:rgba(255,255,255,.12);
    --ok-bg:#163B29;--warn-bg:#3B2D10;--err-bg:#3B1A15;
    --shadow:0 1px 2px rgba(0,0,0,.3),0 8px 24px -12px rgba(0,0,0,.5);
  }
  .pill.ok{color:#7EE2A8}
  .pill.live{background:#3A1A1A;color:#F3A0A0}
  .frames-head{color:#7EE2A8}
  .exc-chips span{color:#E8C878}
  .exc-chips span.error{color:#F3A0A0}
  .badge.warn{color:#E8C878}
  .badge.ok{color:#7EE2A8}
  .msg.err{color:#F3A0A0}
  .msg.warn{color:#E8C878}
  .ds-error{color:#F3A0A0}
  .ds-tag.vista{color:#E8C878}
  .ds-tag.prueba{color:#7EE2A8}
  .drop{border-color:#5A6185}
  .box span{color:#0E1230}
  dialog::backdrop{background:rgba(0,0,0,.7)}
  .live-tag{background:rgba(0,0,0,.75)}
  .summary{border-left-color:var(--ink)}
}

/* ---------- Reduced motion ---------- */
@media (prefers-reduced-motion: reduce){
  *,*::before,*::after{
    animation-duration:.01ms !important;
    animation-iteration-count:1 !important;
    transition-duration:.01ms !important;
    scroll-behavior:auto !important;
  }
  .spinner{border-top-color:var(--ink);animation:none !important;border-style:solid;border-color:var(--line);border-top-color:var(--ink)}
  .progress i{transition:none}
}

/* ---------- Touch targets móvil ---------- */
@media (max-width:640px){
  button{padding:12px 16px;min-height:44px}
  .modos button{flex:1;text-align:center}
  .live-top{grid-template-columns:1fr 1fr}
  .actions{flex-direction:column;align-items:stretch}
  .actions .hint{margin:0 0 4px;text-align:center}
  .actions button{width:100%}
  .toast-container{left:12px;right:12px;bottom:12px;max-width:none}
  .ds-bar{flex-direction:column;align-items:stretch}
  .ds-bar label{justify-content:space-between}
}

/* ---------- Print ---------- */
@media print{
  :root{--paper:#fff;--card:#fff;--ink:#111;--ink-soft:#444;--line:#bbb;--tint:#f4f4f4;--shadow:none;--img-bg:#fff}
  body{background:#fff;color:#111}
  .stepper,.top-side,.modos,.banner,.foot,.noprint,
  #paso1,#paso2,#paso3,.skip-link,.toast-container,.tooltip,
  #tabVisitante,#tabDataset{display:none !important}
  #paso4,#paso1{display:block !important}
  .panel{display:block !important;border:0;box-shadow:none;padding:0;margin:0}
  .result-grid{grid-template-columns:1fr 1fr 1fr}
  .details{grid-template-columns:1fr 1fr;gap:12px}
  .card{break-inside:avoid;page-break-inside:avoid;border-color:#ccc}
  .stage,.preview,.chart,table.mini{break-inside:avoid;page-break-inside:avoid}
  .summary{border-left-width:3px;background:#f6f6f6}
  a[href^="http"]::after{content:" (" attr(href) ")";font-size:.75em;color:#666}
  h1,h2,h3{color:#111}
  .tooltip{display:none !important}
}
/* ---------- evaluación con visitantes ---------- */
.eval h3{margin:0 0 4px}
.eval-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(240px,1fr));gap:12px 16px;margin:12px 0}
.eval-grid label{display:flex;flex-direction:column;gap:4px;font-size:.9rem;color:var(--ink-soft)}
.eval-grid input,.eval-grid select,.eval-grid textarea{font:inherit;color:var(--ink);background:var(--card);border:1.5px solid var(--line);border-radius:8px;padding:7px 9px}
.eval-grid fieldset{border:1.5px solid var(--line);border-radius:8px;padding:8px 10px;margin:0;grid-column:1/-1}
.eval-grid legend{font-size:.9rem;color:var(--ink-soft);padding:0 4px}
.eval-grid .full{grid-column:1/-1}
.likert{display:flex;flex-wrap:wrap;gap:6px 14px}
.likert label{flex-direction:row;align-items:center;gap:6px;color:var(--ink)}
@media print{#cardEval{display:none!important}}'''
print('styles.css:', len(STYLES_CSS), 'caracteres')

In [ ]:
# Interfaz web: app.js (se sirve desde memoria, no se escribe en disco)
APP_JS = r'''/* =====================================================================
   Coincidencia emocional entre obra y visitante — GUI
   Backend FastAPI (mismo origen):
     GET  /api/estado     · GET  /api/metricas
     POST /api/frame      · POST /api/analizar
     GET  /api/dataset/estado | /api/dataset/aleatorias | /api/dataset/imagen
     POST /api/dataset/analizar | /api/dataset/recargar   (carpeta precarga-sin-analizar)
     POST /api/evaluacion  (formulario de evaluación con visitantes)
   ===================================================================== */
"use strict";

const API = location.protocol.startsWith("http") ? "" : "http://localhost:8000";
const CFG = {
  MAX_MB: 10,
  LIVE_ANCHO: 480,
  LIVE_PAUSA_MS: 120,
  OBS_ANCHO: 640,
  OBS_INTERVALO_MS: 400,
  PREP_SEG: 3,
  TIMEOUT_MS: 60000,
  FETCH_TIMEOUT_MS: 15000,
  FETCH_RETRIES: 2,
};

const EMO = {
  felicidad: { es: "Felicidad", color: "#eda100" },
  sorpresa:  { es: "Sorpresa",  color: "#e87ba4" },
  neutral:   { es: "Neutral",   color: "#9aa0b4" },
  tristeza:  { es: "Tristeza",  color: "#2a78d6" },
  miedo:     { es: "Miedo",     color: "#6250d6" },
  enojo:     { es: "Enojo",     color: "#e34948" },
  asco:      { es: "Asco",      color: "#008300" },
  desprecio: { es: "Desprecio", color: "#eb6834" },
};
const MODULOS = [
  ["image_loader","ImageLoader"],["preprocesamiento","PreprocesamientoImagen"],["cnn","ClasificadorDeObras"],
  ["gradcam","Grad-CAM"],["paleta","ExtractorDePaleta"],["gestos","AnalizarGestosFaciales"],
  ["resultado","ResultadoProceso"],["integracion","Integración"],
];
const ICONOS = { servidor:"🛠️", rostro_no_detectado:"🙈", formato:"🖼️", archivo_corrupto:"🧩", archivo_ausente:"📭",
  calidad_insuficiente:"🌫️", fuera_de_rango:"📏", no_reconocido:"❓", entrada_incompleta:"🧷", multiples_rostros:"👥",
  dataset_no_encontrado:"📁", dataset_no_disponible:"📭",
  imagen_no_encontrada:"🔍", imagen_conocida:"🎓", camara:"📷", expresion_neutral:"😐" };
const TITULOS = { servidor:"Error del servidor", rostro_no_detectado:"Rostro no detectado", formato:"Formato no soportado",
  archivo_corrupto:"Archivo dañado", archivo_ausente:"Archivo ausente", calidad_insuficiente:"Calidad de imagen insuficiente",
  fuera_de_rango:"Valor fuera de rango", no_reconocido:"Elemento no reconocido", entrada_incompleta:"Entrada incompleta",
  multiples_rostros:"Varias personas en cuadro", dataset_no_encontrado:"Carpeta de precarga no encontrada",
  dataset_no_disponible:"Carpeta de precarga vacía", imagen_no_encontrada:"Imagen no encontrada",
  imagen_conocida:"Imagen conocida por la CNN", expresion_neutral:"Expresión mayormente neutral" };
const gravedadCls = g => g === "error" ? "error" : g === "info" ? "info" : "aviso";

const $ = id => document.getElementById(id);
const state = {
  obra: null, obraURL: null, stream: null, frames: [], resultado: null, servidorOk: false,
  live: false, liveTimer: null, observando: false, cancelarObs: null, ultimoLive: null,
  analizando: false, enfocarAlCerrarObs: null,
};

/* ============================================================ utilidades */
function el(tag, cls, text) { const e = document.createElement(tag); if (cls) e.className = cls; if (text != null) e.textContent = text; return e; }
function pct(v, d = 0) { return (Number(v || 0) * 100).toFixed(d) + "%"; }
function mostrarError(id, txt) { const n = $(id); n.textContent = txt || ""; n.hidden = !txt; }
function esc(s) { return String(s).replace(/[&<>"']/g, c => ({ "&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;" }[c])); }
function pill(id, txt, cls) { const p = $(id); p.textContent = txt; p.className = "pill " + (cls || ""); }
const sleep = ms => new Promise(r => setTimeout(r, ms));

/* sleep cancelable: responde rápido a la cancelación */
function sleepCancel(ms, cancelCheck) {
  return new Promise(resolve => {
    const start = performance.now();
    const id = setInterval(() => {
      if (cancelCheck() || performance.now() - start >= ms) { clearInterval(id); resolve(); }
    }, 50);
  });
}

/* Toast flotante */
function toast(msg, tipo = "", duracion = 3500) {
  let cont = $("toasts");
  if (!cont) {
    cont = el("div", "toast-container"); cont.id = "toasts";
    cont.setAttribute("role", "status"); cont.setAttribute("aria-live", "polite");
    document.body.append(cont);
  }
  const t = el("div", "toast" + (tipo ? " " + tipo : ""), msg);
  cont.append(t);
  setTimeout(() => {
    t.classList.add("out");
    t.addEventListener("animationend", () => t.remove(), { once: true });
    setTimeout(() => t.remove(), 500);
  }, duracion);
}

/* Anuncio a lectores de pantalla */
let _announceNode;
function announce(msg) {
  if (!_announceNode) {
    _announceNode = el("div");
    _announceNode.setAttribute("aria-live", "polite");
    _announceNode.setAttribute("aria-atomic", "true");
    _announceNode.style.cssText = "position:absolute;left:-9999px;width:1px;height:1px;overflow:hidden";
    document.body.append(_announceNode);
  }
  _announceNode.textContent = "";
  setTimeout(() => { _announceNode.textContent = msg; }, 60);
}

/* fetch con timeout + reintentos con backoff exponencial */
async function fetchRetry(url, opts = {}, { retries = CFG.FETCH_RETRIES, timeout = CFG.FETCH_TIMEOUT_MS } = {}) {
  let lastErr;
  for (let i = 0; i <= retries; i++) {
    const ctrl = new AbortController();
    const to = setTimeout(() => ctrl.abort(), timeout);
    try {
      const r = await fetch(url, { ...opts, signal: ctrl.signal });
      clearTimeout(to);
      if (r.status >= 500 && i < retries) {
        lastErr = new Error("HTTP " + r.status);
        await sleep(300 * Math.pow(2, i));
        continue;
      }
      return r;
    } catch (e) {
      clearTimeout(to);
      lastErr = e;
      if (i < retries) await sleep(300 * Math.pow(2, i));
    }
  }
  throw lastErr;
}

function irAPaso(n) {
  [1,2,3,4].forEach(i => $("paso" + i).hidden = i !== n);
  [...$("stepper").children].forEach((li, i) => {
    li.classList.toggle("active", i + 1 === n);
    li.classList.toggle("done", i + 1 < n);
    li.classList.remove("skip");
  });
  if (n === 4 && state.resultado && !state.resultado.excepciones.some(x => x.gravedad !== "info"))
    $("stepper").children[2].classList.add("skip");
  const h = $("t" + n);
  if (h) { h.setAttribute("tabindex", "-1"); h.focus({ preventScroll: true }); }
  window.scrollTo({ top: 0, behavior: "smooth" });
  announce(`Paso ${n} de 4`);
}

/* tooltip compartido */
const tip = $("tooltip");
document.addEventListener("mousemove", e => {
  const t = e.target.closest("[data-tip]");
  if (!t) { tip.hidden = true; return; }
  tip.innerHTML = t.dataset.tip; tip.hidden = false;
  const x = Math.min(e.clientX + 14, innerWidth - tip.offsetWidth - 8);
  tip.style.left = x + "px"; tip.style.top = (e.clientY + 14) + "px";
});

/* ============================================================ estado del servidor */
async function cargarEstado() {
  try {
    const r = await fetchRetry(API + "/api/estado");
    const s = await r.json();
    state.servidorOk = true;
    const set = (k, ok, t) => { const li = document.querySelector(`#status [data-k="${k}"]`); li.className = ok ? "ok" : "bad"; if (t) li.title = t; };
    set("cnn", s.cnn.disponible, s.cnn.error || "CNN de obras entrenada");
    set("det", !!s.rostro.detector, s.rostro.detector);
    set("emo", s.rostro.emociones, s.rostro.errores.join(" "));
    const avisos = [];
    if (!s.cnn.disponible) avisos.push(s.cnn.error);
    if (!s.rostro.emociones) avisos.push(...s.rostro.errores);
    if (avisos.length) { $("bannerServidor").textContent = "Atención: " + avisos.join(" "); $("bannerServidor").hidden = false; }
  } catch {
    state.servidorOk = false;
    $("bannerServidor").innerHTML = "No se pudo contactar al servidor. Ejecuta el notebook completo (Ejecutar todo) y abre el enlace de la última celda.";
    $("bannerServidor").hidden = false;
    document.querySelectorAll("#status li").forEach(li => li.className = "bad");
  }
}

/* ============================================================ PASO 1a — ImageLoader */
function validarArchivo(f) {
  if (!["image/jpeg","image/png"].includes(f.type)) return "Formato no soportado. Ingresa una imagen JPG o PNG.";
  if (f.size > CFG.MAX_MB * 1024 * 1024) return `La imagen pesa más de ${CFG.MAX_MB} MB. Elige una más ligera.`;
  return null;
}
function setObra(file) {
  mostrarError("errObra");
  const err = validarArchivo(file);
  if (err) { mostrarError("errObra", err); toast(err, "err"); return; }
  if (state.obraURL) URL.revokeObjectURL(state.obraURL);
  const url = URL.createObjectURL(file);
  const img = $("imgObra");
  img.onerror = () => { quitarObra(); mostrarError("errObra", "No se pudo leer la imagen. Prueba con otro archivo JPG o PNG."); toast("No se pudo leer la imagen.", "err"); };
  img.onload = () => {
    state.obra = file; state.obraURL = url;
    $("drop").hidden = true; $("prevObra").hidden = false; $("btnQuitarObra").hidden = false;
    $("metaObra").textContent = `${file.name} · ${img.naturalWidth}×${img.naturalHeight} px · ${(file.size / 1024 / 1024).toFixed(2)} MB`;
    $("metaObra").hidden = false;
    pill("pillObra", "Lista", "ok");
    actualizarBoton();
    toast("Obra cargada.", "ok", 1800);
  };
  img.src = url;
}
function quitarObra() {
  if (state.obraURL) { URL.revokeObjectURL(state.obraURL); state.obraURL = null; }
  state.obra = null; $("imgObra").removeAttribute("src"); $("fileObra").value = "";
  $("prevObra").hidden = true; $("drop").hidden = false; $("btnQuitarObra").hidden = true; $("metaObra").hidden = true;
  pill("pillObra", "Pendiente"); actualizarBoton();
}
$("drop").addEventListener("click", () => $("fileObra").click());
$("drop").addEventListener("keydown", e => { if (e.key === "Enter" || e.key === " ") { e.preventDefault(); $("fileObra").click(); } });
$("fileObra").addEventListener("change", e => e.target.files[0] && setObra(e.target.files[0]));
["dragover","dragenter"].forEach(ev => $("drop").addEventListener(ev, e => { e.preventDefault(); $("drop").classList.add("over"); }));
["dragleave","drop"].forEach(ev => $("drop").addEventListener(ev, e => { e.preventDefault(); $("drop").classList.remove("over"); }));
$("drop").addEventListener("drop", e => e.dataTransfer.files[0] && setObra(e.dataTransfer.files[0]));
$("btnQuitarObra").addEventListener("click", quitarObra);

function actualizarBoton() {
  const listo = !!(state.obra || state.frames.length);
  $("btnAnalizar").disabled = !listo || state.analizando;
  $("hintAnalizar").textContent = state.obra && state.frames.length
    ? "Obra y rostro listos: se calculará la coincidencia emocional."
    : state.obra ? "Obra lista. Puedes analizarla sola o capturar también tu rostro."
    : state.frames.length ? "Rostro capturado. Carga una obra para calcular la coincidencia."
    : "Carga una obra o captura tu rostro para continuar.";
}

/* ============================================================ PASO 1b — Cámara */
$("chkConsent").addEventListener("change", e => { $("btnCam").disabled = !e.target.checked; });

async function iniciarCamara() {
  mostrarError("errCam");
  if (!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia) {
    const m = window.isSecureContext === false
      ? "La cámara solo funciona en http://localhost o https://. Abre la página desde http://localhost:8000."
      : "Tu navegador no permite usar la cámara. Usa Chrome, Edge o Firefox actualizados.";
    mostrarError("errCam", m); toast(m, "err"); return;
  }
  try {
    const dispositivos = await navigator.mediaDevices.enumerateDevices();
    if (!dispositivos.some(d => d.kind === "videoinput")) {
      const m = "No se encontró ninguna cámara en este equipo. Conecta una webcam y vuelve a intentarlo.";
      mostrarError("errCam", m); toast(m, "err"); return;
    }
  } catch { /* algunos navegadores no permiten enumerar antes del permiso */ }
  $("btnCam").disabled = true; $("btnCam").textContent = "Solicitando permiso…";
  try {
    state.stream = await navigator.mediaDevices.getUserMedia({ video: { facingMode: "user", width: { ideal: 1280 }, height: { ideal: 720 } }, audio: false });
  } catch (e) {
    const msgs = {
      NotAllowedError: "Permiso de cámara denegado. Habilítalo en la configuración del navegador y vuelve a intentarlo.",
      NotFoundError: "No se encontró ninguna cámara en este dispositivo.",
      NotReadableError: "La cámara está siendo usada por otra aplicación. Ciérrala e inténtalo de nuevo.",
      OverconstrainedError: "La cámara no cumple con los requisitos solicitados.",
      SecurityError: "El navegador bloqueó la cámara por seguridad. Abre la página desde http://localhost:8000.",
      AbortError: "La cámara no respondió (puede estar ocupada). Ciérrala en otras aplicaciones e inténtalo de nuevo.",
      TrackStartError: "La cámara está siendo usada por otra aplicación. Ciérrala e inténtalo de nuevo.",
      DevicesNotFoundError: "No se encontró ninguna cámara en este dispositivo.",
      PermissionDeniedError: "Permiso de cámara denegado. Habilítalo en la configuración del navegador y vuelve a intentarlo.",
    };
    const m = msgs[e.name] || `No se pudo activar la cámara (${e.name || "error desconocido"}). Inténtalo de nuevo.`;
    mostrarError("errCam", m); toast(m, "err");
    $("btnCam").disabled = !$("chkConsent").checked; $("btnCam").textContent = "Activar cámara";
    return;
  }
  $("btnCam").textContent = "Activar cámara";
  state.stream.getVideoTracks().forEach(t => t.addEventListener("ended", () => {
    if (!state.stream) return;
    if (state.cancelarObs) state.cancelarObs();
    detenerCamara();
    mostrarError("errCam", "La cámara se desconectó o fue tomada por otra aplicación. Vuelve a activarla.");
  }));
  const v = $("video"); v.srcObject = state.stream; v.hidden = false;
  await new Promise(r => v.readyState >= 2 ? r() : v.addEventListener("loadeddata", r, { once: true }));
  $("camOff").hidden = true; $("liveTag").hidden = false; $("livePanel").hidden = false; $("opts").hidden = false;
  $("btnCam").hidden = true; $("btnObservar").hidden = false; $("btnFoto").hidden = false; $("btnApagar").hidden = false;
  $("consent").hidden = true;
  $("btnObservar").textContent = `Iniciar observación (${$("selDur").value} s)`;
  pill("pillCam", "En vivo", "live");
  iniciarLive();
  toast("Cámara activada.", "ok", 1800);
  announce("Cámara activada");
}
function detenerCamara() {
  detenerLive();
  if (state.stream) { state.stream.getTracks().forEach(t => t.stop()); state.stream = null; }
  const v = $("video"); v.srcObject = null; v.hidden = true;
  $("camOff").hidden = false; $("liveTag").hidden = true; $("livePanel").hidden = true; $("opts").hidden = true;
  $("btnCam").hidden = false; $("btnObservar").hidden = true; $("btnFoto").hidden = true; $("btnApagar").hidden = true;
  $("btnCam").disabled = !$("chkConsent").checked;
  $("consent").hidden = false;
  limpiarOverlay();
  pill("pillCam", state.frames.length ? "Capturado" : "Apagada", state.frames.length ? "ok" : "");
}
$("btnCam").addEventListener("click", iniciarCamara);
$("btnApagar").addEventListener("click", detenerCamara);
$("selDur").addEventListener("change", () => $("btnObservar").textContent = `Iniciar observación (${$("selDur").value} s)`);

function capturarBlob(ancho, calidad = 0.85) {
  const v = $("video");
  if (!v.videoWidth) return Promise.resolve(null);
  const e = Math.min(1, ancho / v.videoWidth);
  const c = document.createElement("canvas");
  c.width = Math.round(v.videoWidth * e); c.height = Math.round(v.videoHeight * e);
  c.getContext("2d").drawImage(v, 0, 0, c.width, c.height);
  return new Promise(r => c.toBlob(b => r(b), "image/jpeg", calidad));
}

/* ---------- Vista en vivo ---------- */
function iniciarLive() { state.live = true; cicloLive(); }
function detenerLive() { state.live = false; clearTimeout(state.liveTimer); }
async function cicloLive() {
  if (!state.live) return;
  if (state.observando || document.hidden || !state.servidorOk) { state.liveTimer = setTimeout(cicloLive, 400); return; }
  const t0 = performance.now();
  try {
    const blob = await capturarBlob(CFG.LIVE_ANCHO, 0.75);
    if (blob) {
      const fd = new FormData(); fd.append("frame", blob, "frame.jpg");
      const r = await fetch(API + "/api/frame", { method: "POST", body: fd });
      const d = await r.json();
      if (state.live && !state.observando) pintarLive(d, performance.now() - t0);
    }
  } catch { $("liveTxt").textContent = "Sin conexión"; }
  state.liveTimer = setTimeout(cicloLive, CFG.LIVE_PAUSA_MS);
}

function geometriaVideo() {
  const v = $("video"), cv = $("overlay");
  const cw = cv.clientWidth, ch = cv.clientHeight, vw = v.videoWidth || 4, vh = v.videoHeight || 3;
  const s = Math.max(cw / vw, ch / vh);
  return { cw, ch, dw: vw * s, dh: vh * s, ox: (cw - vw * s) / 2, oy: (ch - vh * s) / 2 };
}
function limpiarOverlay() { const cv = $("overlay"); cv.getContext("2d").clearRect(0, 0, cv.width, cv.height); }
function pintarLive(d, ms) {
  state.ultimoLive = d;
  const cv = $("overlay"), dpr = devicePixelRatio || 1;
  cv.width = cv.clientWidth * dpr; cv.height = cv.clientHeight * dpr;
  const ctx = cv.getContext("2d"); ctx.setTransform(dpr, 0, 0, dpr, 0, 0); ctx.clearRect(0, 0, cv.width, cv.height);
  const g = geometriaVideo();
  const mapX = x => g.cw - (g.ox + x * g.dw);
  const mapY = y => g.oy + y * g.dh;
  (d.detecciones || []).forEach(det => {
    const esPrincipal = d.principal && det.caja.x === d.principal.caja.x && det.caja.y === d.principal.caja.y;
    const x1 = mapX(det.caja.x + det.caja.w), y1 = mapY(det.caja.y), w = det.caja.w * g.dw, h = det.caja.h * g.dh;
    ctx.lineWidth = esPrincipal ? 3 : 2;
    ctx.strokeStyle = esPrincipal ? "#2F9E6A" : "rgba(255,255,255,.6)";
    ctx.beginPath(); ctx.roundRect(x1, y1, w, h, 6); ctx.stroke();
    (det.puntos || []).forEach(([px, py]) => { ctx.fillStyle = "#fff"; ctx.beginPath(); ctx.arc(mapX(px), mapY(py), 2.5, 0, 7); ctx.fill(); });
    if (esPrincipal && det.emocion) {
      const txt = `${EMO[det.emocion].es} ${pct(det.confianza)}`;
      ctx.font = "600 13px Instrument Sans, Segoe UI, sans-serif";
      const tw = ctx.measureText(txt).width + 14;
      ctx.fillStyle = "#2F9E6A"; ctx.beginPath(); ctx.roundRect(x1 - 1.5, Math.max(0, y1 - 24), tw, 22, 5); ctx.fill();
      ctx.fillStyle = "#fff"; ctx.fillText(txt, x1 + 6, Math.max(15, y1 - 8));
    }
  });
  const p = d.principal;
  $("liveEmo").textContent = p && p.emocion ? EMO[p.emocion].es : "—";
  $("liveCalif").textContent = p && p.calificacion ? `${Math.round(p.calificacion.puntaje)} · ${p.calificacion.nivel}` : "—";
  $("liveFace").textContent = d.rostros > 1 ? `${d.rostros} rostros` : p ? `Sí · ${pct(p.score_deteccion)}` : "No detectado";
  const b = d.calidad ? d.calidad.brillo : null;
  $("liveLuz").textContent = b == null ? "—" : b < 45 ? "Muy oscura" : b > 225 ? "Excesiva" : "Adecuada";
  $("liveTxt").textContent = `En vivo · ${Math.round(1000 / Math.max(ms + CFG.LIVE_PAUSA_MS, 1))} fps`;
  renderBarras($("liveBars"), p && p.probabilidades, true);
}

function renderBarras(ul, probs, mini) {
  ul.replaceChildren();
  const orden = Object.keys(EMO);
  const top = probs ? orden.reduce((a, b) => (probs[a] >= probs[b] ? a : b)) : null;
  orden.forEach(k => {
    const v = probs ? probs[k] || 0 : 0;
    const li = el("li", k === top ? "top" : "");
    li.dataset.tip = `<b>${EMO[k].es}</b>: ${pct(v, 1)}`;
    li.append(el("span", "lab", EMO[k].es));
    const tr = el("span", "track"), fi = el("span", "fill");
    fi.style.width = (v * 100).toFixed(1) + "%"; fi.style.background = EMO[k].color;
    tr.append(fi); li.append(tr); li.append(el("span", "num", probs ? pct(v) : "—"));
    ul.append(li);
  });
}

/* ---------- Observación guiada ---------- */
async function observar() {
  if (state.observando) return;
  const dur = Number($("selDur").value) * 1000;
  const mostrar = $("chkMostrarObra").checked && state.obraURL;
  state.observando = true; limpiarOverlay();
  const capa = $("observe"), cuenta = $("obsCount"), prog = $("obsProg");
  $("obsImg").hidden = !mostrar; $("obsFallback").hidden = !!mostrar;
  if (mostrar) $("obsImg").src = state.obraURL;
  capa.hidden = false; prog.style.width = "0";
  state.enfocarAlCerrarObs = document.activeElement;
  setTimeout(() => $("btnCancelarObs").focus(), 80);
  let cancelado = false;
  state.cancelarObs = () => { cancelado = true; };
  announce("Observación iniciada");

  $("obsTitulo").textContent = "Prepárate";
  $("obsSub").textContent = mostrar ? "En un momento verás la obra en grande." : "Mira hacia la cámara.";
  cuenta.hidden = false;
  for (let n = CFG.PREP_SEG; n > 0 && !cancelado; n--) {
    cuenta.innerHTML = `${n}<small>${mostrar ? "Observa la obra con naturalidad" : "Mantén el rostro dentro del cuadro"}</small>`;
    await sleepCancel(1000, () => cancelado);
  }
  cuenta.hidden = true;

  const capturados = [];
  $("obsTitulo").textContent = mostrar ? "Observa la obra" : "Observación en curso";
  $("obsSub").textContent = "La cámara registra tu expresión. No hace falta posar.";
  const inicio = performance.now();
  while (!cancelado && performance.now() - inicio < dur) {
    const t = performance.now() - inicio;
    const blob = await capturarBlob(CFG.OBS_ANCHO, 0.85);
    if (blob) capturados.push({ blob, t: Math.round(t) });
    prog.style.width = Math.min(100, (t / dur) * 100) + "%";
    await sleepCancel(Math.max(0, CFG.OBS_INTERVALO_MS - (performance.now() - inicio - t)), () => cancelado);
  }
  prog.style.width = "100%";
  capa.hidden = true; state.observando = false; state.cancelarObs = null;
  if (state.enfocarAlCerrarObs && state.enfocarAlCerrarObs.focus) {
    try { state.enfocarAlCerrarObs.focus(); } catch {}
    state.enfocarAlCerrarObs = null;
  }
  if (cancelado) { toast("Observación cancelada.", "warn"); announce("Observación cancelada"); return; }
  if (!capturados.length) { mostrarError("errCam", "No se pudo capturar ningún fotograma. Inténtalo de nuevo."); toast("Sin fotogramas capturados.", "err"); return; }
  guardarFrames(capturados, `${capturados.length} fotogramas en ${(dur / 1000).toFixed(0)} s`);
  announce(`Observación completada: ${capturados.length} fotogramas`);
}
async function fotoUnica() {
  const blob = await capturarBlob(CFG.OBS_ANCHO, 0.9);
  if (!blob) { mostrarError("errCam", "La cámara aún no está lista. Espera un segundo e inténtalo de nuevo."); toast("Cámara no lista.", "warn"); return; }
  guardarFrames([{ blob, t: 0 }], "1 foto capturada");
  toast("Foto capturada.", "ok", 1800);
}
function guardarFrames(lista, texto) {
  state.frames.forEach(f => f.url && URL.revokeObjectURL(f.url));
  state.frames = lista.map(f => ({ ...f, url: URL.createObjectURL(f.blob) }));
  const strip = $("strip"); strip.replaceChildren();
  state.frames.forEach((f, i) => { const im = el("img"); im.src = f.url; im.alt = `Fotograma ${i + 1}`; strip.append(im); });
  $("framesTxt").textContent = "✓ " + texto;
  $("framesBox").hidden = false;
  $("btnObservar").textContent = `Repetir observación (${$("selDur").value} s)`;
  pill("pillCam", state.stream ? "Capturado · en vivo" : "Capturado", "ok");
  actualizarBoton();
}
function descartarFrames() {
  state.frames.forEach(f => f.url && URL.revokeObjectURL(f.url)); state.frames = [];
  $("framesBox").hidden = true; actualizarBoton();
  $("btnObservar").textContent = `Iniciar observación (${$("selDur").value} s)`;
  pill("pillCam", state.stream ? "En vivo" : "Apagada", state.stream ? "live" : "");
}
$("btnObservar").addEventListener("click", observar);
$("btnFoto").addEventListener("click", fotoUnica);
$("btnDescartar").addEventListener("click", descartarFrames);
$("btnCancelarObs").addEventListener("click", () => state.cancelarObs && state.cancelarObs());
document.addEventListener("keydown", e => { if (e.key === "Escape" && state.cancelarObs) state.cancelarObs(); });

/* ============================================================ PASO 2 — envío */
function animarPipeline() {
  const ol = $("pipeline"); ol.replaceChildren();
  const lis = MODULOS.map(([, n]) => { const li = el("li", "", n); ol.append(li); return li; });
  let i = 0;
  const id = setInterval(() => { if (i < lis.length) { lis[i].classList.add("run"); if (i) lis[i - 1].classList.replace("run", "done"); i++; } }, 260);
  return () => { clearInterval(id); lis.forEach(li => { li.classList.remove("run"); li.classList.add("done"); }); };
}

async function analizar() {
  if (state.analizando) return;
  state.analizando = true;
  $("btnAnalizar").disabled = true;
  try {
    detenerCamara();
    irAPaso(2);
    const fin = animarPipeline();
    const fd = new FormData();
    if (state.obra) fd.append("obra", state.obra, state.obra.name);
    state.frames.forEach((f, i) => fd.append("frames", f.blob, `frame_${i}.jpg`));
    if (state.frames.length) fd.append("tiempos", JSON.stringify(state.frames.map(f => f.t)));
    const ctrl = new AbortController(); const to = setTimeout(() => ctrl.abort(), CFG.TIMEOUT_MS);
    const t0 = performance.now();
    try {
      const r = await fetch(API + "/api/analizar", { method: "POST", body: fd, signal: ctrl.signal });
      if (!r.ok) throw new Error("HTTP " + r.status);
      state.resultado = await r.json();
    } catch (e) {
      state.resultado = { estado: "error", obra: null, rostro: null, coincidencia: null, tiempos_ms: {}, interpretacion: "",
        excepciones: [{ tipo: "servidor", gravedad: "error", modulo: "GUI",
          mensaje: e.name === "AbortError" ? "El servidor tardó demasiado en responder. Inténtalo de nuevo."
            : `No se pudo contactar al modelo (${e.message || "error de red"}). Verifica que el servidor esté activo.` }] };
    } finally { clearTimeout(to); }
    await sleep(Math.max(0, 900 - (performance.now() - t0)));
    fin(); await sleep(250);
    const r = state.resultado;
    if (r.excepciones.some(x => x.gravedad !== "info")) { renderExcepciones(); irAPaso(3); announce("Análisis con excepciones"); }
    else { renderResultado(); irAPaso(4); announce("Análisis completado"); }
  } finally {
    state.analizando = false;
    actualizarBoton();
  }
}
$("btnAnalizar").addEventListener("click", analizar);

/* ============================================================ PASO 3 — excepciones */
function renderExcepciones() {
  const r = state.resultado, ul = $("listaExc"); ul.replaceChildren();
  const errores = r.excepciones.filter(x => x.gravedad === "error").length;
  $("leadExc").textContent = errores
    ? "El análisis encontró problemas que impiden parte del resultado. Revisa y ajusta la entrada."
    : "El análisis terminó con avisos que conviene revisar antes de interpretar el resultado.";
  r.excepciones.forEach(x => {
    const li = el("li", "exc " + gravedadCls(x.gravedad));
    li.append(el("span", "ic", ICONOS[x.tipo] || "⚠️"));
    li.append(el("b", null, TITULOS[x.tipo] || (x.gravedad === "error" ? "Error" : x.gravedad === "info" ? "Información" : "Aviso")));
    li.append(el("span", "mod", x.modulo || ""));
    li.append(el("span", null, x.mensaje || "Situación no especificada."));
    ul.append(li);
  });
  const hayAlgo = (r.obra && r.obra.categoria) || (r.rostro && !r.rostro.sin_rostro) || (r.obra && r.obra.paleta);
  $("btnVerResultado").disabled = !hayAlgo;
}
$("btnReintentar").addEventListener("click", () => irAPaso(1));
$("btnVerResultado").addEventListener("click", () => { renderResultado(); irAPaso(4); });
$("btnVolverExc").addEventListener("click", () => { renderExcepciones(); irAPaso(3); });

/* ============================================================ PASO 4 — Resultado */
function stage(src, cajas, opts = {}) {
  const wrap = el("div", "stage" + (opts.espejo ? " mirror-img" : ""));
  const img = el("img"); img.alt = opts.alt || "Imagen analizada"; img.src = src; img.loading = "lazy"; wrap.append(img);
  const mx = b => opts.espejo ? 1 - b.x - b.w : b.x;
  const lista = cajas || [];
  lista.forEach((b, i) => {
    const d = el("div", "box " + (opts.cls || "") + (i > 0 ? " bottom" : b.y < 0.08 ? " top" : ""));
    d.style.cssText = `left:${mx(b) * 100}%;top:${b.y * 100}%;width:${b.w * 100}%;height:${b.h * 100}%`;
    if (b.etiqueta) d.append(el("span", null, b.etiqueta));
    wrap.append(d);
  });
  (opts.puntos || []).forEach(([x, y]) => {
    const p = el("i", "lm"); p.style.left = (opts.espejo ? 1 - x : x) * 100 + "%"; p.style.top = y * 100 + "%"; wrap.append(p);
  });
  return wrap;
}

function gaugeSVG(valor, nivel) {
  const p = valor == null ? 0 : Math.round(valor * 100);
  return `<svg viewBox="0 0 200 112" role="img" aria-label="Coincidencia emocional ${valor == null ? "no disponible" : p + " por ciento"}">
    <path d="M14 100 A86 86 0 0 1 186 100" pathLength="100" fill="none" stroke="var(--line-2)" stroke-width="16" stroke-linecap="round"/>
    <path d="M14 100 A86 86 0 0 1 186 100" pathLength="100" fill="none" stroke="var(--ink)" stroke-width="16" stroke-linecap="round" stroke-dasharray="${p} 100"/>
  </svg><div class="val">${valor == null ? "—" : p + "%"}</div><div class="cap">Coincidencia emocional</div>
  ${nivel ? `<span class="lvl">Coincidencia ${nivel}</span>` : ""}`;
}

function escalaValencia(vo, vr) {
  if (vo == null && vr == null) return "";
  const X = v => 10 + (v + 1) / 2 * 180;
  const marca = (v, color, etiqueta, arriba) => v == null ? "" : `
    <g data-tip="<b>${etiqueta}</b>: valencia ${v.toFixed(2)}">
      <circle cx="${X(v)}" cy="30" r="7" fill="${color}" stroke="var(--card)" stroke-width="2"/>
      <text x="${X(v)}" y="${arriba ? 16 : 52}" text-anchor="middle" style="fill:var(--ink);font-weight:600">${etiqueta}</text>
    </g>`;
  return `<div class="chart" style="margin-top:14px"><svg viewBox="0 0 200 76" role="img" aria-label="Valencia de obra y visitante">
    <defs><linearGradient id="gv"><stop offset="0" stop-color="#eb6834"/><stop offset=".5" stop-color="#c9ccd8"/><stop offset="1" stop-color="#2a78d6"/></linearGradient></defs>
    <rect x="10" y="27" width="180" height="6" rx="3" fill="url(#gv)"/>
    <text x="10" y="74" style="font-size:9px">−1 desagradable</text><text x="190" y="74" text-anchor="end" style="font-size:9px">agradable +1</text>
    ${marca(vo, "#232B5D", "Obra", true)}${marca(vr, "#2F9E6A", "Visitante", false)}
  </svg></div>`;
}

function renderResultado() {
  const r = state.resultado;
  const obra = r.obra && r.obra.categoria ? r.obra : null;
  const pal = r.obra && r.obra.paleta ? r.obra.paleta : null;
  const ro = r.rostro && !r.rostro.sin_rostro ? r.rostro : null;
  $("btnVolverExc").hidden = !r.excepciones.length;

  const rObra = $("rObra"); rObra.replaceChildren();
  if (state.obraURL && (obra || pal)) {
    const cont = el("div"); let vista = "original";
    const pintar = () => {
      cont.replaceChildren(vista === "gradcam" && obra && obra.gradcam
        ? stage(obra.gradcam, [], { alt: "Mapa Grad-CAM de la obra" })
        : vista === "rostros" ? stageRostros(state.obraURL, r.obra.rostros, true)
        : stage(state.obraURL, obra ? obra.regiones : [], { alt: "Obra analizada" }));
    };
    if (obra && obra.gradcam) {
      const tg = el("div", "toggle"); tg.setAttribute("role", "group"); tg.setAttribute("aria-label", "Vista de la obra");
      const opciones = [["original", "Regiones"], ["gradcam", "Grad-CAM"]];
      if (r.obra.rostros && r.obra.rostros.n) opciones.push(["rostros", `Rostros (${r.obra.rostros.n})`]);
      opciones.forEach(([k, t]) => {
        const b = el("button", null, t); b.setAttribute("aria-pressed", String(k === vista));
        b.onclick = () => { vista = k; [...tg.children].forEach(c => c.setAttribute("aria-pressed", String(c === b))); pintar(); };
        tg.append(b);
      });
      rObra.append(tg);
    }
    pintar(); rObra.append(cont);
    if (obra) {
      const t = el("div", "tag"); t.innerHTML = `${esc(obra.etiqueta)}: ${pct(obra.confianza)}<small>${esc(obra.descripcion)}</small>`;
      rObra.append(t);
    } else rObra.append(el("div", "tag", "Solo paleta (CNN no disponible)"));
  } else rObra.append(el("div", "vacio", "Sin imagen de obra analizada."));

  const rRo = $("rRostro"); rRo.replaceChildren();
  if (ro) {
    rRo.append(stage(ro.frame, [{ ...ro.caja, etiqueta: ro.etiqueta }], { cls: "face", espejo: true, puntos: ro.puntos, alt: "Fotograma representativo del visitante" }));
    const t = el("div", "tag face"); t.innerHTML = `${esc(ro.etiqueta)} (${pct(ro.confianza)})<small>${ro.frames_con_rostro} de ${ro.frames_analizados} fotogramas con rostro</small>`;
    rRo.append(t);
  } else if (r.rostro && r.rostro.sin_rostro) {
    rRo.append(stage(r.rostro.frame, [], { espejo: true, alt: "Fotograma sin rostro detectado" }));
    rRo.append(el("div", "tag", "Rostro no detectado"));
  } else rRo.append(el("div", "vacio", "Sin rostro analizado."));

  $("gauge").innerHTML = gaugeSVG(r.coincidencia, r.nivel_coincidencia) + escalaValencia(obra ? obra.valencia : null, ro ? (r.coincidencia_detalle ? r.coincidencia_detalle.valencia_visitante : ro.valencia) : null);
  $("resumen").textContent = r.interpretacion || "";
  const chips = $("excChips"); chips.replaceChildren();
  r.excepciones.forEach(x => { const c = el("span", gravedadCls(x.gravedad), `${ICONOS[x.tipo] || "⚠️"} ${TITULOS[x.tipo] || "Aviso"}`); c.dataset.tip = esc(x.mensaje); chips.append(c); });

  renderClasificacion(obra); renderPaleta(pal); renderEmociones(ro); renderGestos(ro); renderTiempo(ro); renderTiempos(r.tiempos_ms);
  renderRostrosObra(r.obra && r.obra.rostros, $("cardRostrosObra"), "Rostros dentro de la obra");
}

function renderClasificacion(o, c = $("cardClasif")) {
  c.replaceChildren();
  if (!o) return;
  c.innerHTML = `<h3>Clasificación de la obra <span class="badge ${o.revision_requerida ? "warn" : "ok"}">${o.revision_requerida ? "Revisión requerida" : "Confiable"}</span></h3>
    <p class="note">CNN entrenada en el Parcial 2 · Softmax sobre 2 logits · umbral de aceptación ${pct(o.umbral)}</p>`;
  const ul = el("ul", "bars");
  [["positivo", "Positiva", "var(--pos)"], ["negativo", "Negativa", "var(--neg)"]].forEach(([k, n, col]) => {
    const v = o.puntuaciones[k];
    const li = el("li", o.categoria === k ? "top" : ""); li.dataset.tip = `<b>${n}</b>: ${pct(v, 1)} · logit ${o.logits[k === "negativo" ? 0 : 1]}`;
    li.append(el("span", "lab", n));
    const tr = el("span", "track"), fi = el("span", "fill"), th = el("span", "thr");
    fi.style.width = v * 100 + "%"; fi.style.background = col; th.style.left = o.umbral * 100 + "%"; th.title = "Umbral";
    tr.append(fi, th); li.append(tr, el("span", "num", pct(v)));
    ul.append(li);
  });
  c.append(ul);
  const kv = el("div", "kv");
  kv.innerHTML = `<div><span>Logits [neg, pos]</span><b>${o.logits.map(v => v.toFixed(2)).join(", ")}</b></div>
    <div><span>Tensor de entrada</span><b>[${o.tensor.forma.join(", ")}]</b></div>
    <div><span>Dimensiones</span><b>${o.dimensiones.ancho}×${o.dimensiones.alto}</b></div>
    <div><span>Brillo · nitidez</span><b>${o.calidad.brillo} · ${Math.round(o.calidad.nitidez)}</b></div>`;
  c.append(kv);
}

function renderPaleta(p, c = $("cardPaleta")) {
  c.replaceChildren();
  if (!p) return;
  c.innerHTML = `<h3>Paleta cromática dominante</h3><p class="note">K-Means (k=${p.colores.length}) en espacio CIE Lab · clic en un color para copiar su HEX</p>`;
  const bar = el("div", "palette-bar"); bar.setAttribute("role", "img");
  bar.setAttribute("aria-label", "Distribución de colores: " + p.colores.map(x => `${x.nombre} ${pct(x.proporcion)}`).join(", "));
  p.colores.forEach(x => { const i = el("i"); i.style.cssText = `flex:${x.proporcion};background:${x.hex}`; i.dataset.tip = `<b>${x.hex}</b> · ${x.nombre} · ${pct(x.proporcion, 1)}`; bar.append(i); });
  c.append(bar);
  const sw = el("div", "swatches");
  p.colores.forEach(x => {
    const b = el("button", "sw"); b.type = "button"; b.title = "Copiar " + x.hex;
    b.innerHTML = `<i style="background:${x.hex}"></i><span><b>${x.hex}</b>${esc(x.nombre)} · ${pct(x.proporcion)}</span>`;
    b.onclick = () => {
      const done = () => {
        b.querySelector("b").textContent = "¡Copiado!";
        setTimeout(() => b.querySelector("b").textContent = x.hex, 900);
        toast(`Copiado ${x.hex}`, "ok", 1400);
      };
      if (navigator.clipboard && navigator.clipboard.writeText) {
        navigator.clipboard.writeText(x.hex).then(done).catch(() => toast("No se pudo copiar al portapapeles.", "warn"));
      } else done();
    };
    sw.append(b);
  });
  c.append(sw);
  const kv = el("div", "kv");
  kv.innerHTML = `<div><span>Brillo medio</span><b>${pct(p.brillo_medio)}</b></div><div><span>Saturación media</span><b>${pct(p.saturacion_media)}</b></div>
    <div><span>Colores cálidos</span><b>${pct(p.proporcion_calida)}</b></div><div><span>Colores fríos/neutros</span><b>${pct(1 - p.proporcion_calida)}</b></div>`;
  c.append(kv);
}

function renderEmociones(ro, c = $("cardEmociones")) {
  c.replaceChildren();
  if (!ro) return;
  c.innerHTML = `<h3>Emociones del visitante</h3><p class="note">FER+ · probabilidad media en los ${ro.frames_con_rostro} fotograma(s) con rostro válido</p>`;
  const ul = el("ul", "bars"); renderBarras(ul, ro.probabilidades); c.append(ul);
}

function renderGestos(ro, c = $("cardGestos")) {
  c.replaceChildren();
  if (!ro) return;
  const g = ro.gestos || {};
  const incl = g.inclinacion_cabeza_grados, giro = g.giro_relativo;
  c.innerHTML = `<h3>Gestos faciales observables</h3><p class="note">Calculados con los 5 puntos faciales de YuNet (ojos, nariz y comisuras), promediados en la secuencia</p>
  <div class="kv">
    <div><span>Inclinación de cabeza</span><b>${incl == null ? "—" : incl.toFixed(1) + "°"}</b></div>
    <div><span>Giro (yaw relativo)</span><b>${giro == null ? "—" : (Math.abs(giro) < 0.1 ? "De frente" : giro > 0 ? "Hacia un lado" : "Hacia el otro")}</b></div>
    <div><span>Ancho de boca / ojos</span><b>${g.ancho_boca_relativo == null ? "—" : g.ancho_boca_relativo.toFixed(2)}</b></div>
    <div><span>Mirando al frente</span><b>${g.porcentaje_mirando_al_frente == null ? "—" : pct(g.porcentaje_mirando_al_frente)}</b></div>
    <div><span>Cambios de expresión</span><b>${ro.cambios_de_expresion}</b></div>
    <div><span>Cámara: brillo · nitidez</span><b>${ro.calidad.brillo} · ${Math.round(ro.calidad.nitidez)}</b></div>
    <div><span>Calificación del rostro</span><b>${ro.calificacion ? `${Math.round(ro.calificacion.puntaje)}/100 · ${ro.calificacion.nivel}` : "—"}</b></div>
  </div>${ro.calificacion && ro.calificacion.componentes ? componentesCalif(ro.calificacion.componentes) : ""}`;
}

function renderTiempo(ro) {
  const c = $("cardTiempo"); c.replaceChildren();
  if (!ro) return;
  const pts = ro.linea_tiempo;
  c.innerHTML = `<h3>Reacción a lo largo de la observación</h3><p class="note">Valencia estimada de la expresión en cada fotograma (−1 desagradable … +1 agradable). Cada punto toma el color de su emoción dominante.</p>`;
  if (pts.length < 2) { c.append(el("p", "note", "Se capturó una sola foto: usa «Iniciar observación» para obtener la secuencia temporal.")); return; }
  const W = 720, H = 200, L = 40, R = 12, T = 12, B = 30;
  const tmax = Math.max(...pts.map(p => p.t), 1);
  const X = t => L + (t / tmax) * (W - L - R), Y = v => T + (1 - (v + 1) / 2) * (H - T - B);
  let path = "", abierto = false;
  pts.forEach(p => { if (p.rostro && p.valencia != null) { path += (abierto ? "L" : "M") + X(p.t).toFixed(1) + " " + Y(p.valencia).toFixed(1); abierto = true; } else abierto = false; });
  const grid = [-1, -0.5, 0, 0.5, 1].map(v => `<line x1="${L}" x2="${W - R}" y1="${Y(v)}" y2="${Y(v)}" ${v === 0 ? 'style="stroke:var(--muted)"' : ""}/><text x="${L - 8}" y="${Y(v) + 4}" text-anchor="end">${v > 0 ? "+" : ""}${v}</text>`).join("");
  const ticks = Array.from({ length: 6 }, (_, i) => tmax * i / 5).map(t => `<text x="${X(t)}" y="${H - 8}" text-anchor="middle">${(t / 1000).toFixed(1)} s</text>`).join("");
  const dots = pts.map(p => p.rostro && p.valencia != null
    ? `<g data-tip="<b>${(p.t / 1000).toFixed(1)} s</b> · ${EMO[p.emocion].es} ${pct(p.confianza)}<br>valencia ${p.valencia.toFixed(2)}"><circle cx="${X(p.t)}" cy="${Y(p.valencia)}" r="12" fill="transparent"/><circle cx="${X(p.t)}" cy="${Y(p.valencia)}" r="5" fill="${EMO[p.emocion].color}" stroke="var(--card)" stroke-width="2"/></g>`
    : `<g data-tip="<b>${(p.t / 1000).toFixed(1)} s</b> · sin rostro"><line x1="${X(p.t)}" x2="${X(p.t)}" y1="${H - B - 6}" y2="${H - B + 2}" stroke="#C2503A" stroke-width="2"/></g>`).join("");
  const wrap = el("div", "chart");
  wrap.innerHTML = `<svg viewBox="0 0 ${W} ${H}" role="img" aria-label="Línea de tiempo de la valencia facial"><g class="grid">${grid}</g>${ticks}
    <path d="${path}" fill="none" stroke="var(--ink)" stroke-width="2" stroke-linejoin="round" stroke-linecap="round"/>${dots}</svg>`;
  c.append(wrap);
  const usadas = [...new Set(pts.filter(p => p.emocion).map(p => p.emocion))];
  const lg = el("div", "legend");
  lg.innerHTML = usadas.map(k => `<span><i style="background:${EMO[k].color}"></i>${EMO[k].es}</span>`).join("") +
    (pts.some(p => !p.rostro) ? `<span><i style="background:#C2503A;border-radius:1px;width:3px"></i>Sin rostro</span>` : "");
  c.append(lg);
  const det = el("details", "tabla");
  det.innerHTML = `<summary>Ver tabla de fotogramas</summary><table class="mini"><thead><tr><th>Tiempo</th><th>Rostro</th><th>Emoción</th><th>Confianza</th><th>Valencia</th></tr></thead><tbody>${
    pts.map(p => `<tr><td>${(p.t / 1000).toFixed(1)} s</td><td>${p.rostro ? "Sí" : "No"}</td><td>${p.emocion ? EMO[p.emocion].es : "—"}</td><td>${p.confianza != null ? pct(p.confianza) : "—"}</td><td>${p.valencia != null ? p.valencia.toFixed(2) : "—"}</td></tr>`).join("")}</tbody></table>`;
  c.append(det);
}

function renderTiempos(t, c = $("cardTiempos")) {
  c.replaceChildren();
  if (!t || !t.total) return;
  c.innerHTML = `<h3>Tiempo de procesamiento por módulo</h3><p class="note">Total en el servidor: ${Math.round(t.total)} ms</p>`;
  const nombres = { image_loader: "ImageLoader", rostros_obra: "Rostros en la imagen", preprocesamiento: "Preprocesamiento", cnn: "CNN",
    gradcam: "Grad-CAM", paleta: "Paleta", gestos: "Gestos (cámara)" };
  const filas = Object.entries(nombres).filter(([k]) => t[k] != null);
  const max = Math.max(...filas.map(([k]) => t[k]), 1);
  const ul = el("ul", "bars ancho");
  filas.forEach(([k, n]) => {
    const li = el("li"); li.dataset.tip = `<b>${n}</b>: ${t[k]} ms`;
    li.append(el("span", "lab", n));
    const tr = el("span", "track"), fi = el("span", "fill"); fi.style.width = (t[k] / max * 100) + "%"; fi.style.background = "var(--ink)";
    tr.append(fi); li.append(tr, el("span", "num", Math.round(t[k]) + " ms")); ul.append(li);
  });
  c.append(ul);
}

function stageRostros(src, rostros, etiquetas) {
  const cajas = (rostros && rostros.detecciones || []).map(d => ({ ...d.caja, etiqueta: etiquetas ? `#${d.id} ${d.etiqueta || ""}` : "" }));
  return stage(src, cajas, { cls: "face", alt: "Rostros detectados en la imagen" });
}
function componentesCalif(c) {
  const nombres = { deteccion: "Detección", tamano: "Tamaño", frontalidad: "Frontalidad", nitidez: "Nitidez", iluminacion: "Iluminación" };
  const ul = Object.entries(nombres).map(([k, n]) => `<li data-tip="<b>${n}</b>: ${pct(c[k])}"><span class="lab">${n}</span><span class="track"><span class="fill" style="width:${c[k] * 100}%;background:var(--ok)"></span></span><span class="num">${pct(c[k])}</span></li>`).join("");
  return `<p class="note" style="margin:12px 0 6px">Componentes de la calificación (30 % detección · 20 % tamaño · 20 % frontalidad · 15 % nitidez · 15 % iluminación)</p><ul class="bars">${ul}</ul>`;
}
function califHTML(cal) {
  return cal ? `<span class="calif" data-tip="Calificación del rostro: ${cal.puntaje}/100"><i style="--v:${cal.puntaje}%"></i>${Math.round(cal.puntaje)} · ${cal.nivel}</span>` : "—";
}
function renderRostrosObra(rostros, c, titulo) {
  c.replaceChildren();
  if (!rostros) return;
  if (!rostros.n) { c.innerHTML = `<h3>${titulo}</h3><p class="note">No se detectaron rostros en la imagen (umbral del detector ≥ 60 %).</p>`; return; }
  c.innerHTML = `<h3>${titulo} <span class="badge ok">${rostros.n}</span></h3>
    <p class="note">YuNet + FER+ sobre la propia imagen · emoción predominante: <b>${EMO[rostros.emocion_predominante] ? EMO[rostros.emocion_predominante].es : "—"}</b> · calificación media ${rostros.calificacion_media != null ? Math.round(rostros.calificacion_media) + "/100" : "—"}</p>
    <table class="mini"><thead><tr><th>#</th><th>Emoción</th><th>Confianza</th><th>Valencia</th><th>Detección</th><th>Calificación</th></tr></thead><tbody>${
    rostros.detecciones.map(d => `<tr><td>${d.id}</td><td class="l"><i style="display:inline-block;width:9px;height:9px;border-radius:50%;margin-right:6px;background:${d.emocion ? EMO[d.emocion].color : "#ccc"}"></i>${d.etiqueta || "—"}</td><td>${d.confianza != null ? pct(d.confianza) : "—"}</td><td>${d.valencia != null ? d.valencia.toFixed(2) : "—"}</td><td>${pct(d.score_deteccion)}</td><td>${califHTML(d.calificacion)}</td></tr>`).join("")}</tbody></table>`;
  const p = rostros.principal;
  if (p && p.probabilidades) {
    c.append(el("p", "note", `Rostro principal (#${p.id}): probabilidades por emoción`));
    const ul = el("ul", "bars"); renderBarras(ul, p.probabilidades); c.append(ul);
  }
  if (p && p.calificacion) c.insertAdjacentHTML("beforeend", componentesCalif(p.calificacion.componentes));
}

/* ---------- reporte ---------- */
$("btnJSON").addEventListener("click", () => {
  const r = { generado: new Date().toISOString(), obra_archivo: state.obra ? state.obra.name : null, ...state.resultado };
  const blob = new Blob([JSON.stringify(r, null, 2)], { type: "application/json" });
  const a = el("a"); a.href = URL.createObjectURL(blob); a.download = `reporte_coincidencia_${Date.now()}.json`; a.click();
  setTimeout(() => URL.revokeObjectURL(a.href), 2000);
  toast("Reporte descargado.", "ok", 1800);
});
$("btnImprimir").addEventListener("click", () => window.print());

/* ---------- reiniciar ---------- */
function reiniciar() {
  detenerCamara(); quitarObra(); descartarFrames(); mostrarError("errCam"); mostrarError("errObra");
  state.resultado = null; pill("pillCam", "Apagada"); actualizarBoton(); irAPaso(1);
}
$("btnNueva").addEventListener("click", reiniciar);

/* ---------- cleanup global ---------- */
window.addEventListener("pagehide", () => {
  detenerCamara();
  if (state.obraURL) URL.revokeObjectURL(state.obraURL);
  state.frames.forEach(f => f.url && URL.revokeObjectURL(f.url));
  ds.items.forEach(item => { if (item.url && item.url.startsWith("blob:")) URL.revokeObjectURL(item.url); });
});

/* ============================================================ Acerca del modelo */
async function abrirAcerca() {
  const dlg = $("dlgAcerca"), body = $("dlgBody"); dlg.showModal();
  let m = null;
  try { const r = await fetch(API + "/api/metricas"); if (r.ok) m = await r.json(); } catch { /* sin servidor */ }
  const flujo = `<div class="flow"><span>ImageLoader</span><em>→</em><span>PreprocesamientoImagen</span><em>→</em><span>ClasificadorDeObras</span><em>+</em><span>ExtractorDePaleta</span></div>
    <div class="flow"><span>CámaraAutorizada</span><em>→</em><span>CapturaDeFrame</span><em>→</em><span>AnalizarGestosFaciales</span></div>
    <div class="flow"><em>→</em><span>ResultadoProceso</span><em>→</em><span>Integración</span><em>→</em><span>Presentación y uso</span></div>`;
  let html = `<p>La obra se clasifica en <b>positiva</b> o <b>negativa</b> con la CNN de obras (ResNet18 con transfer learning); el rostro se detecta con YuNet y su expresión con FER+ (8 emociones). Ambas lecturas se llevan a una misma escala de <b>valencia</b> (−1 a +1) y la coincidencia es <code>1 − |v_obra − v_visitante| / 2</code>. Por defecto (fórmula v3) la valencia del visitante usa solo la parte <b>expresiva</b> del rostro, <code>v / (1 − p(neutral))</code>. Si la intensidad <code>1 − p(neutral)</code> es baja, el resultado se marca como poco concluyente.</p>
  <h3>Arquitectura</h3>${flujo}`;
  if (m) {
    const p = m.prueba, cm = p.matriz_confusion;
    html += `<h3>Desempeño de la CNN (${m.k_folds ? "validación cruzada de " + m.k_folds + " folds" : "conjunto de prueba"}, ${p.n} imágenes)</h3>
    <div class="kv"><div><span>Accuracy</span><b>${p.accuracy}</b></div><div><span>Precision</span><b>${p.precision}</b></div><div><span>Recall</span><b>${p.recall}</b></div><div><span>F1-score</span><b>${p.f1}</b></div></div>
    <table class="mini"><thead><tr><th>Real \\ Predicho</th><th>Negativa</th><th>Positiva</th></tr></thead>
    <tbody><tr><th>Negativa</th><td>${cm[0][0]}</td><td>${cm[0][1]}</td></tr><tr><th>Positiva</th><td>${cm[1][0]}</td><td>${cm[1][1]}</td></tr></tbody></table>
    <h3>Curva de entrenamiento</h3>${curvaPerdida(m.historial, m.mejor_epoca)}
    <p>${esc(m.modelo)}. Mejor época: ${m.mejor_epoca}. ${m.k_folds ? "Cada imagen se evaluó con un modelo que no la vio; accuracy por fold " + m.accuracy_media + " ± " + m.accuracy_std + (p.auc != null ? " · AUC " + p.auc : "") + ". Modelo final entrenado con las " + m.particion.entrenamiento + " imágenes" : "Partición: " + m.particion.entrenamiento + " / " + m.particion.validacion + " / " + m.particion.prueba}. Normalización: media ${m.media.map(v => v.toFixed(3)).join(", ")} · std ${m.std.map(v => v.toFixed(3)).join(", ")}.</p>`;
  } else html += `<p>No hay métricas disponibles (el servidor no respondió o el modelo no está entrenado).</p>`;
  html += `<h3>Valencia asignada a cada emoción</h3><p>Felicidad +1 · Sorpresa +0.3 · Neutral 0 · Desprecio −0.5 · Miedo −0.7 · Tristeza −0.8 · Asco −0.8 · Enojo −0.9</p>
  <p><small>Los resultados son exploratorios: la CNN se entrenó con ${m ? m.particion.entrenamiento : "pocas"} imágenes y la expresión facial no equivale a la emoción interna.</small></p>`;
  body.innerHTML = html;
}
function curvaPerdida(h, mejor) {
  const W = 640, H = 200, L = 40, R = 90, T = 10, B = 26, n = h.train_loss.length;
  const vals = [...h.train_loss, ...h.val_loss], lo = Math.min(...vals) * 0.95, hi = Math.max(...vals) * 1.02;
  const X = i => L + (i / Math.max(n - 1, 1)) * (W - L - R), Y = v => T + (1 - (v - lo) / (hi - lo)) * (H - T - B);
  const linea = arr => arr.map((v, i) => (i ? "L" : "M") + X(i).toFixed(1) + " " + Y(v).toFixed(1)).join("");
  const serie = (arr, col, nom) => `<path d="${linea(arr)}" fill="none" stroke="${col}" stroke-width="2" stroke-linejoin="round"/>` +
    arr.map((v, i) => `<g data-tip="<b>${nom}</b> · época ${i + 1}: ${v.toFixed(3)}"><circle cx="${X(i)}" cy="${Y(v)}" r="9" fill="transparent"/></g>`).join("") +
    `<text x="${X(n - 1) + 8}" y="${Y(arr[n - 1]) + 4}" style="fill:#232B5D">${nom}</text>`;
  const yt = [lo, (lo + hi) / 2, hi].map(v => `<line x1="${L}" x2="${W - R}" y1="${Y(v)}" y2="${Y(v)}"/><text x="${L - 6}" y="${Y(v) + 4}" text-anchor="end">${v.toFixed(2)}</text>`).join("");
  const xt = h.train_loss.map((_, i) => (i % 2 === 0 || i === n - 1) ? `<text x="${X(i)}" y="${H - 6}" text-anchor="middle">${i + 1}</text>` : "").join("");
  return `<div class="chart"><svg viewBox="0 0 ${W} ${H}" role="img" aria-label="Pérdida de entrenamiento y validación por época">
    <g class="grid">${yt}</g>${xt}<line x1="${X(mejor - 1)}" x2="${X(mejor - 1)}" y1="${T}" y2="${H - B}" stroke="#9aa0b4" stroke-dasharray="4 4"/>
    <text x="${X(mejor - 1) + 4}" y="${T + 10}">mejor época</text>
    ${serie(h.train_loss, "#2a78d6", "Entrenamiento")}${serie(h.val_loss, "#eb6834", "Validación")}</svg></div>
    <div class="legend"><span><i style="background:#2a78d6"></i>Pérdida de entrenamiento</span><span><i style="background:#eb6834"></i>Pérdida de validación</span></div>`;
}
$("btnAcerca").addEventListener("click", abrirAcerca);
$("btnCerrarDlg").addEventListener("click", () => $("dlgAcerca").close());

/* =====================================================================
   PESTAÑAS: Obra y visitante  |  Imágenes precargadas
   ===================================================================== */
function cambiarModo(modo, enfocar = false) {
  const ds = modo === "dataset";
  const tabs = [[$("tabVisitante"), !ds], [$("tabDataset"), ds]];
  tabs.forEach(([t, sel]) => { t.setAttribute("aria-selected", String(sel)); t.tabIndex = sel ? 0 : -1; });
  $("vistaVisitante").hidden = ds; $("vistaDataset").hidden = !ds;
  if (ds) { if (state.stream) detenerCamara(); cargarEstadoDataset(); }   // libera la cámara al salir del flujo con visitante
  if (enfocar) (ds ? $("tabDataset") : $("tabVisitante")).focus();
  window.scrollTo({ top: 0, behavior: "smooth" });
}
$("tabVisitante").addEventListener("click", () => cambiarModo("visitante"));
$("tabDataset").addEventListener("click", () => cambiarModo("dataset"));
[$("tabVisitante"), $("tabDataset")].forEach(t => t.addEventListener("keydown", e => {   // flechas entre pestañas
  if (["ArrowLeft", "ArrowRight", "Home", "End"].includes(e.key)) {
    e.preventDefault();
    cambiarModo(t.id === "tabVisitante" ? "dataset" : "visitante", true);
  }
}));

/* =====================================================================
   IMÁGENES PRECARGADAS — carpeta precarga-sin-analizar (Google Drive)
   Mismo flujo de análisis que una imagen subida.
   ===================================================================== */
const ds = { items: new Map(), cola: [], activos: 0, propias: 0 };
const DS_CONCURRENCIA = 2;
const RUTA_PRECARGA = "Mi unidad / Proyecto-final-de-PDI / dataset usado / precarga-sin-analizar";

async function cargarEstadoDataset() {
  try {
    const e = await (await fetchRetry(API + "/api/dataset/estado")).json();
    pintarEstadoDataset(e);
  } catch { $("dsEstado").textContent = "No se pudo consultar la carpeta de imágenes precargadas (servidor no disponible)."; }
}
function pintarEstadoDataset(e) {
  if (!e) return;
  const carpeta = `<code>${esc(e.carpeta || "precarga-sin-analizar")}</code>`;
  $("dsEstado").innerHTML = e.listo
    ? `<b>${e.n_imagenes} imágenes</b> disponibles<br>${carpeta}`
    : `<b>Carpeta no disponible.</b><br>Se busca en ${carpeta}`;
}
function mostrarErrorDs(err) {
  const box = $("dsError");
  if (!err) { box.hidden = true; return; }
  const consejos = {
    dataset_no_encontrado: `<ol><li>Ejecuta la celda <b>«Localizar el dataset»</b> del notebook y autoriza el acceso a Google Drive.</li>
      <li>Verifica que exista la carpeta <code>${esc(RUTA_PRECARGA)}</code>.</li>
      <li>Si está en otro lugar, cambia <code>config.PRECARGA_DIR</code> en la celda de configuración y vuelve a ejecutar el notebook.</li></ol>`,
    dataset_no_disponible: `<ol><li>Copia imágenes JPG o PNG en <code>${esc(RUTA_PRECARGA)}</code>.</li>
      <li>Pulsa «Volver a leer la carpeta» (no hace falta reiniciar el servidor).</li></ol>`,
  };
  box.innerHTML = `<b>${ICONOS[err.tipo] || "⚠️"} ${esc(TITULOS[err.tipo] || "Error")}</b>${esc(err.mensaje)}${consejos[err.tipo] || ""}`;
  box.hidden = false;
}
async function fetchJSON(url, opts) {
  const r = await fetch(API + url, opts);
  let d; try { d = await r.json(); } catch { throw { tipo: "servidor", mensaje: `Respuesta inválida del servidor (HTTP ${r.status}).` }; }
  if (d && d.estado === "error" && d.error) { if (d.dataset) pintarEstadoDataset(d.dataset); throw d.error; }
  if (!r.ok) throw { tipo: "servidor", mensaje: (d && d.detail) || `HTTP ${r.status}` };
  return d;
}
function setCargando(btn, cargando, texto) {
  if (cargando) { btn.dataset.txt = btn.textContent; btn.innerHTML = `<span class="spin-sm"></span>${texto}`; btn.disabled = true; }
  else { btn.textContent = btn.dataset.txt || btn.textContent; btn.disabled = false; }
}

/* Paso 1: si el visitante no quiere subir una imagen propia, se toma una al azar de la carpeta */
async function obraPrecargada() {
  const btn = $("btnObraPrecargada");
  mostrarError("errObra");
  setCargando(btn, true, "Eligiendo…");
  try {
    const excluir = state.obra ? state.obra.name : "";
    const d = await fetchJSON(`/api/dataset/aleatorias?n=1&excluir=${encodeURIComponent(excluir)}`);
    const info = d.imagenes[0];
    const r = await fetch(API + info.url);
    if (!r.ok) throw { mensaje: `No se pudo cargar «${info.nombre}» (HTTP ${r.status}).` };
    const blob = await r.blob();
    setObra(new File([blob], info.nombre, { type: blob.type || "image/jpeg" }));
  } catch (e) {
    const msg = e.mensaje || "No se pudo contactar al servidor.";
    mostrarError("errObra", msg); toast(msg, "err");
  } finally { setCargando(btn, false); }
}
$("btnObraPrecargada").addEventListener("click", obraPrecargada);

async function pedirAleatorias() {
  const btn = $("btnDsAleatorias");
  mostrarErrorDs(null);
  setCargando(btn, true, "Eligiendo al azar…");
  try {
    const excluir = [...ds.items.keys()].filter(k => !k.startsWith("propia:")).join("|");
    const d = await fetchJSON(`/api/dataset/aleatorias?n=${$("dsN").value}&excluir=${encodeURIComponent(excluir)}`);
    ds.items.clear(); ds.cola = []; $("dsGrid").replaceChildren(); $("dsVacio").hidden = true;
    d.imagenes.forEach(info => agregarTarjeta(info));
    if (d.avisos && d.avisos.length) mostrarErrorDs({ tipo: "dataset_no_disponible", mensaje: `${d.avisos.length} imagen(es) no se pudieron leer y se omitieron.` });
    $("btnDsAnalizarTodas").disabled = false;
    cargarEstadoDataset();
    if ($("chkDsAuto").checked) analizarTodas();
  } catch (e) {
    mostrarErrorDs(e.tipo ? e : { tipo: "servidor", mensaje: "No se pudo contactar al servidor. ¿Sigue ejecutándose el notebook?" });
  } finally { setCargando(btn, false); }
}

function etiquetaParticion(cnn, propia) {
  if (propia) return ["Imagen propia", ""];
  if (!cnn) return ["Nunca vista por la CNN", "prueba"];
  return cnn.particion === "prueba" ? ["Conjunto de prueba de la CNN", "prueba"]
    : [cnn.particion === "entrenamiento" ? "Usada para entrenar la CNN" : "Usada en validación de la CNN", "vista"];
}
function agregarTarjeta(info, archivoPropio) {
  const item = { info, archivo: archivoPropio || null, resultado: null, estado: "pendiente", url: info.url.startsWith("/api") ? API + info.url : info.url };
  ds.items.set(info.nombre, item);
  const card = el("article", "ds-card"); card.dataset.nombre = info.nombre;
  item.card = card;
  $("dsGrid").append(card);
  pintarTarjeta(item);
}
function pintarTarjeta(item) {
  const { info, resultado: r, card } = item;
  const [txtTag, clsTag] = etiquetaParticion(info.cnn, !!item.archivo);
  const rostros = r && r.obra && r.obra.rostros;
  card.className = "ds-card" + (item.estado === "analizando" ? " analizando" : "");
  card.replaceChildren();

  const zona = el("div", "ds-img");
  zona.append(rostros && rostros.n ? stageRostros(item.url, rostros, false) : stage(item.url, [], { alt: info.nombre }));
  const tag = el("span", "ds-tag " + clsTag, txtTag); tag.dataset.tip = info.cnn ? `Etiqueta real en el entrenamiento: <b>${info.cnn.clase}</b>` : "La CNN no vio esta imagen al entrenar";
  zona.append(tag);
  card.append(zona);

  const body = el("div", "ds-body");
  const nm = el("div", "ds-name", info.nombre.replace(/^propia:\d+:/, "")); nm.title = info.nombre; body.append(nm);
  if (item.estado === "pendiente") body.append(el("p", "ds-msg", "Sin analizar."));
  else if (item.estado === "analizando") body.insertAdjacentHTML("beforeend", `<p class="ds-msg"><span class="spin-sm"></span>Analizando: CNN, rostros, emociones…</p>`);
  else if (item.estado === "error") body.append(el("p", "ds-msg err", item.error || "No se pudo analizar la imagen."));
  else if (r) {
    const o = r.obra || {};
    const p = rostros && rostros.principal;
    const g = el("div", "ds-res");
    g.innerHTML = `
      <div><span>CNN</span><b>${o.etiqueta ? `${esc(o.etiqueta)} ${pct(o.confianza)}` : "—"}</b></div>
      <div><span>Rostros</span><b>${rostros ? rostros.n : "—"}</b></div>
      <div><span>Emoción</span><b>${p && p.etiqueta ? `${p.etiqueta} ${pct(p.confianza)}` : "Sin rostro"}</b></div>
      <div><span>Calificación</span><b>${p && p.calificacion ? `${Math.round(p.calificacion.puntaje)} · ${p.calificacion.nivel}` : "—"}</b>
        ${p && p.calificacion ? `<div class="meter"><i style="width:${p.calificacion.puntaje}%"></i></div>` : ""}</div>
      ${r.coherencia != null ? `<div class="full" data-tip="1 − |valencia de la obra − valencia expresiva del rostro| / 2"><span>Coherencia obra ↔ rostro</span><b>${pct(r.coherencia)} · ${r.nivel_coherencia}</b></div>` : ""}`;
    body.append(g);
    const errores = r.excepciones.filter(x => x.gravedad === "error");
    if (errores.length) body.append(el("p", "ds-msg err", errores[0].mensaje));
  }
  card.append(body);

  const acc = el("div", "ds-actions");
  const bA = el("button", null, item.estado === "listo" ? "Reanalizar" : "Analizar");
  bA.disabled = item.estado === "analizando"; bA.onclick = () => encolar(item.info.nombre, true);
  const bD = el("button", null, "Ver detalle"); bD.disabled = !r; bD.onclick = () => abrirDetalle(item.info.nombre);
  const bU = el("button", "ghost", "Usar con mi cámara"); bU.onclick = () => usarConCamara(item.info.nombre);
  acc.append(bA, bD, bU);
  card.append(acc);
}

function encolar(nombre, prioridad) {
  const item = ds.items.get(nombre);
  if (!item || item.estado === "analizando" || ds.cola.includes(nombre)) return;
  prioridad ? ds.cola.unshift(nombre) : ds.cola.push(nombre);
  bombear();
}
function bombear() {
  while (ds.activos < DS_CONCURRENCIA && ds.cola.length) {
    const nombre = ds.cola.shift();
    ds.activos++;
    analizarUna(nombre).finally(() => { ds.activos--; bombear(); });
  }
}
function analizarTodas() { [...ds.items.keys()].forEach(n => { if (ds.items.get(n).estado !== "listo") encolar(n); }); }

async function analizarUna(nombre) {
  const item = ds.items.get(nombre);
  if (!item) return;
  item.estado = "analizando"; pintarTarjeta(item);
  try {
    const fd = new FormData();
    let url = "/api/dataset/analizar";
    if (item.archivo) { url = "/api/analizar"; fd.append("obra", item.archivo, item.archivo.name); fd.append("modo", "dataset"); }
    else fd.append("nombre", nombre);
    item.resultado = await fetchJSON(url, { method: "POST", body: fd });
    item.estado = item.resultado.obra ? "listo" : "error";
    if (!item.resultado.obra) item.error = (item.resultado.excepciones[0] || {}).mensaje;
  } catch (e) {
    item.estado = "error"; item.error = e.mensaje || "No se pudo contactar al servidor.";
    if (e.tipo && e.tipo !== "servidor") mostrarErrorDs(e);
  }
  if (ds.items.get(nombre) === item) pintarTarjeta(item);
}

function analizarPropia(file) {
  const err = validarArchivo(file);
  if (err) { mostrarErrorDs({ tipo: "formato", mensaje: err }); return; }
  mostrarErrorDs(null);
  const nombre = `propia:${++ds.propias}:${file.name}`;
  $("dsVacio").hidden = true;
  agregarTarjeta({ nombre, cnn: null, url: URL.createObjectURL(file) }, file);
  ds.items.get(nombre).card.querySelector(".ds-name").textContent = file.name;
  $("btnDsAnalizarTodas").disabled = false;
  encolar(nombre, true);
}

/* ---------- detalle ---------- */
let detalleActual = null;
function abrirDetalle(nombre) {
  const item = ds.items.get(nombre);
  if (!item || !item.resultado) return;
  detalleActual = nombre;
  const r = item.resultado, o = r.obra || {}, rostros = o.rostros;
  $("detT").textContent = item.archivo ? item.archivo.name : nombre;
  const body = $("detBody"); body.replaceChildren();
  const grid = el("div", "det-grid");

  const izq = el("div", "det-img");
  const cont = el("div");
  const vistas = [];
  if (rostros && rostros.n) vistas.push(["rostros", `Rostros (${rostros.n})`]);
  vistas.push(["regiones", "Regiones CNN"]);
  if (o.gradcam) vistas.push(["gradcam", "Grad-CAM"]);
  let vista = vistas[0][0];
  const pintar = () => cont.replaceChildren(
    vista === "rostros" ? stageRostros(item.url, rostros, true)
      : vista === "gradcam" ? stage(o.gradcam, [], { alt: "Grad-CAM" }) : stage(item.url, o.regiones || [], { alt: "Regiones de la CNN" }));
  const tg = el("div", "toggle");
  vistas.forEach(([k, t]) => { const b = el("button", null, t); b.setAttribute("aria-pressed", String(k === vista));
    b.onclick = () => { vista = k; [...tg.children].forEach(c => c.setAttribute("aria-pressed", String(c === b))); pintar(); }; tg.append(b); });
  izq.append(tg, cont); pintar();
  const sum = el("div", "summary", r.interpretacion); izq.append(sum);
  const chips = el("div", "exc-chips");
  r.excepciones.forEach(x => { const c = el("span", gravedadCls(x.gravedad), `${ICONOS[x.tipo] || "⚠️"} ${TITULOS[x.tipo] || "Aviso"}`); c.dataset.tip = esc(x.mensaje); chips.append(c); });
  izq.append(chips);
  if (r.excepciones.length) {
    const ul = el("ul", "exc-list"); ul.style.marginTop = "10px";
    r.excepciones.forEach(x => { const li = el("li", "exc " + gravedadCls(x.gravedad));
      li.append(el("span", "ic", ICONOS[x.tipo] || "⚠️"), el("b", null, TITULOS[x.tipo] || "Aviso"), el("span", "mod", x.modulo || ""), el("span", null, x.mensaje)); ul.append(li); });
    izq.append(ul);
  }

  const der = el("div", "det-cards");
  const cR = el("article", "card"), cC = el("article", "card");
  renderRostrosObra(rostros, cR, "Rostros, emociones y calificación");
  renderClasificacion(o.categoria ? o : null, cC);
  der.append(cR, cC);
  grid.append(izq, der);
  body.append(grid);

  const abajo = el("div", "details");
  const cP = el("article", "card"), cT = el("article", "card");
  renderPaleta(o.paleta, cP); renderTiempos(r.tiempos_ms, cT);
  abajo.append(cP, cT);
  body.append(abajo);
  $("dlgDetalle").showModal();
}
$("btnCerrarDet").addEventListener("click", () => $("dlgDetalle").close());
$("btnDetUsar").addEventListener("click", () => { $("dlgDetalle").close(); if (detalleActual) usarConCamara(detalleActual); });
$("btnDetJSON").addEventListener("click", () => {
  const item = ds.items.get(detalleActual); if (!item) return;
  const blob = new Blob([JSON.stringify({ generado: new Date().toISOString(), imagen: detalleActual, ...item.resultado }, null, 2)], { type: "application/json" });
  const a = el("a"); a.href = URL.createObjectURL(blob); a.download = `analisis_${detalleActual.replace(/[^\w.-]+/g, "_")}.json`; a.click();
  setTimeout(() => URL.revokeObjectURL(a.href), 2000);
});

/* ---------- llevar una imagen precargada al flujo con cámara ---------- */
async function usarConCamara(nombre) {
  const item = ds.items.get(nombre);
  if (!item) return;
  try {
    let file = item.archivo;
    if (!file) {
      const r = await fetch(item.url);
      if (!r.ok) throw new Error("HTTP " + r.status);
      const blob = await r.blob();
      file = new File([blob], nombre, { type: blob.type || "image/jpeg" });
    }
    cambiarModo("visitante"); irAPaso(1);
    setObra(file);
    $("hintAnalizar").textContent = "Imagen precargada lista como obra. Activa la cámara y observa la obra.";
  } catch (e) { mostrarErrorDs({ tipo: "imagen_no_encontrada", mensaje: `No se pudo cargar «${nombre}» (${e.message}).` }); }
}

$("btnDsAleatorias").addEventListener("click", pedirAleatorias);
$("btnDsAnalizarTodas").addEventListener("click", analizarTodas);
$("btnDsPropia").addEventListener("click", () => $("fileDsPropia").click());
$("fileDsPropia").addEventListener("change", e => { if (e.target.files[0]) analizarPropia(e.target.files[0]); e.target.value = ""; });
$("btnDsRecargar").addEventListener("click", async () => {
  const btn = $("btnDsRecargar"); mostrarErrorDs(null); setCargando(btn, true, "Leyendo carpeta…");
  try { const e = await fetchJSON("/api/dataset/recargar", { method: "POST" }); pintarEstadoDataset(e); toast(`${e.n_imagenes} imágenes en la carpeta.`, "ok", 2200); }
  catch (e) { mostrarErrorDs(e.tipo ? e : { tipo: "servidor", mensaje: "No se pudo contactar al servidor." }); }
  finally { setCargando(btn, false); }
});

/* ============================================================ evaluación con visitantes */
const EVAL_KEY = "pdi_eval_participante";
function radioValor(nombre) { const x = document.querySelector(`input[name="${nombre}"]:checked`); return x ? x.value : ""; }
function prepararEvaluacion() {
  const r = state.resultado, card = $("cardEval");
  const listo = !!(r && r.modo === "visitante" && r.obra && r.obra.categoria);
  card.hidden = !listo;
  if (!listo) return;
  try { if (!$("evPart").value) $("evPart").value = localStorage.getItem(EVAL_KEY) || ""; } catch { /* almacenamiento no disponible */ }
  $("evObra").value = state.obra && state.obra.name ? state.obra.name : "";
  $("evEmo").value = ""; $("evCom").value = "";
  document.querySelectorAll('input[name="evVal"], input[name="evAcu"]').forEach(x => { x.checked = false; });
  $("evMsg").textContent = ""; $("btnEval").disabled = false;
}
function resultadoLiviano(r) {
  // Quita imágenes (data URLs) y la línea de tiempo: el servidor solo guarda valores numéricos.
  return JSON.parse(JSON.stringify(r, (k, v) => (typeof v === "string" && v.startsWith("data:")) || k === "linea_tiempo" ? undefined : v));
}
async function enviarEvaluacion() {
  const btn = $("btnEval"), msg = $("evMsg");
  const autorreporte = { participante: $("evPart").value.trim(), emocion: $("evEmo").value,
    valencia_obra: radioValor("evVal"), acuerdo_coincidencia: radioValor("evAcu"), comentario: $("evCom").value.trim() };
  if (!autorreporte.participante || !autorreporte.emocion || autorreporte.valencia_obra === "") {
    msg.textContent = "Completa el código, la emoción y lo que te transmitió la obra."; return;
  }
  if (state.resultado.coincidencia != null && !autorreporte.acuerdo_coincidencia) {
    msg.textContent = "Indica qué tan de acuerdo estás con la coincidencia."; return;
  }
  btn.disabled = true; msg.textContent = "Guardando…";
  try {
    const r = await fetch(API + "/api/evaluacion", { method: "POST", headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ autorreporte, obra_id: $("evObra").value.trim(), resultado: resultadoLiviano(state.resultado) }) });
    const d = await r.json();
    if (!r.ok || d.estado !== "ok") throw new Error((d.error && d.error.mensaje) || ("HTTP " + r.status));
    try { localStorage.setItem(EVAL_KEY, autorreporte.participante); } catch { /* sin almacenamiento */ }
    msg.textContent = `Respuesta guardada (${d.total} en total). ¡Gracias!`; toast("Evaluación guardada", "ok", 2500);
  } catch (e) { msg.textContent = "No se pudo guardar: " + e.message; btn.disabled = false; }
}
$("btnEval").addEventListener("click", enviarEvaluacion);
const _renderResultadoBase = renderResultado;
renderResultado = function () { _renderResultadoBase(); prepararEvaluacion(); };

/* ---------- inicio ---------- */
cargarEstado();
actualizarBoton();
irAPaso(1);
'''
print('app.js:', len(APP_JS), 'caracteres')

### Servidor (FastAPI)

In [ ]:
# API del aplicativo + la interfaz servida desde memoria (al exportar quedan en frontend/).
app = FastAPI(title="Coincidencia emocional · PDI UAO", version="1.3")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


@app.get("/", response_class=HTMLResponse)
def inicio():
    return HTMLResponse(INDEX_HTML)


@app.get("/styles.css")
def estilos():
    return Response(STYLES_CSS, media_type="text/css; charset=utf-8")


@app.get("/app.js")
def script():
    return Response(APP_JS, media_type="application/javascript; charset=utf-8")


@app.get("/api/estado")
def estado():
    info = sistema.estado()
    info["descargas"] = errores_descarga
    return info


@app.get("/api/metricas")
def metricas():
    if not config.RUTA_METRICAS.exists():
        raise HTTPException(404, "Aún no hay métricas: ejecuta la celda de entrenamiento.")
    return json.loads(config.RUTA_METRICAS.read_text(encoding="utf-8"))


@app.post("/api/frame")
def analizar_frame(frame: UploadFile = File(...)):
    """CapturaDeFrame -> AnalizarGestosFaciales para la vista en vivo."""
    try:
        return sistema.analizar_frame(frame.file.read())
    except ErrorProceso as e:
        return JSONResponse({"estado": "error", "excepciones": [e.a_dict()]}, status_code=200)


@app.post("/api/analizar")
def analizar(
    obra: Optional[UploadFile] = File(None),
    frames: Optional[List[UploadFile]] = File(None),
    rostro: Optional[UploadFile] = File(None),
    tiempos: Optional[str] = Form(None),
    modo: str = Form("visitante"),
):
    """obra (opcional) · frames (uno o varios) · rostro (foto única) · tiempos (JSON ms) · modo visitante|dataset"""
    obra_bytes = obra.file.read() if obra is not None else None
    lista = [f.file.read() for f in (frames or [])]
    if rostro is not None:
        lista.append(rostro.file.read())
    try:
        marcas = json.loads(tiempos) if tiempos else []
    except json.JSONDecodeError:
        marcas = []
    marcas = [float(m) for m in marcas] + [float(i * 500) for i in range(len(marcas), len(lista))]
    if obra_bytes is None and not lista:
        raise HTTPException(400, "Envía al menos la obra o un fotograma del rostro.")
    if modo == "dataset":
        if obra_bytes is None:
            raise HTTPException(400, "En modo dataset debes enviar la imagen en el campo «obra».")
        return sistema.analizar(obra_bytes, [], modo="dataset", que="la imagen subida")
    return sistema.analizar(obra_bytes, list(zip(lista, marcas)))


_HTTP_ERROR = {"dataset_no_encontrado": 404, "imagen_no_encontrada": 404, "dataset_no_disponible": 404}


def _respuesta_error(e: ErrorProceso) -> JSONResponse:
    return JSONResponse({"estado": "error", "error": e.a_dict(), "dataset": sistema.dataset.estado()},
                        status_code=_HTTP_ERROR.get(e.tipo, 400))


@app.get("/api/dataset/estado")
def dataset_estado():
    return sistema.dataset.estado()


@app.post("/api/dataset/recargar")
def dataset_recargar():
    """Vuelve a leer la carpeta precarga-sin-analizar (por si se agregaron imágenes)."""
    try:
        sistema.dataset.asegurar(forzar=True)
        return sistema.dataset.estado()
    except ErrorProceso as e:
        return _respuesta_error(e)


@app.get("/api/dataset/aleatorias")
def dataset_aleatorias(n: int = 6, excluir: str = ""):
    try:
        res = sistema.dataset.aleatorias(n, [x for x in excluir.split("|") if x])
        res["estado"] = "ok"
        return res
    except ErrorProceso as e:
        return _respuesta_error(e)


@app.get("/api/dataset/imagen")
def dataset_imagen(nombre: str):
    try:
        ruta = sistema.dataset.ruta(nombre)
    except ErrorProceso as e:
        return _respuesta_error(e)
    tipo = "image/png" if ruta.suffix.lower() == ".png" else "image/jpeg"
    return FileResponse(ruta, media_type=tipo, headers={"Cache-Control": "public, max-age=86400"})


@app.post("/api/dataset/analizar")
def dataset_analizar(nombre: str = Form(...)):
    try:
        return sistema.analizar_dataset(nombre)
    except ErrorProceso as e:
        return _respuesta_error(e)


@app.post("/api/evaluacion")
def guardar_evaluacion(payload: dict = Body(...)):
    """Formulario de la GUI: autorreporte del visitante + resultado del análisis (sin imágenes)."""
    resultado = payload.get("resultado") or {}
    if resultado.get("modo") != "visitante" or not (resultado.get("obra") or {}).get("categoria"):
        return JSONResponse({"estado": "error", "error": {"tipo": "entrada_incompleta", "gravedad": "error",
                             "mensaje": "Solo se registran análisis del modo «Obra y visitante» con la obra clasificada.",
                             "modulo": "RegistroEvaluacion"}}, status_code=400)
    try:
        reg = guardar_registro(payload.get("autorreporte") or {}, resultado, payload.get("obra_id"), "gui")
    except ErrorProceso as e:
        return JSONResponse({"estado": "error", "error": e.a_dict()}, status_code=400)
    return {"estado": "ok", "id": reg["id"], "total": len(leer_registros())}


@app.get("/api/evaluacion/resumen")
def resumen_evaluacion():
    regs = leer_registros()
    return {"total": len(regs), "participantes": len({r["autorreporte"]["participante"] for r in regs}),
            "archivo": str(ruta_registros())}

print("API lista:", len([r for r in app.routes if getattr(r, "path", "").startswith("/api")]), "endpoints")

In [ ]:
# Levanta FastAPI en segundo plano (no bloquea el notebook). Si el puerto está ocupado prueba el siguiente.
def _puerto_libre(puerto: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", puerto)) != 0


def detener_servidor():
    srv = globals().get("servidor_uvicorn")
    if srv is not None and srv.started:
        srv.should_exit = True
        for _ in range(50):
            if not srv.started:
                break
            time.sleep(0.1)
        print("Servidor detenido.")


def levantar_servidor(puerto_inicial: int = 8000, intentos: int = 10):
    global servidor_uvicorn
    detener_servidor()                                   # si la celda se vuelve a ejecutar, no deja dos servidores
    puerto = next((p for p in range(puerto_inicial, puerto_inicial + intentos) if _puerto_libre(p)), None)
    if puerto is None:
        raise RuntimeError(f"Los puertos {puerto_inicial}-{puerto_inicial + intentos - 1} están ocupados.")
    config.PUERTO = puerto
    servidor_uvicorn = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=puerto, log_level="warning"))
    hilo = threading.Thread(target=servidor_uvicorn.run, name="HiloServidorFastAPI", daemon=True)
    hilo.start()
    for _ in range(100):                                 # espera hasta 10 s a que arranque
        if servidor_uvicorn.started:
            break
        time.sleep(0.1)
    print(("✅ Servidor en" if servidor_uvicorn.started else "⚠ El servidor no confirmó el arranque en"),
          f"http://127.0.0.1:{puerto} · para detenerlo: detener_servidor()")
    return hilo


servidor_uvicorn = globals().get("servidor_uvicorn")
hilo_servidor = levantar_servidor(config.PUERTO)

### Diagnóstico del servidor

In [ ]:
import urllib.request
import json
import threading

print("=== DIAGNÓSTICO DEL SERVIDOR ===\n")

# 1. Verificar hilos activos del servidor
hilos_activos = [t.name for t in threading.enumerate()]
print(f"[Hilos] Hilos activos en el sistema: {hilos_activos}")
if any("servidor" in name.lower() for name in hilos_activos) or any("uvicorn" in name.lower() for name in hilos_activos):
    print("✅ Se detectaron hilos activos relacionados con el servidor.")
else:
    print("⚠️ No se detecta un hilo explícito de servidor activo. Puede que no esté corriendo o use otro nombre.")

# 2. Intentar petición de loopback local a la API utilizando el puerto configurado
puerto_servidor = config.PUERTO
url_test = f"http://127.0.0.1:{puerto_servidor}/api/estado"

try:
    print(f"\n[Loopback] Intentando conectar localmente a: {url_test} ...")
    with urllib.request.urlopen(url_test, timeout=3) as response:
        codigo = response.getcode()
        datos = json.loads(response.read().decode('utf-8'))
        print(f"✅ ¡Conexión exitosa! Código HTTP: {codigo}")
        print("Respuesta de la API:")
        print(json.dumps(datos, indent=2, ensure_ascii=False))
except Exception as e:
    print(f"❌ Error al intentar conectar localmente al servidor: {e}")
    print("Esto ocurre si el servidor de FastAPI aún no se ha iniciado con uvicorn en segundo plano.")

# 3. Mostrar variables clave de red
print(f"\n[Configuración] Puerto configurado: {puerto_servidor}")
print(f"[Configuración] URL local sugerida: http://127.0.0.1:{puerto_servidor}")

### Abrir el aplicativo
- **En Colab (recomendado):** la celda siguiente crea un enlace del proxy de Colab, que solo funciona en tu navegador con tu sesión de Google. No expone ninguna IP.
- **Desde otro dispositivo** (celular, tableta de la exposición): usa la celda de Localtunnel. Pide como contraseña la IP pública de este entorno. Ponla en pantalla solo con `config.MOSTRAR_IP_TUNEL = True` y **borra esa salida antes de subir el notebook a GitHub**.
- **En tu computador** (Jupyter local o el notebook exportado a `.py`): abre `http://127.0.0.1:8000`.

In [ ]:
# Enlace del proxy de Colab (sin IP ni contraseña); en Jupyter local se abre el navegador directamente
URL_LOCAL = f"http://127.0.0.1:{config.PUERTO}"
try:
    from google.colab.output import eval_js
    URL_APP = eval_js(f"google.colab.kernel.proxyPort({config.PUERTO})")
    print("Abre el aplicativo aquí (misma cuenta de Google):", URL_APP)
except ImportError:
    URL_APP = URL_LOCAL
    print("Aplicativo en", URL_APP)
    if config.ABRIR_NAVEGADOR:
        webbrowser.open(URL_APP)

In [ ]:
# Enlace público con Localtunnel (opcional, para abrir la GUI desde otro dispositivo). No bloquea el notebook.
import shutil as _sh
import subprocess as _sp

USAR_TUNEL = False   # pon True para crear el enlace público


def iniciar_tunel(puerto: int = config.PUERTO, espera_s: int = 25):
    if not _sh.which("lt"):
        _sp.run(["npm", "install", "-g", "localtunnel", "--silent"], check=True)
    proceso = _sp.Popen(["lt", "--port", str(puerto)], stdout=_sp.PIPE, stderr=_sp.STDOUT, text=True)
    url = {}

    def leer_salida():                                 # hilo aparte: la lectura no puede bloquear el notebook
        for linea in proceso.stdout:
            if "your url is" in linea.lower():
                url["valor"] = linea.split(":", 1)[1].strip()

    threading.Thread(target=leer_salida, name="HiloTunel", daemon=True).start()
    t0 = time.time()
    while "valor" not in url and time.time() - t0 < espera_s and proceso.poll() is None:
        time.sleep(0.5)
    print("Enlace público:", url["valor"]) if "valor" in url else print("Localtunnel no respondió a tiempo.")
    if config.MOSTRAR_IP_TUNEL:
        ip = urllib.request.urlopen("https://ipv4.icanhazip.com", timeout=10).read().decode().strip()
        print("Contraseña del túnel (IP pública de este entorno):", ip, "→ borra esta salida antes de subir el notebook.")
    else:
        print("La página de Localtunnel pide una contraseña: es la IP pública de este entorno. "
              "Ponla en pantalla con config.MOSTRAR_IP_TUNEL = True.")
    return proceso


if USAR_TUNEL:
    proceso_tunel = iniciar_tunel()
else:
    print("Túnel desactivado (USAR_TUNEL = False). En Colab basta con el enlace de la celda anterior.")

## 6. Evaluación con visitantes reales
La rúbrica pide evaluar **el sistema completo** con datos reales capturados por el equipo. Las métricas de la sección 4 solo cubren la CNN. Aquí se evalúan las otras dos piezas, el **rostro del visitante** y la **coincidencia emocional**, y además se contrasta la CNN con la **percepción de personas reales** sobre obras que no vio en el entrenamiento.

**Protocolo** (detalle completo en `docs/protocolo_evaluacion.md` del repositorio):
1. **Participantes:** ≥ 10 personas (recomendado 15), cada una con consentimiento informado firmado. Código anónimo `P01`, `P02`, …
2. **Obras:** 6 obras por participante elegidas de `precarga-sin-analizar` (no vistas por la CNN): 3 que la CNN clasifica como positivas y 3 como negativas, en orden aleatorio.
3. **Captura (opción A, en vivo):** en la GUI, pestaña *Obra y visitante*: cargar la obra → activar cámara → *Iniciar observación* de 6 s → *Confirmar y analizar*. Al final del resultado, el participante llena el formulario **Evaluación del sistema**: emoción sentida, qué le transmitió la obra (−2 a +2) y acuerdo con la coincidencia (1-5).
4. **Captura (opción B, videos):** grabar 6-10 s del rostro de frente mientras mira la obra en pantalla y anotar sus respuestas en `protocolo_offline.csv`. La celda siguiente procesa esos videos con el mismo flujo del aplicativo.
5. **Usabilidad (opcional):** al terminar, cada participante responde el cuestionario SUS (10 ítems, escala 1-5) en `sus.csv`.

Todo queda en `Drive/Proyecto-final-de-PDI/evaluacion/`. Las celdas se pueden volver a ejecutar cuantas veces haga falta a medida que se recogen datos.

| Archivo | Columnas |
|---|---|
| `registros_visitantes.jsonl` | lo escribe la GUI y la celda de videos (no editar a mano) |
| `protocolo_offline.csv` | `participante, obra, video, emocion_reportada, valencia_obra_reportada, acuerdo_coincidencia, comentario` |
| `sus.csv` | `participante, q1 … q10` |

`emocion_reportada` ∈ {felicidad, sorpresa, neutral, tristeza, miedo, enojo, asco, desprecio} · `valencia_obra_reportada` ∈ {−2…2} · `obra` = nombre del archivo en `precarga-sin-analizar` o ruta dentro de `evaluacion/obras/` · `video` = ruta dentro de `evaluacion/videos/` (archivo de video o carpeta de fotos).

In [ ]:
# Crea las plantillas de la evaluación si aún no existen (no sobrescribe datos ya recogidos)
DIR_EVAL = dir_evaluacion()
for sub in ("videos", "obras"):
    (DIR_EVAL / sub).mkdir(exist_ok=True)
plantillas = {
    "protocolo_offline.csv": "participante,obra,video,emocion_reportada,valencia_obra_reportada,acuerdo_coincidencia,comentario\n",
    "sus.csv": "participante,q1,q2,q3,q4,q5,q6,q7,q8,q9,q10\n",
}
for nombre, encabezado in plantillas.items():
    ruta = DIR_EVAL / nombre
    if not ruta.exists():
        ruta.write_text(encabezado, encoding="utf-8")
        print("Plantilla creada:", ruta)
print("Carpeta de evaluación:", DIR_EVAL)
for p in sorted(DIR_EVAL.iterdir()):
    print("  ", p.name, "(carpeta)" if p.is_dir() else f"({p.stat().st_size} bytes)")

### Opción B: procesar videos registrados por el equipo
Lee `protocolo_offline.csv` y analiza cada video con el mismo flujo del aplicativo (fotogramas cada 400 ms, máximo 60). Las filas ya procesadas se omiten, así que se puede ejecutar varias veces.

In [ ]:
# Opción B: procesa los videos (o carpetas de fotos) registrados por el equipo con el mismo flujo del aplicativo
EXT_VIDEO = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v"}


def fotogramas_de(ruta: Path, intervalo_ms: int = config.EVAL_INTERVALO_MS, maximo: int = config.MAX_FRAMES):
    """Devuelve [(jpeg_bytes, t_ms)] muestreando un video cada intervalo_ms, o las fotos de una carpeta en orden."""
    if ruta.is_dir():
        fotos = sorted(p for p in ruta.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})[:maximo]
        return [(p.read_bytes(), float(i * intervalo_ms)) for i, p in enumerate(fotos)]
    cap = cv2.VideoCapture(str(ruta))
    if not cap.isOpened():
        raise ErrorProceso("archivo_corrupto", f"No se pudo abrir el video {ruta.name}.", "error", "CapturaDeFrame")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    paso = max(1, int(round(fps * intervalo_ms / 1000)))
    salida, k = [], 0
    while len(salida) < maximo:
        ok, bgr = cap.read()
        if not ok:
            break
        if k % paso == 0:
            h, w = bgr.shape[:2]
            if w > 960:                                    # como la GUI: fotogramas de tamaño moderado
                bgr = cv2.resize(bgr, (960, int(h * 960 / w)), interpolation=cv2.INTER_AREA)
            salida.append((cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 90])[1].tobytes(), k * 1000 / fps))
        k += 1
    cap.release()
    return salida


def leer_obra_eval(nombre: str) -> bytes:
    local = DIR_EVAL / "obras" / nombre
    if local.is_file():
        return local.read_bytes()
    return sistema.dataset.leer(Path(nombre).name)          # imagen de precarga-sin-analizar


def procesar_protocolo_offline() -> int:
    ruta_csv = DIR_EVAL / "protocolo_offline.csv"
    if not ruta_csv.exists():
        print("No existe", ruta_csv); return 0
    filas = pd.read_csv(ruta_csv, dtype=str).fillna("")
    if filas.empty:
        print("protocolo_offline.csv no tiene filas todavía (opción B sin usar)."); return 0
    hechos = {(r["autorreporte"]["participante"], r["obra_id"], r.get("video")) for r in leer_registros() if r["fuente"] == "offline"}
    nuevos = 0
    for k, f in filas.iterrows():
        clave = (f["participante"].strip().upper(), f["obra"].strip(), f["video"].strip())
        if clave in hechos:
            continue
        try:
            frames = fotogramas_de(DIR_EVAL / "videos" / f["video"].strip())
            if not frames:
                raise ErrorProceso("archivo_ausente", "El video no tiene fotogramas legibles.", "error", "CapturaDeFrame")
            resultado = sistema.analizar(leer_obra_eval(f["obra"].strip()), frames)
            guardar_registro({"participante": f["participante"], "emocion": f["emocion_reportada"].strip().lower(),
                                    "valencia_obra": f["valencia_obra_reportada"], "acuerdo_coincidencia": f["acuerdo_coincidencia"],
                                    "comentario": f["comentario"]}, resultado, f["obra"].strip(), "offline",
                                   extra={"video": f["video"].strip()})        # el video evita procesarlo dos veces
            nuevos += 1
            print(f"  fila {k + 2}: {clave[0]} · {clave[1]} → coincidencia {resultado['coincidencia']}")
        except ErrorProceso as e:
            print(f"  ⚠ fila {k + 2} ({clave[0]} · {clave[1]}): {e.mensaje}")
    print(f"Videos procesados en esta ejecución: {nuevos}")
    return nuevos


procesar_protocolo_offline()

### Métricas del sistema completo
1. **Detección**: porcentaje de pruebas con rostro válido.
2. **Expresión (FER+)**: acuerdo con la emoción que reporta el visitante, en 8 clases y en valencia (negativa / neutra / positiva), con κ de Cohen.
3. **CNN frente a personas**: accuracy y κ de la categoría de la CNN frente a lo que la obra transmitió al visitante, en obras **no vistas**.
4. **Coincidencia**: error (MAE) frente a la coincidencia calculada con lo reportado y correlación de Spearman con el acuerdo del visitante (1-5), para las fórmulas v2 y v3.
5. **Latencia real** y **SUS** (usabilidad), si hay datos.

Los intervalos de confianza del 95 % (Wilson) indican cuánto pesa el tamaño de la muestra.

In [ ]:
# Métricas del sistema completo con los datos de los visitantes
def wilson(aciertos: int, n: int, z: float = 1.96):
    """Intervalo de confianza del 95 % para una proporción (adecuado con muestras pequeñas)."""
    if n == 0:
        return (None, None)
    p = aciertos / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    m = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (round(float(c - m), 3), round(float(c + m), 3))


def signo_valencia(v, tol: float = 0.15) -> str:
    return "positiva" if v > tol else "negativa" if v < -tol else "neutra"


def spearman(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = ~(np.isnan(a) | np.isnan(b))
    if ok.sum() < 4 or np.std(a[ok]) == 0 or np.std(b[ok]) == 0:
        return {"rho": None, "p": None, "n": int(ok.sum())}
    r = spearmanr(a[ok], b[ok])
    return {"rho": round(float(r.statistic), 3), "p": round(float(r.pvalue), 4), "n": int(ok.sum())}


def tabla_registros(registros: list[dict]) -> pd.DataFrame:
    filas = []
    for r in registros:
        a, s = r["autorreporte"], r["sistema"]
        o, ro = s.get("obra") or {}, s.get("rostro") or {}
        v_obra_rep = a["valencia_obra"] / 2                              # −2…2 → −1…1
        v_vis_rep = VALENCIA[a["emocion"]]
        filas.append({
            "id": r["id"], "fuente": r["fuente"], "participante": a["participante"], "obra": r["obra_id"],
            "emocion_rep": a["emocion"], "valencia_obra_rep": a["valencia_obra"], "acuerdo": a["acuerdo_coincidencia"],
            "cnn_categoria": o.get("categoria"), "cnn_valencia": o.get("valencia"),
            "rostro_detectado": bool(ro), "emocion_sis": ro.get("emocion"),
            "valencia_vis_v2": ro.get("valencia_v2"), "valencia_vis_v3": ro.get("valencia_v3"), "intensidad": ro.get("intensidad"),
            "frames_analizados": s.get("frames_analizados"), "frames_con_rostro": s.get("frames_con_rostro"),
            "coinc_v2": s.get("coincidencia_v2"), "coinc_v3": s.get("coincidencia_v3"), "concluyente": s.get("concluyente"),
            # coincidencia "real": la misma fórmula aplicada a lo que la persona dijo
            "coinc_referencia": round(1 - abs(v_obra_rep - v_vis_rep) / 2, 4),
            "t_total_ms": (s.get("tiempos_ms") or {}).get("total"),
        })
    df = pd.DataFrame(filas)
    numericas = ["valencia_obra_rep", "acuerdo", "cnn_valencia", "valencia_vis_v2", "valencia_vis_v3", "intensidad",
                 "frames_analizados", "frames_con_rostro", "coinc_v2", "coinc_v3", "coinc_referencia", "t_total_ms"]
    df[numericas] = df[numericas].apply(pd.to_numeric, errors="coerce")
    return df


def metricas_sus(ruta: Path):
    if not ruta.exists():
        return None
    df = pd.read_csv(ruta).dropna()
    if df.empty:
        return None
    q = df[[f"q{i}" for i in range(1, 11)]].astype(float)
    impares, pares = q[[f"q{i}" for i in (1, 3, 5, 7, 9)]], q[[f"q{i}" for i in (2, 4, 6, 8, 10)]]
    puntaje = ((impares - 1).sum(axis=1) + (5 - pares).sum(axis=1)) * 2.5
    return {"n": int(len(puntaje)), "media": round(float(puntaje.mean()), 1), "std": round(float(puntaje.std(ddof=0)), 1),
            "interpretacion": "por encima del promedio (≥ 68)" if puntaje.mean() >= 68 else "por debajo del promedio (< 68)"}


def evaluar_visitantes(df: pd.DataFrame) -> dict:
    out = {"n_registros": int(len(df)), "n_participantes": int(df["participante"].nunique()),
           "n_obras": int(df["obra"].nunique()), "por_fuente": df["fuente"].value_counts().to_dict()}

    # 1) Detección de rostro
    det = df["rostro_detectado"]
    frames_ok = (df["frames_con_rostro"] / df["frames_analizados"]).replace([np.inf], np.nan)
    out["deteccion"] = {"tasa_sesiones_con_rostro": round(float(det.mean()), 3), "ic95": wilson(int(det.sum()), len(df)),
                        "fraccion_media_frames_con_rostro": round(float(frames_ok.mean()), 3) if frames_ok.notna().any() else None}

    # 2) Expresión facial (FER+) frente al autorreporte
    d = df[det]
    if len(d):
        acc8 = int((d["emocion_sis"] == d["emocion_rep"]).sum())
        sis3 = d["emocion_sis"].map(lambda e: signo_valencia(VALENCIA[e]))
        rep3 = d["emocion_rep"].map(lambda e: signo_valencia(VALENCIA[e]))
        sis3_v3 = d["valencia_vis_v3"].map(signo_valencia)
        out["expresion"] = {
            "n": int(len(d)),
            "accuracy_8_emociones": round(acc8 / len(d), 3), "ic95_8": wilson(acc8, len(d)),
            "accuracy_valencia_3_clases": round(float((sis3 == rep3).mean()), 3), "ic95_3": wilson(int((sis3 == rep3).sum()), len(d)),
            "accuracy_valencia_3_clases_v3": round(float((sis3_v3 == rep3).mean()), 3),
            "kappa_valencia": round(float(cohen_kappa_score(rep3, sis3)), 3) if rep3.nunique() > 1 else None,
            "proporcion_expresion_neutral_predicha": round(float((d["emocion_sis"] == "neutral").mean()), 3),
            "proporcion_concluyente": round(float(d["concluyente"].astype(bool).mean()), 3),
        }

    # 3) CNN frente a la percepción humana (obras no vistas; se excluyen las respuestas "Neutra")
    c = df[(df["valencia_obra_rep"] != 0) & df["cnn_categoria"].notna()]
    if len(c):
        y_h = (c["valencia_obra_rep"] > 0).astype(int)
        y_s = (c["cnn_categoria"] == "positivo").astype(int)
        acc = int((y_h == y_s).sum())
        por_obra = c.groupby("obra").agg(humano=("valencia_obra_rep", "mean"), cnn=("cnn_valencia", "first"))
        out["cnn_vs_humano"] = {
            "n_respuestas": int(len(c)), "accuracy": round(acc / len(c), 3), "ic95": wilson(acc, len(c)),
            "f1_positivo": round(float(f1_score(y_h, y_s, zero_division=0)), 3),
            "kappa": round(float(cohen_kappa_score(y_h, y_s)), 3) if y_h.nunique() > 1 else None,
            "spearman_valencia_por_obra": spearman(por_obra["cnn"], por_obra["humano"]),
            "acuerdo_entre_personas_por_obra": round(float(c.groupby("obra")["valencia_obra_rep"].apply(
                lambda s: max((s > 0).mean(), (s < 0).mean())).mean()), 3),
        }

    # 4) Coincidencia emocional: ¿se parece a lo que la persona vivió y a lo que opina del resultado?
    k = df[df["coinc_v2"].notna()]
    if len(k):
        out["coincidencia"] = {"n": int(len(k))}
        for f in ("v2", "v3"):
            col = f"coinc_{f}"
            out["coincidencia"][f] = {
                "media": round(float(k[col].mean()), 3),
                "mae_vs_referencia": round(float((k[col] - k["coinc_referencia"]).abs().mean()), 3),
                "spearman_vs_referencia": spearman(k[col], k["coinc_referencia"]),
                "spearman_vs_acuerdo_usuario": spearman(k[col], k["acuerdo"]),
            }
        out["coincidencia"]["acuerdo_medio_1a5"] = round(float(k["acuerdo"].mean()), 2) if k["acuerdo"].notna().any() else None
        mejor = min(("v2", "v3"), key=lambda f: out["coincidencia"][f]["mae_vs_referencia"])
        out["coincidencia"]["formula_con_menor_error"] = mejor

    # 5) Latencia observada en las pruebas reales (servidor)
    t = df["t_total_ms"].dropna()
    if len(t):
        out["latencia_real_ms"] = {"media": round(float(t.mean()), 1), "std": round(float(t.std(ddof=0)), 1),
                                   "p95": round(float(t.quantile(0.95)), 1)}
    out["sus"] = metricas_sus(DIR_EVAL / "sus.csv")
    return out


registros_eval = leer_registros()
evaluacion_visitantes = None
if not registros_eval:
    print("Aún no hay registros de visitantes.\n"
          "→ Usa el formulario «Evaluación del sistema» de la GUI (sección 5) o llena protocolo_offline.csv y "
          "vuelve a ejecutar esta celda y la anterior.")
else:
    df_eval = tabla_registros(registros_eval)
    df_eval.to_csv(config.DIR_RESULTADOS / "evaluacion_visitantes.csv", index=False, encoding="utf-8")
    evaluacion_visitantes = evaluar_visitantes(df_eval)
    with open(config.DIR_RESULTADOS / "evaluacion_visitantes.json", "w", encoding="utf-8") as f:
        json.dump(evaluacion_visitantes, f, ensure_ascii=False, indent=2, default=str)
    print(json.dumps(evaluacion_visitantes, ensure_ascii=False, indent=1, default=str))
    if evaluacion_visitantes["n_participantes"] < 10:
        print(f"\n⚠ Solo hay {evaluacion_visitantes['n_participantes']} participante(s): los intervalos de confianza "
              "serán amplios. El protocolo recomienda ≥ 10.")

    # Figuras de evidencia
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
    d = df_eval[df_eval["rostro_detectado"]]
    orden3 = ["negativa", "neutra", "positiva"]
    if len(d):
        ConfusionMatrixDisplay(confusion_matrix(d["emocion_rep"].map(lambda e: signo_valencia(VALENCIA[e])),
                                                d["emocion_sis"].map(lambda e: signo_valencia(VALENCIA[e])), labels=orden3),
                               display_labels=orden3).plot(cmap="Greens", ax=ax[0], colorbar=False)
        ax[0].set_xlabel("Expresión detectada (FER+)"); ax[0].set_ylabel("Emoción reportada")
    ax[0].set_title("Rostro: valencia detectada vs. reportada")
    c = df_eval[(df_eval["valencia_obra_rep"] != 0) & df_eval["cnn_categoria"].notna()]
    if len(c):
        ConfusionMatrixDisplay(confusion_matrix((c["valencia_obra_rep"] > 0).astype(int),
                                                (c["cnn_categoria"] == "positivo").astype(int), labels=[0, 1]),
                               display_labels=CLASES).plot(cmap="Blues", ax=ax[1], colorbar=False)
        ax[1].set_xlabel("CNN"); ax[1].set_ylabel("Percepción del visitante")
    ax[1].set_title("CNN vs. percepción humana (obras no vistas)")
    k = df_eval[df_eval["coinc_v2"].notna()]
    if len(k):
        ax[2].scatter(k["coinc_referencia"], k["coinc_v2"], alpha=0.6, label="v2 (original)", color="#7D839F")
        ax[2].scatter(k["coinc_referencia"], k["coinc_v3"], alpha=0.8, label="v3 (expresiva)", color="#232B5D", marker="^")
        ax[2].plot([0, 1], [0, 1], ls=":", color="#c33")
        ax[2].set_xlim(-0.02, 1.02); ax[2].set_ylim(-0.02, 1.02); ax[2].legend()
        ax[2].set_xlabel("Coincidencia según lo reportado"); ax[2].set_ylabel("Coincidencia calculada")
    ax[2].set_title("Coincidencia calculada vs. reportada")
    fig.suptitle("Evaluación con visitantes reales"); plt.tight_layout(); plt.show()

## 7. Análisis de resultados y conclusiones
> Las cifras entre paréntesis son de la **ejecución de referencia** (Colab, GPU T4, semilla 42). Las de tu última ejecución quedan en `results/RESULTADOS.md`, que genera la celda siguiente. Completa los apartados marcados con **[completar]** cuando tengas los datos de la sección 6.

### Logros
- **Clasificación de obras.** La ResNet18 con *transfer learning* alcanza en validación cruzada de 5 folds una accuracy de 0.764 ± 0.037, F1 de 0.771 y AUC de 0.842 sobre 500 imágenes. Cada imagen se evaluó con un modelo que no la había visto. Supera con claridad a la CNN del Parcial 2 (0.642) y a la línea base de solo color: el contenido de la obra aporta información que la paleta sola no tiene.
- **Confianza útil.** El 91 % de las predicciones superan el umbral del 60 % y en ellas la accuracy sube a 0.789. El aviso de "revisión requerida" filtra los casos dudosos.
- **Explicabilidad.** Grad-CAM y la paleta en CIE Lab muestran qué zonas y colores sostienen la decisión, lo que permite discutir los errores en vez de tratarlos como una caja negra.
- **Sistema integrado.** El flujo completo (carga → CNN + paleta + rostro → reglas → GUI) responde en el orden de cientos de milisegundos por obra (ver la tabla de latencia) y maneja las excepciones con mensajes comprensibles.
- **Evaluación con personas. [completar]** Resume aquí: tasa de detección del rostro, acuerdo entre FER+ y la emoción reportada, accuracy de la CNN frente a la percepción de los visitantes y la fórmula de coincidencia con menor error.

### Limitaciones
- **Sobreajuste.** Desde la época 6 la accuracy de entrenamiento pasa de 0.95 mientras la de validación se estanca en ~0.76 y su pérdida sube. Elegir la época por pérdida de validación lo mitiga, pero el modelo sigue memorizando un dataset pequeño (500 imágenes).
- **Ruido en las etiquetas.** En la galería de errores más seguros hay obras cuya carpeta parece discutible al revisarlas a ojo (p. ej. retratos y escenas apacibles en `negativo/`). La emoción que provoca una obra es subjetiva, y una sola etiqueta por imagen no captura el desacuerdo entre personas.
- **Expresión ≠ emoción.** FER+ reconoce gestos faciales, no la emoción interna. Ante el arte, la mayoría de las personas mantiene una expresión neutral. Con la fórmula original (v2) eso fijaba la coincidencia en ~50 % frente a cualquier obra clasificada con seguridad. La v3 usa solo la parte expresiva del rostro y marca el resultado como **poco concluyente** cuando la intensidad es baja.
- **Dominio de FER+.** FER+ se entrenó con fotografías de rostros reales. Su lectura sobre rostros *pintados* dentro de las obras es solo informativa y no se validó.
- **Muestra de la evaluación.** Con 10-15 participantes los intervalos de confianza son amplios. Los resultados de la sección 6 son indicativos, no concluyentes.
- **Despliegue.** El servidor de Colab es temporal y el túnel público no está pensado para producción (CORS abierto, sin autenticación).

### Oportunidades de mejora
1. Ampliar el dataset y usar etiquetas **graduadas** (varias personas por obra, p. ej. ArtEmis) en lugar de una carpeta binaria.
2. Regularizar más: aumento de datos más fuerte, *label smoothing*, congelar más capas o probar modelos más robustos (EfficientNet, ViT o CLIP como extractor).
3. Combinar la CNN con los rasgos de color en un modelo conjunto, ya que la línea base muestra que el color tiene señal propia.
4. Medir la reacción con más señales que la expresión facial (mirada, tiempo de observación, micro-expresiones) y calibrar la fórmula de coincidencia con los datos de la sección 6.
5. Desplegar en un servidor permanente con HTTPS y almacenamiento seguro para pruebas en un museo real.

### Conclusiones
El sistema cumple el objetivo de estimar, en una misma escala de valencia, el carácter emocional de una obra y la expresión del visitante, y de presentarlo en una interfaz interactiva. La CNN generaliza moderadamente (≈ 76 %) en un problema subjetivo y con pocas imágenes. El módulo facial funciona en tiempo real. **[completar con la sección 6]** La evaluación con visitantes indica que *…* y que la fórmula *v2/v3* es la que mejor reproduce lo que las personas reportan. La principal limitación es la distancia entre la expresión facial y la emoción sentida, que la herramienta reconoce explícitamente al marcar los resultados poco concluyentes.

### Resumen automático de resultados

In [ ]:
# Genera results/RESULTADOS.md con las cifras de ESTA ejecución (para pegar en el README y el póster)
def fila_modelo(nombre, r):
    g = r["global"]
    return f"| {nombre} | {g['accuracy']:.3f} ± {r['accuracy_std']:.3f} | {g['precision']:.3f} | {g['recall']:.3f} | {g['f1']:.3f} | {g['auc']:.3f} |"


lineas = [f"# Resultados · ejecución del {time.strftime('%Y-%m-%d %H:%M')}", "",
          f"Dataset: {N} imágenes ({(etiquetas == 1).sum()} positivas / {(etiquetas == 0).sum()} negativas) · "
          f"validación cruzada estratificada de {config.K_FOLDS} folds · mejor época {resultados_cv[NOMBRE_PRINCIPAL]['mejor_epoca']}.", "",
          "## Clasificación de obras (validación cruzada)", "",
          "| Modelo | Accuracy ± folds | Precision | Recall | F1 | AUC |", "|---|---|---|---|---|---|",
          *[fila_modelo(n, r) for n, r in resumenes.items()], "",
          "### Por fold (modelo principal)", "", "| Fold | n | Accuracy | F1 | AUC |", "|---|---|---|---|---|",
          *[f"| {m['fold']} | {m['n']} | {m['accuracy']:.3f} | {m['f1']:.3f} | {m['auc']:.3f} |" for m in resumenes[NOMBRE_PRINCIPAL]["por_fold"]],
          "", f"Matriz de confusión (filas = real, columnas = predicho; negativo, positivo): "
              f"`{resumenes[NOMBRE_PRINCIPAL]['global']['matriz_confusion']}`", "",
          "## Rendimiento", "", f"Dispositivo: {rendimiento['dispositivo']} · {rendimiento['n_imagenes']} imágenes de {rendimiento['origen_imagenes']}", "",
          "| Módulo | Media (ms) | Desv. (ms) | p95 (ms) |", "|---|---|---|---|",
          *[f"| {k} | {v['media_ms']} | {v['std_ms']} | {v['p95_ms']} |" for k, v in rendimiento["latencia_ms"].items()],
          "", f"Análisis facial: {rendimiento['frame_ms']['media']} ms por fotograma → **{rendimiento['fps_analisis_facial']} fps**.", ""]

ev = evaluacion_visitantes
lineas += ["## Evaluación con visitantes reales", ""]
if not ev:
    lineas += ["_Aún no hay registros. Ejecuta la sección 6 después de las pruebas._", ""]
else:
    lineas += [f"{ev['n_registros']} pruebas · {ev['n_participantes']} participantes · {ev['n_obras']} obras · fuentes {ev['por_fuente']}", "",
               "| Métrica | Valor | IC 95 % |", "|---|---|---|",
               f"| Sesiones con rostro detectado | {ev['deteccion']['tasa_sesiones_con_rostro']:.1%} | {ev['deteccion']['ic95']} |"]
    if "expresion" in ev:
        e = ev["expresion"]
        lineas += [f"| FER+ vs. emoción reportada (8 clases) | {e['accuracy_8_emociones']:.1%} | {e['ic95_8']} |",
                   f"| FER+ vs. reportada (valencia: neg/neu/pos) | {e['accuracy_valencia_3_clases']:.1%} | {e['ic95_3']} |",
                   f"| Expresión predicha neutral | {e['proporcion_expresion_neutral_predicha']:.1%} | |",
                   f"| Coincidencias concluyentes (v3) | {e['proporcion_concluyente']:.1%} | |"]
    if "cnn_vs_humano" in ev:
        c = ev["cnn_vs_humano"]
        lineas += [f"| CNN vs. percepción del visitante | {c['accuracy']:.1%} (κ = {c['kappa']}) | {c['ic95']} |",
                   f"| Spearman valencia CNN vs. humanos por obra | ρ = {c['spearman_valencia_por_obra']['rho']} (p = {c['spearman_valencia_por_obra']['p']}) | |"]
    if "coincidencia" in ev:
        k = ev["coincidencia"]
        for f in ("v2", "v3"):
            lineas.append(f"| Coincidencia {f}: MAE vs. reportada / ρ vs. acuerdo | {k[f]['mae_vs_referencia']} / "
                          f"{k[f]['spearman_vs_acuerdo_usuario']['rho']} | |")
        lineas.append(f"| Acuerdo medio del visitante con la coincidencia (1-5) | {k['acuerdo_medio_1a5']} | |")
    if ev.get("sus"):
        lineas.append(f"| SUS (usabilidad, 0-100) | {ev['sus']['media']} ± {ev['sus']['std']} ({ev['sus']['interpretacion']}) | n = {ev['sus']['n']} |")
    lineas.append("")
lineas += ["## Figuras", "", *[f"- `figuras/{p.name}`" for p in sorted(config.DIR_FIGURAS.glob("*.png"))], ""]

(config.DIR_RESULTADOS / "RESULTADOS.md").write_text("\n".join(lineas), encoding="utf-8")
with open(config.DIR_RESULTADOS / "metricas_resumen.json", "w", encoding="utf-8") as f:
    json.dump({"modelos": resumenes, "rendimiento": rendimiento, "evaluacion_visitantes": ev}, f,
              ensure_ascii=False, indent=2, default=str)
print("\n".join(lineas))

## 8. Exportar al repositorio
Escribe en `app_pdi/repo/` (y en `Drive/Proyecto-final-de-PDI/repo/`) los archivos generados para GitHub: la interfaz separada en `frontend/`, la carpeta `results/` (métricas, tablas y figuras), `models/metricas_cnn.json` y un `requirements.txt` con las versiones exactas de este entorno. El `README.md` y `docs/` del repositorio se mantienen a mano: copia encima esta carpeta exportada.

In [ ]:
import importlib.metadata as _md

REPO = config.DIR_TRABAJO / "repo"
if REPO.exists():
    shutil.rmtree(REPO)
(REPO / "frontend").mkdir(parents=True)
(REPO / "models").mkdir()
for nombre, contenido in {"index.html": INDEX_HTML, "styles.css": STYLES_CSS, "app.js": APP_JS}.items():
    (REPO / "frontend" / nombre).write_text(contenido, encoding="utf-8")
shutil.copytree(config.DIR_RESULTADOS, REPO / "results")
for ruta in (config.RUTA_METRICAS, config.RUTA_PREDICCIONES):
    if ruta.exists():
        shutil.copy2(ruta, REPO / "models" / ruta.name)

paquetes = ["torch", "torchvision", "numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "opencv-python",
            "onnxruntime", "pillow", "fastapi", "uvicorn", "python-multipart"]
lineas_req = []
for p in paquetes:
    try:
        lineas_req.append(f"{p}=={_md.version(p)}")
    except _md.PackageNotFoundError:
        alt = {"opencv-python": "opencv-python-headless"}.get(p)
        try:
            lineas_req.append(f"{alt}=={_md.version(alt)}" if alt else p)
        except _md.PackageNotFoundError:
            lineas_req.append(p)
(REPO / "requirements.txt").write_text("\n".join(lineas_req) + "\n", encoding="utf-8")

archivo_zip = shutil.make_archive(str(config.DIR_TRABAJO / "repo_export"), "zip", REPO)
if config.DRIVE_PROYECTO.is_dir():
    destino = config.DRIVE_PROYECTO / "repo"
    if destino.exists():
        shutil.rmtree(destino)
    shutil.copytree(REPO, destino)
    shutil.copy2(archivo_zip, config.DRIVE_PROYECTO / "repo_export.zip")
    print("Copia en Drive:", destino)
print("Repositorio exportado en", REPO, "·", archivo_zip)
for p in sorted(REPO.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(REPO))
print("\nNota: cnn_obras.pt (~45 MB) no se incluye; súbelo a Drive o a GitHub Releases y enlázalo en el README.")

In [ ]:
# Solo cuando el notebook se ejecuta como script .py: mantiene vivo el servidor hasta Ctrl+C.
if not EN_JUPYTER:
    print(f"Aplicativo en http://127.0.0.1:{config.PUERTO} · Ctrl+C para detener")
    try:
        hilo_servidor.join()
    except KeyboardInterrupt:
        detener_servidor()